# Building an Enterprise Data Agent on Oracle AI Database

![Oracle-native enterprise data agent — reference architecture](images/cover-oracle-reference-arch.png)

### A step-by-step construction of an agent *harness*, built from Oracle primitives.

## Why an enterprise data agent?

Most of the value in an enterprise database is locked behind people who know how to query it. An analyst who understands the schema. A DBA who knows which tables actually drive the metric. A senior fleet engineer who remembers that `vessels.capacity_teu` is measured in 20-foot equivalent units, not tons or cubic metres. The data is *there* — sitting in tables, indexed, transactionally consistent — but the path from *"how is fleet utilization trending across our active voyages?"* to a SQL result that answers the question runs through a human bottleneck.

An **enterprise data agent** is the abstraction that closes that gap. It exposes the database through natural language to people who shouldn't need to learn the schema:

- **Operators** asking questions of live data without filing a ticket with the analytics team.
- **Domain experts** — finance, HR, ops — who know what they need but not how to phrase it as a join.
- **Other agents and applications** that need a vetted, audited interface to the data layer.

Done well, the agent learns the database the way a senior engineer does — accumulating knowledge of which tables matter, what the columns *really* mean, what corrections users have made — and grounds every answer in retrievable institutional context rather than the model's training-time guesses. Done badly, it hallucinates a `WAREHOUSES` table that doesn't exist and confidently quotes the declared cargo value in dollars when the schema actually stores it in cents.

This notebook walks through the second-by-second mechanics of building one: how the model talks to the database, how it remembers what it has learned, how it stays inside the boundaries of what the user is allowed to see, and how all of that is assembled from primitives the database already ships.

## Oracle-first stack

Every layer of the harness is something Oracle AI Database already provides. The only piece outside the database is the chat LLM endpoint (and even that can be OCI GenAI if you don't want to leave Oracle).

| Concern | What we use |
|---|---|
| Chat LLM provider | **OpenAI** directly, or **OCI GenAI** (Grok via the OpenAI-compatible endpoint) — switchable via `LLM_PROVIDER` |
| Database | **Oracle AI Database 26ai Free** (Docker, on your laptop) |
| Long-term / semantic memory | **Oracle AI Agent Memory Package (OAMP)** on Oracle AI Database — memories, threads, context cards |
| Conversational / episodic memory | Each (user, assistant) pair stored as an OAMP memory tagged `kind=episodic` with `thread_id` — retrievable via `search_knowledge` so the agent can recall prior interactions across threads |
| Embeddings | **In-database ONNX** (`all-MiniLM-L12-v2` augmented) via `DBMS_VECTOR.LOAD_ONNX_MODEL` + `VECTOR_EMBEDDING` |
| Reranking | **In-database ONNX cross-encoder** (`PREDICTION` over a regression-mode reranker), with graceful pass-through when none is loaded |
| Hybrid retrieval | **Vector + Oracle Text** combined via Reciprocal Rank Fusion (`RRF`) — semantic recall from `VECTOR_DISTANCE` plus exact-keyword precision from a `CTXSYS.CONTEXT` full-text index, both fused server-side in one SQL statement |
| Vector search | **`VECTOR_DISTANCE` + HNSW** indexes (`ORGANIZATION INMEMORY NEIGHBOR GRAPH`) |
| Tool registry | **`toolbox`** table with vector-indexed embeddings; tools retrieved per turn by similarity |
| Skill registry | **`skillbox`** table with vector-indexed embeddings; skills sourced from [`oracle/skills`](https://github.com/oracle/skills) repo, top-k surfaced per turn as a manifest in the system prompt and full body loaded on demand via `load_skill` |
| Short-term scratchpad | **Oracle DBFS** (Database File System over SecureFile LOBs) |
| Structured metadata | **OSON** (Oracle's binary JSON format) on `metadata` columns |
| Sandboxed code execution | **Oracle MLE** — JavaScript running inside the database via `DBMS_MLE` |

Everything below runs on your laptop inside a single Docker container.


## What you'll learn

This notebook builds a complete enterprise data agent on a single Oracle AI Database 26ai container — every primitive listed below is wired up, tested, and demoed end to end.

| Theme | What you'll wire in |
|---|---|
| **Memory** | Long-term semantic memory via the Oracle AI Agent Memory Package (OAMP) — durable memories, conversation threads, rolling LLM-written summaries; a schema scanner that turns `ALL_TABLES` / `ALL_TAB_COLUMNS` / `ALL_CONSTRAINTS` / `V$SQL` into embedded facts; per-turn episodic memory so prior conversations are recallable across threads |
| **Retrieval** | In-database ONNX embeddings via `DBMS_VECTOR.LOAD_ONNX_MODEL` + HNSW vector indexes; cross-encoder reranking via `DBMS_VECTOR.RERANK`; hybrid vector + Oracle Text (`CTXSYS.CONTEXT`) retrieval fused server-side via Reciprocal Rank Fusion |
| **Tools & skills** | A vector-indexed `toolbox` that surfaces only the top-k relevant tools per turn (so the registry can grow past 30 tools without bloating prompts); a `skillbox` of prose playbooks ingested from the `oracle/skills` repo — manifest in the prompt, full body loaded on demand; a DBFS scratchpad for multi-step work; in-database JavaScript via Oracle MLE for deterministic compute the LLM shouldn't do in its head |
| **The agent loop** | One `agent_turn(...)` function: build context → call LLM → dispatch tools → persist outputs → stop on iteration count or wall-clock budget; tool-output offload with compact `fetch_tool_output(id)` references when results would otherwise blow the context window |
| **Documents** | JSON Relational Duality Views — the same row served as a relational tuple *and* a JSON document; writable views with `WITH UPDATE` and kernel-enforced ETag-based optimistic concurrency (ORA-42699 on stale writes, no application-level locks) |
| **Production concerns** | Continuous re-scans via `DBMS_SCHEDULER` so institutional knowledge stays current; identity-aware authorization with Deep Data Security (DDS) — kernel-enforced row + column policies that follow the end-user the agent is acting for, not the AGENT DB user; trusted-procedure-protected application context for identity propagation |

Everything fits in roughly 300 lines of harness code. The rest is Oracle.


## Part 1 — What is an agent harness?

*Starting point. Before any code we need a shared mental model: where does the LLM end and the harness begin? Every later Part adds one specific component to the harness sketched here.*

Before we write a single line of code, let's get the mental model right, because the rest of the notebook depends on it.



### 1.1 The formula

```
Agent = Model + Harness
```

A **model** takes in tokens and produces tokens. That's it. Out of the box it cannot:

- maintain state across interactions
- execute code or SQL
- access the world (filesystem, database, internet)
- set up its own environment

Everything else — every piece of code, configuration, and execution logic that is *not the model* — is the **harness**. Tool dispatch, system prompts, memory read/write, context compaction, the retry-with-timeout wrapper around `client.chat.completions.create(...)`. All harness.

### 1.2 Working backwards from what we want

We don't start with "what features should the harness have?" — we start with a concrete scenario the agent has to handle and derive the harness piece that makes it possible. The scenarios below are the kind of thing an analyst or operator actually asks for:

| Desired behaviour | Harness component |
|---|---|
| "What did dispatch correct me about last week when I asked the same question?" | **Persistent memory** — Oracle AI Agent Memory Package (OAMP) backed by Oracle AI Database |
| "Don't re-run that 4-minute aggregate query — re-use the result you got last time." | **Tool-output offloading** — full tool outputs stored as OAMP memories, compact references inlined into context |
| "Which tables in *this* database hold the voyage-delay signal?" | **Institutional knowledge** — schema scanned into vectors, retrieved by meaning |
| "Find the rows most relevant to my question even if my keywords don't match." | **In-database ONNX embeddings + cross-encoder rerank** — `VECTOR_EMBEDDING` for retrieval, `PREDICTION` for rescoring |
| "Show me the voyages above the 95th percentile by declared cargo value, broken out by ocean region." | **In-database compute** — Oracle MLE for reshaping/aggregating without leaving the DB |
| "Make sure the analyst from APAC can only see voyages in the Pacific and Indian oceans." | **Identity-aware kernel policy** — DDS / DBMS_RLS row policy on `voyages.ocean_region` (§14.4) |
| "My session crashed — pick up where I left off, on the same thread, with the same memory." | **Durable, transactional state** — every memory, message, and tool log lives in the database |
| "Stop the runaway agent before it bills me for a thousand tool calls." | **Guarded stop** — iteration count + wall-clock budget around the while loop |

### 1.3 Architecture — what we're going to build, end to end

The diagram below is the whole picture in one frame. Every box maps to a specific Part later in the notebook (the §-numbers in the diagram are clickable navigation hints, not running totals — read top-to-bottom for the build order).

![Enterprise Data Agent — Oracle-Native Architecture](images/cover-oracle-native-arch.png)


**How to read this:**

- **Top half** is what the agent process does in memory: a tight `while` loop that builds context, calls the LLM, dispatches tools, and updates persistent state.
- **Middle band** is the tool surface — the only way the agent affects the world. Always-on tools are guaranteed in every turn's prompt; retrieved tools are surfaced via vector search over the `toolbox` so the registry can grow large without bloating per-turn token cost.
- **Inside the double-bordered box** is everything that lives in Oracle. Three sub-sections: the in-DB engines (vector / text / MLE / spatial), the data (AGENT schema = the harness's own state, SUPPLYCHAIN schema = the business data), and the kernel-enforced policies that apply *no matter what SQL the agent writes*.
- **Outside the box** are the only two off-database dependencies: the chat LLM endpoint and the GitHub-hosted `oracle/skills` tarball that seeds the skillbox at startup.
- **Per-turn flow** at the bottom is a closeup of the `agent_turn` loop that drives everything else.

Read the rest of the notebook top-to-bottom — every box on this diagram is built, demoed, and explained in its own Part.


> ### 💡 Key takeaways — Part 1
>
> - **Agent = Model + Harness.** The model emits tokens; everything else (state, dispatch, memory, identity, budgets) is harness code. Most "agent quality" complaints are harness problems, not model problems.
> - **Work backwards from concrete scenarios.** Each desired user behaviour ("don't recompute that 4-minute query", "filter by acting identity") maps to one harness component. Avoid building generic infra ahead of a scenario that needs it.
> - **The harness is smaller than people think.** The whole loop in this notebook is ~300 lines of Python — the rest is database primitives. Optimise for legibility, not abstraction.


## Part 2 — Environment setup

*Part 1 gave us the mental model (`Agent = Model + Harness`). Part 2 stands up the runtime substrate every later Part depends on: an Oracle AI Database 26ai container, a Python virtualenv with the three packages we need, and credentials for the chat LLM.*

This section sets up the things our harness needs to run:

1. **Docker** hosting **Oracle AI Database 26ai Free**.
2. A Python virtualenv with `oracledb`, `openai`, and `oracleagentmemory`.
3. **Credentials** for the chat LLM (OpenAI directly, or OCI GenAI — your pick).

ONNX embedding (and optionally a reranker) get loaded *into the database* later in §3.4 / §3.5 — no Python embedding model on your laptop, no `pip install sentence-transformers`. Sandboxed code execution uses **Oracle MLE** (in-database JavaScript) — no GraalVM install on your machine.

Run the cells in order. Cells marked *"shell — run once"* are one-time setup; you don't re-run them every session.


### 2.1 Start Oracle AI Database in Docker  *(shell — run once)*

The `container-registry.oracle.com/database/free:latest` image is the official image. On first run it initializes the database, which can take 3–5 minutes. Subsequent `docker start oracle-free` calls are a few seconds.

We expose:
- `1521` — SQL*Net listener (what `oracledb` connects to)
- `5500` — Enterprise Manager Express (web UI, optional)

We persist data to `$HOME/oracle/data` so the database survives container rebuilds.


In [ ]:
%%bash
# One-time: ensure an Oracle AI Database is running on localhost:1521.
# If any container is already publishing that port (e.g. you're sharing a
# pre-existing Oracle Free instance), we use it. Otherwise we start (or create)
# a container named 'oracle-free'.
set -e

# Jupyter kernels typically don't inherit the shell's full PATH; make sure
# Homebrew (Apple Silicon + Intel) bins are reachable so `docker` resolves.
export PATH="/opt/homebrew/bin:/usr/local/bin:$PATH"

EXISTING=$(docker ps --filter "publish=1521" --format '{{.Names}}' | head -n1 || true)

if [ -n "$EXISTING" ]; then
    echo "Port 1521 already served by container '$EXISTING' — using it."
    echo "(§3.1 will probe FREEPDB1 with the actual driver — that's the source of truth.)"
    exit 0
fi

if docker ps -a --format '{{.Names}}' | grep -q '^oracle-free$'; then
    echo "Container 'oracle-free' exists — starting it"
    docker start oracle-free
else
    echo "Creating new 'oracle-free' container"
    mkdir -p "$HOME/oracle/data"
    docker run -d \
        --name oracle-free \
        -p 1521:1521 \
        -p 5500:5500 \
        -e ORACLE_PWD=OraclePwd_2025 \
        -v "$HOME/oracle/data:/opt/oracle/oradata" \
        container-registry.oracle.com/database/free:latest
fi

echo
echo "Waiting for database to open (this can take several minutes on first boot)..."
for i in $(seq 1 60); do
    status=$(docker exec oracle-free bash -c \
        "echo 'select open_mode from v\$pdbs where name=\"FREEPDB1\";' | sqlplus -s sys/OraclePwd_2025@localhost:1521/FREE as sysdba 2>/dev/null" \
        | tr -d ' \n\r' || true)
    if echo "$status" | grep -qi "READWRITE"; then
        echo "FREEPDB1 is OPEN (READ WRITE)."
        break
    fi
    echo "  attempt $i: not ready yet, sleeping 5s"
    sleep 5
done


### 2.3 Python dependencies

Minimal — we are not using an agent framework.

- `oracledb` — the official Python driver for Oracle.
- `openai` — used *only* as the HTTP client for the OCI GenAI endpoint (which speaks the OpenAI wire protocol). No OpenAI models are called.
- `sentence-transformers` — local embedding model so we can vectorize schema metadata without round-tripping to a hosted embedding API.
- `numpy` — companion for `sentence-transformers`.


In [ ]:
%pip install --quiet "oracledb>=2.4" "openai>=1.50" "sentence-transformers>=2.5" "numpy>=1.26" "tqdm>=4.66" "oracleagentmemory>=26.4" "gdown>=5.2"


### 2.4 Credentials

Pick a chat-LLM provider with `LLM_PROVIDER`:

| `LLM_PROVIDER` | What runs the loop | Default model | Key to set |
|---|---|---|---|
| `openai` *(default)* | OpenAI directly | `gpt-5.5` | `OPENAI_API_KEY` |
| `oci` | Grok via OCI GenAI's OpenAI-compatible endpoint | `xai.grok-3-fast` | `OCI_GENAI_API_KEY` (+ optional `OCI_GENAI_ENDPOINT`) |

Override the model with `LLM_MODEL` (e.g. `gpt-5.4`, `gpt-5.5-mini`, `xai.grok-3`).

Either way you'll also need `OPENAI_API_KEY` for the OAMP embedder (`text-embedding-3-small`) — we prompt for it in §5.1 just before the embedder is first used. If you've already set it here for the chat LLM, that prompt is a no-op.


In [ ]:
import os, getpass, warnings

# Silence OAMP's "calling sync from async context" UserWarnings.
# Jupyter always has a running event loop, and the sync methods work fine
# under it - the warnings are noise for our use.
warnings.filterwarnings("ignore", category=UserWarning, module="oracleagentmemory")


LLM_PROVIDER = os.environ.setdefault("LLM_PROVIDER", "oci").lower() #oci or #openai
assert LLM_PROVIDER in ("openai", "oci"), "LLM_PROVIDER must be 'openai' or 'oci'"

if LLM_PROVIDER == "oci":
    if not os.environ.get("OCI_GENAI_API_KEY"):
        os.environ["OCI_GENAI_API_KEY"] = "" #getpass.getpass("OCI GenAI API key: ")
    os.environ.setdefault(
        "OCI_GENAI_ENDPOINT",
        "https://inference.generativeai.us-phoenix-1.oci.oraclecloud.com",
    )
    os.environ.setdefault("LLM_MODEL", "xai.grok-4-1-fast-reasoning") # gpt-5.5-mini | xai.grok-4 | xai.grok-4-1-fast-reasoning | xai.grok-4.20-reasoning | xai.grok-4.3
    print(f"Provider: OCI GenAI ({os.environ['LLM_MODEL']})")
    print(f"Endpoint: {os.environ['OCI_GENAI_ENDPOINT']}")
else:  # openai
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] =  "..." #getpass.getpass("OpenAI API key: ")
    os.environ.setdefault("LLM_MODEL", "gpt-5.5")
    print(f"Provider: OpenAI ({os.environ['LLM_MODEL']})")


> ### 💡 Key takeaways — Part 2
>
> - **One container, whole stack.** Oracle AI Database 26ai Free runs the embedder, reranker, vector index, text index, MLE sandbox, and storage in one process. No separate vector DB, no separate embedding service, no separate sandbox runtime.
> - **LLM provider is swappable; everything else isn't.** `LLM_PROVIDER` switches OpenAI ↔ OCI GenAI without code changes. The DB layer is the invariant.
> - **Get credentials in once, reuse them.** `getpass` once at the top, environment variables thereafter. Never hardcode keys into notebook cells — they end up in git history.


## Part 3 — Connectivity: DSN → database, key → LLM

*Part 2 gave us the substrate. Part 3 opens the two channels every later Part will reuse — one to the database via `oracledb`, one to the chat model via the OpenAI-compatible client. We also load the in-database ONNX embedder and reranker here, so embedding and rescoring happen inside Oracle from §4 onward instead of via network calls.*

Two clients to initialize. Both are plain, unopinionated — no framework wrappers.

### 3.1 The DSN

An Oracle DSN is a string of the form `HOST:PORT/SERVICE_NAME`. For our local Docker container that's `localhost:1521/FREEPDB1`. In production it'll be a TNS alias or an EZCONNECT string pointing at your enterprise database — our agent is indifferent; it takes the DSN as a parameter.


In [ ]:
import oracledb
import time

SYS_DSN    = "localhost:1521/FREEPDB1"
SYS_USER   = "sys"
SYS_PASS   = "OraclePwd_2025"

AGENT_USER = "AGENT"
AGENT_PASS = "AgentPwd_2025"


def connect(user: str, password: str, dsn: str, mode: int | None = None, retries: int = 5) -> oracledb.Connection:
    """Thin wrapper around oracledb.connect with retry - the one we reuse everywhere."""
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            kwargs = dict(user=user, password=password, dsn=dsn)
            if mode is not None:
                kwargs["mode"] = mode
            conn = oracledb.connect(**kwargs)
            with conn.cursor() as cur:
                cur.execute("SELECT banner FROM v$version WHERE rownum = 1")
                print(f"connected as {user}@{dsn} - {cur.fetchone()[0]}")
            return conn
        except Exception as e:
            last_err = e
            print(f"  attempt {attempt}/{retries} failed: {e}")
            time.sleep(3)
    raise RuntimeError(f"could not connect to {dsn}: {last_err}")


### 3.2 Create a dedicated `AGENT` user

Running the agent as `SYS` would be reckless — any hallucinated `DROP TABLE` becomes a real outage. We create a dedicated low-privilege user, `AGENT`, that owns its own schema (where we'll keep memory) and is granted `SELECT` on what it's allowed to *read* from the business schemas we want it to reason about.

We also grant `DBMS_DBFS_CONTENT` and `DBMS_DBFS_SFS` so the agent can manage its scratchpad in DBFS (§7).


In [ ]:
sys_conn = connect(SYS_USER, SYS_PASS, SYS_DSN, mode=oracledb.AUTH_MODE_SYSDBA)

bootstrap_stmts = [
    # Create AGENT user if missing
    f"DECLARE n NUMBER; BEGIN "
    f"  SELECT COUNT(*) INTO n FROM all_users WHERE username = '{AGENT_USER}'; "
    f"  IF n = 0 THEN "
    f"    EXECUTE IMMEDIATE 'CREATE USER {AGENT_USER} IDENTIFIED BY {AGENT_PASS}'; "
    f"  END IF; "
    f"END;",
    f"GRANT CONNECT, RESOURCE, CREATE SESSION TO {AGENT_USER}",
    f"GRANT CREATE TABLE, CREATE SEQUENCE, CREATE VIEW, CREATE PROCEDURE TO {AGENT_USER}",
    f"GRANT UNLIMITED TABLESPACE TO {AGENT_USER}",
    # Privileges used by the schema scanner in section 5
    f"GRANT SELECT ON SYS.V_$SQL TO {AGENT_USER}",
    f"GRANT SELECT ON SYS.V_$SQLSTATS TO {AGENT_USER}",
    f"GRANT SELECT_CATALOG_ROLE TO {AGENT_USER}",
    # Privileges for DBFS scratchpad in section 7
    f"GRANT EXECUTE ON DBMS_DBFS_CONTENT TO {AGENT_USER}",
    # Privileges for in-database Python (MLE) in section 9
    f"GRANT EXECUTE ON DBMS_MLE TO {AGENT_USER}",
    f"GRANT DB_DEVELOPER_ROLE TO {AGENT_USER}",
    f"GRANT EXECUTE DYNAMIC MLE TO {AGENT_USER}",
    f"GRANT EXECUTE ON DBMS_DBFS_SFS TO {AGENT_USER}",
    f"GRANT DBFS_ROLE TO {AGENT_USER}",
]

with sys_conn.cursor() as cur:
    for stmt in bootstrap_stmts:
        try:
            cur.execute(stmt)
            print("ok:", stmt.splitlines()[0][:80])
        except oracledb.DatabaseError as e:
            print("skip:", e)
sys_conn.commit()


### 3.2.1 Allocate the vector memory pool *(one-time, requires bounce)*

Oracle 23ai/26ai stores HNSW vector indexes in a dedicated in-memory pool sized by `vector_memory_size`. On a stock Free image it ships at `0`, so any `CREATE VECTOR INDEX ... ORGANIZATION INMEMORY ...` statement raises `ORA-51962: The vector memory area is out of space for the current container.`

We need this for two reasons later:

- §4.1 — OAMP creates vector indexes on its memory tables.
- §10 — the `toolbox` table has an HNSW index on the tool embeddings.

The cell below sets the parameter at SPFILE scope. The change takes effect after a DB restart, so the *first* time you run this notebook against a fresh container you need to `docker restart oracle-free` and re-execute cells from §3.1 onward. Subsequent runs detect that vector memory is already configured and no-op.


In [ ]:
with sys_conn.cursor() as cur:
    cur.execute("SELECT value FROM v$parameter WHERE name = 'vector_memory_size'")
    current = int(cur.fetchone()[0] or 0)

TARGET_VECTOR_MEMORY = 512 * 1024 * 1024  # 512 MiB - plenty for this notebook

if current >= TARGET_VECTOR_MEMORY:
    print(f"vector_memory_size already configured ({current/1024/1024:.0f} MiB). OK.")
else:
    with sys_conn.cursor() as cur:
        # vector_memory_size is an instance-wide CDB parameter. The sys_conn here is
            # logged into FREEPDB1 (PDB), so we must add CONTAINER=ALL — otherwise the
            # ALTER scopes to the PDB only, the SPFILE entry never feeds the SGA, and
            # post-bounce v$parameter still reads 0 (which is what bit us until we
            # found the alert log entry "ALTER SYSTEM SET ... PDB='FREEPDB1'").
            cur.execute("ALTER SYSTEM SET vector_memory_size = 512M SCOPE=SPFILE CONTAINER=ALL")
    sys_conn.commit()
    print("vector_memory_size set to 512M (SPFILE).")
    print()
    print("ACTION REQUIRED: bounce the database to pick up the new pool, then re-run §3.1+:")
    print("    docker restart oracle-free")
    print("Once the container is back up and FREEPDB1 reports READ WRITE, restart the kernel")
    print("and re-run the cells from §3.1 down.")


# pga_aggregate_limit governs how much PGA the instance can consume in
# aggregate. DBMS_VECTOR.RERANK on a candidate set ~20 docs allocates enough
# transient PGA per call that the Free build's default (~2 GiB) is exceeded
# and the call falls back to plain cosine ordering with ORA-04036. We bump
# it to 4 GiB at the CDB level (the only level where pga_aggregate_limit
# lives) using SCOPE=BOTH so the change applies live and persists.
#
# Several clause combinations to handle:
#   - sys_conn here is logged into FREEPDB1 (PDB). Running ALTER SYSTEM with
#     CONTAINER=ALL from a PDB raises ORA-02065 ("illegal option") because
#     CONTAINER=ALL is only valid from CDB$ROOT.
#   - Running with SCOPE=BOTH (no CONTAINER clause) from the PDB still
#     mutates the instance-wide PGA pool because pga_aggregate_limit is a
#     CDB-only parameter — Oracle silently routes the change to the CDB.
#   - On some images even that is rejected; we catch ORA-02065 / ORA-65040
#     and just print an actionable message rather than failing the cell.
TARGET_PGA = 4 * 1024 * 1024 * 1024  # 4 GiB
with sys_conn.cursor() as cur:
    cur.execute("SELECT value FROM v$parameter WHERE name = 'pga_aggregate_limit'")
    pga_now = int(cur.fetchone()[0] or 0)
if pga_now >= TARGET_PGA:
    print(f"pga_aggregate_limit already configured ({pga_now/1024/1024/1024:.1f} GiB). OK.")
else:
    try:
        with sys_conn.cursor() as cur:
            cur.execute("ALTER SYSTEM SET pga_aggregate_limit = 4G SCOPE=BOTH")
        sys_conn.commit()
        print(f"pga_aggregate_limit raised to 4 GiB (was {pga_now/1024/1024/1024:.2f} GiB). "
              "No bounce required (SCOPE=BOTH).")
    except oracledb.DatabaseError as e:
        code_ = e.args[0].code
        print(f"!! Could not raise pga_aggregate_limit from this PDB session "
              f"(ORA-{code_:05d}). DBMS_VECTOR.RERANK may fall back to cosine order "
              f"under load. To bump it manually, connect to CDB$ROOT as SYSDBA and run:")
        print(f'    ALTER SYSTEM SET pga_aggregate_limit = 4G SCOPE=BOTH;')


### 3.2.2 Raise the PGA aggregate limit *(one-time, no bounce)*

`DBMS_VECTOR.RERANK` (used by the §3.5 reranker) allocates enough transient PGA per call that the Free build's default ceiling (~2 GiB) is exceeded under modest load — the call falls back to plain cosine ordering with `ORA-04036`. We raise the limit to **4 GiB**.

`pga_aggregate_limit` is a **CDB-only** parameter — `ALTER SYSTEM` from inside a PDB session is rejected (`ORA-02065` / `ORA-02097`), even with SYSDBA. The cell below uses `docker exec` to drop into a SQL\*Plus session attached to `CDB$ROOT`, which is where the change has to be made. `SCOPE=BOTH` means it applies live (no bounce required) and persists across restarts.

Idempotent: re-running it just sets the value to 4G again.


In [ ]:
%%bash
# Raise pga_aggregate_limit at CDB$ROOT — the only level where the parameter
# accepts changes. SCOPE=BOTH applies live + persists; no docker bounce needed.
docker exec -i oracle-free sqlplus -s / as sysdba <<'SQL'
WHENEVER SQLERROR EXIT SQL.SQLCODE
ALTER SESSION SET CONTAINER = CDB$ROOT;
ALTER SYSTEM SET pga_aggregate_limit = 4G SCOPE=BOTH;
SELECT name, value/1024/1024/1024 AS gib
  FROM v$parameter WHERE name = 'pga_aggregate_limit';
EXIT
SQL


Above we did all the privileged setup as `SYS`. From here on the agent operates as the dedicated low-privilege `AGENT` user — every memory write, every `run_sql` dispatch, every MLE call uses this connection. Keeping a separate `agent_conn` makes it safe to grant only what the agent strictly needs and audit accordingly.


In [ ]:
# From here on the agent uses its own connection - NOT sys.
agent_conn = connect(AGENT_USER, AGENT_PASS, SYS_DSN)


### 3.3 LLM client — OpenAI direct or Grok via OCI GenAI

The agent talks to the model through the plain `openai` SDK. Two configurations:

- **`LLM_PROVIDER=openai`** — talks to OpenAI directly. The SDK uses `OPENAI_API_KEY` and the default `https://api.openai.com/v1` base URL. Models like `gpt-5.5` / `gpt-5.4` work out of the box.
- **`LLM_PROVIDER=oci`** — talks to OCI GenAI's OpenAI-compatible endpoint. We override `base_url` and pass `OCI_GENAI_API_KEY`. Grok 3 Fast is the default model.

Either way, we wrap `chat.completions.create` in a thin retry that backs off on HTTP 429 — that's the only layer between our loop and the model.


In [ ]:
from openai import OpenAI

LLM_MODEL = os.environ["LLM_MODEL"]

if LLM_PROVIDER == "oci":
    llm = OpenAI(
        base_url=os.environ["OCI_GENAI_ENDPOINT"],
        api_key=os.environ["OCI_GENAI_API_KEY"],
    )
else:  # openai (default base_url)
    llm = OpenAI(
        api_key=os.environ["OPENAI_API_KEY"],
    )


def chat(messages: list[dict], tools: list[dict] | None = None,
          model: str = LLM_MODEL, max_retries: int = 3):
    """Call the LLM with retry on 429. Returns the raw ChatCompletion object."""
    kwargs = {"model": model, "messages": messages}
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = "auto"

    delay = 2.0
    for attempt in range(max_retries + 1):
        try:
            return llm.chat.completions.create(**kwargs)
        except Exception as e:
            status = getattr(e, "status_code", None) or 0
            if status == 429 and attempt < max_retries:
                print(f"  429 rate limit - retrying in {delay:.0f}s")
                time.sleep(delay); delay *= 2
            else:
                raise


# Smoke test
resp = chat([
    {"role": "system", "content": "Be terse."},
    {"role": "user",   "content": "Say 'pong'."},
])
print(f"[{LLM_PROVIDER}/{LLM_MODEL}]", resp.choices[0].message.content)


### 3.4 Load the ONNX embedding model into Oracle

We replace the hosted OpenAI embedder with an **Oracle-augmented ONNX model running inside the database**. The pattern is from the developer-hub `onnx_embeddings_oracle_ai_database.ipynb` reference notebook:

1. Download the augmented `all-MiniLM-L12-v2` ONNX model (~117 MB) from Oracle's object store.
2. Copy the file into the running `oracle-free` container under `/opt/oracle/onnx_models/`.
3. As `SYSDBA`, create an Oracle directory pointing at that path and grant the `AGENT` user read/write.
4. As `AGENT`, register the model with `DBMS_VECTOR.LOAD_ONNX_MODEL` so it's usable from SQL via `VECTOR_EMBEDDING`.

**Why this matters for the harness:**

- Embedding stops being a network call to OpenAI. Every `add_memory`, every `search`, every   `toolbox` insert/retrieve happens *inside* the database in the same transaction — no OpenAI round-trip, no `OPENAI_API_KEY` for the embedding path.
- 384 dims (vs 1536 for `text-embedding-3-small`) keeps the vector indexes cheaper to build and scan.
- The model lives where the data lives. Same backups, same audit, same security model.

We also stamp the loaded model name into a Python constant `ONNX_EMBED_MODEL` so the rest of the notebook can reference it.


In [ ]:
import subprocess, urllib.request, urllib.error, tempfile, os, zipfile, pathlib, shutil

# --- 3.4.1 Download + stage the ONNX model in the container ---
CONTAINER_NAME       = "oracle-free"
CONTAINER_MODEL_DIR  = "/opt/oracle/onnx_models"
ONNX_FILE            = "all_MiniLM_L12_v2.onnx"
ONNX_DIRECTORY       = "ONNX_DIR"
ONNX_EMBED_MODEL     = "ALL_MINILM_L12_V2"   # Oracle convention: uppercase
ORACLE_MODEL_URL = (
    "https://adwc4pm.objectstorage.us-ashburn-1.oci.customer-oci.com"
    "/p/TtH6hL2y25EypZ0-rrczRZ1aXp7v1ONbRBfCiT-BDBN8WLKQ3lgyW6RxCfIFLdA6"
    "/n/adwc4pm/b/OML-ai-models/o/all_MiniLM_L12_v2_augmented.zip"
)
# Jupyter kernels often miss /opt/homebrew/bin etc - extend PATH so shutil.which
# finds the docker (or podman) binary.
os.environ["PATH"] = ":".join([
    "/opt/homebrew/bin", "/usr/local/bin", "/usr/bin", "/bin",
    os.environ.get("PATH", ""),
])
CONTAINER_CLI = "docker" if shutil.which("docker") else "podman"

In [ ]:

def _exists_in_container(path):
    return subprocess.run(
        [CONTAINER_CLI, "exec", CONTAINER_NAME, "test", "-f", path],
        capture_output=True,
    ).returncode == 0

# Make sure the container-side directory exists.
subprocess.run(
    [CONTAINER_CLI, "exec", CONTAINER_NAME, "mkdir", "-p", CONTAINER_MODEL_DIR],
    check=True,
)

target_path = f"{CONTAINER_MODEL_DIR}/{ONNX_FILE}"
if _exists_in_container(target_path):
    print(f"'{ONNX_FILE}' already present in {CONTAINER_NAME} - skipping download.")
else:
    print("Downloading Oracle augmented all-MiniLM-L12-v2 ONNX model (~117 MB)...")
    with tempfile.TemporaryDirectory() as tmp:
        zip_path = os.path.join(tmp, "model.zip")
        try:
            urllib.request.urlretrieve(ORACLE_MODEL_URL, zip_path)
            print("  download complete.")
        except urllib.error.URLError as e:
            raise SystemExit(
                f"Download failed: {e}. The pre-signed URL may have rotated; "
                "see https://docs.oracle.com/en/database/oracle/oracle-database/23/vecse/"
                "import-onnx-models-oracle-database-end-end-example.html"
            )

        print("  extracting...")
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(tmp)
        onnx_path = next(pathlib.Path(tmp).glob("*.onnx"))

        print(f"  copying into {CONTAINER_NAME}:{target_path}...")
        subprocess.run(
            [CONTAINER_CLI, "cp", str(onnx_path), f"{CONTAINER_NAME}:{target_path}"],
            check=True,
        )
        # chmod as root (the default exec user often can't change perms on
        # files it didn't create); ignore output - readability is what matters.
        subprocess.run(
            [CONTAINER_CLI, "exec", "--user", "0", CONTAINER_NAME,
             "chmod", "644", target_path],
            check=False, capture_output=True,
        )
    print(f"\n'{ONNX_FILE}' is now in {CONTAINER_NAME}:{target_path}")


Oracle needs an `OR_DIRECTORY` mapped to the host path inside the container so `DBMS_VECTOR.LOAD_ONNX_MODEL` can read the file we just copied in. Created as `SYSDBA`, granted to `AGENT` so subsequent steps don't need elevated privileges.


In [ ]:
# --- 3.4.2 Create the Oracle directory and grant AGENT access (SYSDBA) ---
with sys_conn.cursor() as cur:
    cur.execute(
        f"CREATE OR REPLACE DIRECTORY {ONNX_DIRECTORY} AS '{CONTAINER_MODEL_DIR}'"
    )
    cur.execute(
        f"GRANT READ, WRITE ON DIRECTORY {ONNX_DIRECTORY} TO {AGENT_USER}"
    )
    # CREATE MINING MODEL is required for LOAD_ONNX_MODEL to register the model.
    cur.execute(f"GRANT CREATE MINING MODEL TO {AGENT_USER}")
sys_conn.commit()
print(f"directory {ONNX_DIRECTORY} -> {CONTAINER_MODEL_DIR}; granted to {AGENT_USER}.")


Final step of the embedder install: register the ONNX file as a named model in Oracle's mining-model registry. After this cell, every later Part can call `VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :text AS DATA)` from SQL — no Python embedder, no network round-trip, embedding happens inside the database.


In [ ]:
# --- 3.4.3 Register the ONNX model in Oracle (AGENT user) ---
with agent_conn.cursor() as cur:
    cur.execute(
        "SELECT COUNT(*) FROM user_mining_models WHERE model_name = :m",
        m=ONNX_EMBED_MODEL,
    )
    already_loaded = cur.fetchone()[0] > 0

if already_loaded:
    print(f"model {ONNX_EMBED_MODEL!r} is already loaded.")
else:
    print(f"loading ONNX model {ONNX_EMBED_MODEL!r} from {ONNX_DIRECTORY}/{ONNX_FILE}...")
    with agent_conn.cursor() as cur:
        cur.execute(
            "BEGIN "
            "  DBMS_VECTOR.LOAD_ONNX_MODEL("
            "    directory  => :d, "
            "    file_name  => :f, "
            "    model_name => :m, "
            "    metadata   => JSON('{\"function\":\"embedding\",\"embeddingOutput\":\"embedding\",\"input\":{\"input\":[\"DATA\"]}}') "
            "  ); "
            "END;",
            d=ONNX_DIRECTORY, f=ONNX_FILE, m=ONNX_EMBED_MODEL,
        )
    agent_conn.commit()
    print(f"loaded model {ONNX_EMBED_MODEL!r}.")

# Smoke test: produce one embedding and report its dimension.
with agent_conn.cursor() as cur:
    cur.execute(
        f"SELECT VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :t AS DATA) FROM dual",
        t="Oracle in-database embedding round-trip.",
    )
    vec = cur.fetchone()[0]

ONNX_EMBED_DIM = len(vec)
print(f"VECTOR_EMBEDDING produced a {ONNX_EMBED_DIM}-dim vector.")


### 3.5 Load the ONNX reranker (and the `rerank()` helper)

After cosine top-k, a **cross-encoder reranker** rescores the candidates by reading the query and each candidate together. This usually buys a meaningful precision lift on the first few hits, at the cost of one extra model call per turn.

Oracle's local reranker pipeline is registered via the **same** `DBMS_VECTOR.LOAD_ONNX_MODEL` call we used in §3.4, but the metadata is different — rerankers are loaded as a **regression** model with two inputs (`DATA1` for the query, `DATA2` for the candidate document). Once loaded, you score a (query, document) pair via the standard SQL `PREDICTION()` operator:

```sql
SELECT PREDICTION(reranker_model USING :q AS DATA1, :doc AS DATA2) FROM DUAL;
```

Higher scores mean more relevant. (Range is unbounded — apply a sigmoid if you want [0,1].)

**Model URL.** Oracle's `PREDICTION` invocation needs an *augmented* ONNX file (tokenizer pipeline baked in) — vanilla Hugging Face exports won't work because they expect tokenized input arrays, not raw strings. The cell below defaults `RERANKER_URL` to a community-built augmented `bge-reranker-base.onnx` (275 MB) hosted on Google Drive (see [OML4Py 2.1 Quickstart](https://practicalplsql.org/2025/05/25/oml4py-2-1-quickstart/) for provenance and the OML4Py recipe used to produce it). `gdown` handles the Drive download. Override with your own pre-built augmented reranker via env vars:

```bash
export RERANKER_URL='<your URL>'
export RERANKER_FILE='your_file.onnx'
```

If you set `RERANKER_URL=""`, the cell skips the load and `rerank()` falls back to a cosine-order pass-through.


In [ ]:
import subprocess, urllib.request, urllib.error, tempfile, os, re, zipfile, pathlib, json

# Pre-augmented BGE-reranker-base ONNX (275 MB), hosted on Google Drive by the
# OML4Py community (see https://practicalplsql.org/2025/05/25/oml4py-2-1-quickstart/).
# It already has the tokenizer pipeline baked in, so PREDICTION(... USING :q AS DATA1,
# :doc AS DATA2) accepts raw strings.
DEFAULT_RERANKER_URL = (
    "https://drive.google.com/file/d/1-xDRSHr_ulbO7MqVlWLu6ZA2J-bCjUoY/view?usp=drive_link"
)
RERANKER_MODEL = os.environ.get("RERANKER_MODEL_NAME", "RERANKER_ONNX")
RERANKER_URL   = os.environ.get("RERANKER_URL", DEFAULT_RERANKER_URL).strip()
RERANKER_FILE  = os.environ.get("RERANKER_FILE", "bge_reranker_base.onnx")


def _reranker_loaded() -> bool:
    with agent_conn.cursor() as cur:
        cur.execute(
            "SELECT COUNT(*) FROM user_mining_models WHERE model_name = :m",
            m=RERANKER_MODEL,
        )
        return cur.fetchone()[0] > 0


if _reranker_loaded():
    print(f"reranker {RERANKER_MODEL!r} already loaded.")
elif not RERANKER_URL:
    print("RERANKER_URL not set - skipping reranker load.")
    print("rerank() will operate as a pass-through (cosine ordering preserved).")
    print("Set RERANKER_URL to an Oracle-augmented reranker .onnx (or .zip) to enable it again.")
else:
    target = f"{CONTAINER_MODEL_DIR}/{RERANKER_FILE}"
    if not _exists_in_container(target):
        print(f"downloading reranker from {RERANKER_URL}...")
        with tempfile.TemporaryDirectory() as tmp:
            local_path = os.path.join(tmp, RERANKER_FILE)
            drive_match = re.search(r"drive\.google\.com/.*?/d/([A-Za-z0-9_-]+)", RERANKER_URL)
            try:
                if drive_match:
                    import gdown
                    file_id = drive_match.group(1)
                    gdown.download(id=file_id, output=local_path, quiet=False)
                else:
                    urllib.request.urlretrieve(RERANKER_URL, local_path)
            except (urllib.error.URLError, Exception) as e:
                raise SystemExit(f"reranker download failed: {e}")

            # If it's a zip, extract; otherwise assume it is the .onnx itself.
            if local_path.endswith(".zip"):
                with zipfile.ZipFile(local_path) as zf:
                    zf.extractall(tmp)
                onnx_candidate = next(pathlib.Path(tmp).glob("*.onnx"), None)
                if onnx_candidate is None:
                    raise SystemExit("no .onnx file found in reranker zip")
                src_path = str(onnx_candidate)
            else:
                src_path = local_path

            subprocess.run(
                [CONTAINER_CLI, "cp", src_path, f"{CONTAINER_NAME}:{target}"],
                check=True,
            )
            subprocess.run(
                [CONTAINER_CLI, "exec", "--user", "0", CONTAINER_NAME,
                 "chmod", "644", target],
                check=False, capture_output=True,
            )

    print(f"loading reranker {RERANKER_MODEL!r} from {ONNX_DIRECTORY}/{RERANKER_FILE}...")
    # Reranker metadata: regression with two text inputs.
    rerank_meta = json.dumps({
        "function": "regression",
        "regressionOutput": "output",
        "input": {
            "first_input":  ["DATA1"],
            "second_input": ["DATA2"],
        },
    })
    with agent_conn.cursor() as cur:
        cur.execute(
            "BEGIN "
            "  DBMS_VECTOR.LOAD_ONNX_MODEL("
            "    directory  => :d, "
            "    file_name  => :f, "
            "    model_name => :m, "
            "    metadata   => JSON(:meta) "
            "  ); "
            "END;",
            d=ONNX_DIRECTORY, f=RERANKER_FILE, m=RERANKER_MODEL, meta=rerank_meta,
        )
    agent_conn.commit()
    print(f"reranker {RERANKER_MODEL!r} loaded.")

print(f"rerank ready: {_reranker_loaded()}")


The retrieval-side companion to §3.5's reranker model: a Python helper that ships query + each candidate text to `PREDICTION()` server-side and returns the top-k ordered by score. Falls back to a pass-through (cosine ordering preserved) when no reranker is loaded — every later call site stays the same regardless of whether a reranker is in place.


In [ ]:
def rerank(query: str, candidates: list[dict], top_k: int = 5,
            content_key: str = "body") -> list[dict]:
    """Cross-encoder rescoring of a candidate set against `query`.

    When a reranker model is loaded (see §3.5), this builds a JSON_TABLE of
    candidates and calls Oracle's `PREDICTION(reranker USING :q AS DATA1,
    content AS DATA2)` for each, returning the top-k by score.

    When no model is loaded - or the call errors - falls through to the
    candidates' input order, sliced to `top_k`. That way the agent's
    retrieval code is identical whether reranking is on or off.
    """
    if not candidates:
        return candidates
    if not _reranker_loaded():
        return candidates[:top_k]

    docs = [
        {"index": i, "content": str(c.get(content_key, ""))[:1500]}
        for i, c in enumerate(candidates)
    ]

    sql = (
        f"SELECT t.idx, "
        f"       PREDICTION({RERANKER_MODEL} USING :q AS DATA1, t.content AS DATA2) AS score "
        "  FROM JSON_TABLE(:docs, '$[*]' COLUMNS ("
        "         idx     NUMBER          PATH '$.index', "
        "         content VARCHAR2(4000)  PATH '$.content'"
        "       )) t "
        " ORDER BY score DESC "
        " FETCH FIRST :k ROWS ONLY"
    )
    try:
        with agent_conn.cursor() as cur:
            cur.execute(sql, q=query, docs=json.dumps(docs), k=top_k)
            ranked = list(cur)
    except oracledb.DatabaseError as e:
        print(f"rerank failed - falling back to cosine order: {e}")
        return candidates[:top_k]

    out: list[dict] = []
    for idx, score in ranked:
        if idx is None or int(idx) >= len(candidates):
            continue
        item = dict(candidates[int(idx)])
        item["rerank_score"] = float(score) if score is not None else 0.0
        out.append(item)
    return out


# Smoke test against a handful of inline candidates.
_demo = [
    {"body": "Oracle AI Database supports vector search natively."},
    {"body": "Bananas are yellow."},
    {"body": "PREDICTION() rescores documents against a query using a loaded reranker."},
]
for hit in rerank("how does in-database reranking work?", _demo, top_k=3):
    print(f"{hit.get('rerank_score', 0.0):>10.4f}  {hit['body']}")


> ### 💡 Key takeaways — Part 3
>
> - **Separate the agent's DB user from the data's DB user.** `AGENT` owns harness state (memory, scratch, tools); `SUPPLYCHAIN` owns business data. The trust boundary is grants, not Python code.
> - **`vector_memory_size` is non-optional for HNSW.** Without it, `CREATE VECTOR INDEX ORGANIZATION INMEMORY NEIGHBOR GRAPH` raises `ORA-51962` and you silently fall back to full-table cosine scans. Bounce the DB after setting it (`SCOPE=SPFILE`).
> - **`pga_aggregate_limit` lives at CDB$ROOT only.** PDB sessions can't modify it (`ORA-02065/02097`), even with SYSDBA. Use `docker exec` into a CDB$ROOT SQL\*Plus session — same pattern for any CDB-only parameter.
> - **ONNX models load *into* the database.** Embeddings come from `VECTOR_EMBEDDING(...)` SQL calls, not network round-trips to a hosted embedding service. Same trust boundary as your data, lower latency, no egress.


## Part 4 — Tables: the long-term store

*Part 3 gave us live connections and an in-database embedder. Part 4 hands the connection to OAMP (Oracle AI Agent Memory Package) so it owns all the durable-memory tables — memories, threads, context cards. We add one bespoke bookkeeping table, `scan_history`, alongside it.*

The Oracle AI Agent Memory Package (OAMP) owns the long-term store: it manages the schema, the embedding pipeline, and the retrieval surface.

Our job in this section is to point a memory client at our database connection created in the previous section and register the user/agent identities that scope every memory.

We do keep one bespoke table — `scan_history`. It records *that* a scan ran, not *what was learned*, so it's bookkeeping rather than memory and stays outside OAMP.


### 4.1 Memory client (OAMP) + a small bookkeeping table

The long-term store *is* the [Oracle AI Agent Memory Package](https://www.oracle.com/database/ai-agent-memory/) (OAMP).

Instead of hand-rolling the `institutional_knowledge` / `conversation` / `tool_log` schema, we hand a connection to `OracleAgentMemory` and let it own the DDL, the embedding pipeline, and the retrieval surface.

| OAMP primitive | What it stores | Replaces |
|---|---|---|
| `memory` (via `client.add_memory`) | Durable facts — scanned schema entries, user corrections, tool outputs (with metadata). | `institutional_knowledge`, `tool_log` |
| `thread` (via `client.create_thread`) | A conversation. Holds messages and exposes a context card. | `conversation` |
| `context_card` (via `thread.get_context_card`) | Compact, query-relevant block of memories + recent turns. | The `build_context` we'd otherwise hand-write |

We still keep one bespoke table: `scan_history`. It's the agent's **procedural memory** — memory of *how* and *when* it does things, distinct from the semantic memory of facts in `institutional_knowledge` and the episodic memory of `conversation` turns. We keep it outside OAMP because it's queried by time/owner (regular SQL indexes) rather than by meaning (cosine distance). §12.1's incremental-scan optimisation is the simplest decision conditioned on this memory.


In [ ]:
import numpy as np
from oracleagentmemory.core import OracleAgentMemory
from oracleagentmemory.core.llms import Llm
from oracleagentmemory.apis.embedders.embedder import IEmbedder


class OracleONNXEmbedder(IEmbedder):
    """OAMP-compatible embedder backed by Oracle's in-database VECTOR_EMBEDDING.

    Every `embed(...)` call issues `SELECT VECTOR_EMBEDDING(MODEL USING :t AS DATA)`
    against `agent_conn`. No external API calls; embedding happens in the same
    process as the rest of the database work.
    """

    def __init__(self, conn, model_name: str = ONNX_EMBED_MODEL, dim: int = ONNX_EMBED_DIM):
        self._conn = conn
        self._model = model_name
        self._dim = dim

    def embed(self, texts: list[str], *, is_query: bool = False) -> np.ndarray:
        out = np.zeros((len(texts), self._dim), dtype=np.float32)
        sql = f"SELECT VECTOR_EMBEDDING({self._model} USING :t AS DATA) FROM dual"
        with self._conn.cursor() as cur:
            for i, t in enumerate(texts):
                cur.execute(sql, t=t)
                vec = cur.fetchone()[0]
                out[i] = np.asarray(list(vec), dtype=np.float32)
        return out

    async def embed_async(self, texts: list[str], *, is_query: bool = False) -> np.ndarray:
        return self.embed(texts, is_query=is_query)



OAMP can use an LLM in two ways: (1) extract durable memories from each thread, and (2) maintain a rolling thread summary. Both want the same chat model the agent uses. We point a `litellm`-compatible client at whichever provider §3.3 picked, so OAMP and the agent loop share one LLM endpoint.


In [ ]:
# Wire OAMP's extraction LLM to whichever chat provider §3.3 picked.
# OpenAI direct: defaults to api.openai.com using OPENAI_API_KEY.
# OCI GenAI:    point LiteLLM at the OCI endpoint via api_base/api_key.
# Note: GPT-5.x and many newer reasoning-tuned models only allow the default
# temperature; we don't override it here so the same code works across models.
if LLM_PROVIDER == "oci":
    extraction_llm = Llm(
        f"openai/{LLM_MODEL}",
        api_base=os.environ["OCI_GENAI_ENDPOINT"],
        api_key=os.environ["OCI_GENAI_API_KEY"],
    )
else:  # openai
    extraction_llm = Llm(LLM_MODEL)


The OAMP client itself. Three things to notice: it takes the `AGENT` connection (so OAMP-managed tables live in the `AGENT` schema), the in-DB ONNX embedder we registered in §3.4 (zero network calls for embedding), and an extraction LLM (§3.3) for auto-summary. `schema_policy="create_if_necessary"` lets OAMP own its DDL — we never write `CREATE TABLE` for memory tables.


In [ ]:
memory_client = OracleAgentMemory(
    connection=agent_conn,
    embedder=OracleONNXEmbedder(agent_conn),  # in-DB ONNX from §3.4
    llm=extraction_llm,                        # same chat provider/model as §3.3
    extract_memories=True,                     # mine durable facts from threads automatically
    schema_policy="create_if_necessary",
    table_name_prefix="eda_onnx_",
)

Every memory record OAMP stores carries a `user_id` (the operator/end-user) and an `agent_id` (which agent wrote it). Registering both up front means the rest of the notebook can reference them by stable strings — and in a multi-agent or multi-tenant deployment, they're how you partition memory.


In [ ]:
# Register the operator (user) and the agent. These IDs scope every memory.
USER_ID = "enterprise-operator"
AGENT_ID = "enterprise-data-agent"
for register_fn, eid, info in [
    (memory_client.add_user,  USER_ID,  "Operator querying the enterprise database in natural language."),
    (memory_client.add_agent, AGENT_ID, "Data agent grounded in scanned schema metadata."),
]:
    try:
        register_fn(eid, info)
        print(f"registered {eid}")
    except ValueError as e:
        if "already exists" in str(e):
            print(f"(already exists) {eid}")
        else:
            raise

OAMP holds the *content* of what the scanner learned. `scan_history` is the agent's *procedural* memory of what scans ran, when, on which schema, and to what effect. Plain SQL table — we query it by time/owner, not by meaning, so cosine search would buy nothing.


In [ ]:
# scan_history is the agent's PROCEDURAL memory of its own scans:
# *that* a scan ran, on what schema, when, and to what effect. Lives in
# a plain SQL table (not OAMP) because it's queried by time/owner rather
# than by meaning - cosine distance buys nothing here. The §12.1 incremental
# scan optimisation is the simplest thing this memory enables; richer
# policies (retry budgets, scan-cadence rules, "skip workload scan on this
# DB") fall out of the same shape.
with agent_conn.cursor() as cur:
    try:
        cur.execute(
            "CREATE TABLE scan_history ("
            "  scan_id          VARCHAR2(64) DEFAULT SYS_GUID() PRIMARY KEY,"
            "  target_owner     VARCHAR2(128) NOT NULL,"
            "  objects_scanned  NUMBER,"
            "  facts_written    NUMBER,"
            "  notes            VARCHAR2(4000),"
            "  started_at       TIMESTAMP DEFAULT CURRENT_TIMESTAMP,"
            "  finished_at      TIMESTAMP)"
        )
        print("created scan_history")
    except oracledb.DatabaseError as e:
        if e.args[0].code != 955:
            raise
        print("(already exists) scan_history")
agent_conn.commit()

print(f"Memory client ready (in-DB ONNX embedder + {LLM_PROVIDER}/{LLM_MODEL} extraction LLM).")


> ### 💡 Key takeaways — Part 4
>
> - **Don't hand-roll the memory schema.** OAMP gives you `memory` (durable facts), `thread` (conversations), and a `context_card` retrieval surface. Skipping it costs weeks of bookkeeping code that has nothing to do with the agent's actual job.
> - **Three memory kinds, one store.** Schema facts (`kind=table/column/...`), corrections (`kind=correction`), and tool outputs (`kind=tool_output`) coexist in the same OAMP store and are filtered at retrieve time. Use the `metadata.kind` tag, not separate tables.
> - **Procedural memory is different.** `scan_history` (when/how the agent ran) is queried by time and owner, not by meaning — keep it as a regular indexed table, not an OAMP memory. Cosine retrieval is wasted on bookkeeping.


## Part 5 — Retrieval strategies (and testing them)

*Part 4 set up the storage; Part 5 makes it useful. We mine Oracle's catalog views into facts, embed each fact via the in-DB ONNX model, and confirm the read path with a probe query. By the end of Part 5 the agent has institutional knowledge it can retrieve by meaning.*

Tables are storage. *Retrieval* is what makes them useful. The §5.x sub-sections below build the read-side machinery and test each piece in isolation; each step ends with a small probe so you can see the strategy working before we plug it into the agent loop.


### 5.2 The schema scanner — turning metadata into institutional knowledge

This is the heart of what makes our agent "enterprise-aware". The scanner reads Oracle's catalog views and converts each fact into a natural-language entry that goes into the OAMP memory store, embedded and ready for semantic retrieval.

We mine **five** sources:

1. **Structural** — `ALL_TABLES`, `ALL_TAB_COLUMNS`: names, types, nullability. *What shapes exist.*
2. **Annotation** — `ALL_TAB_COMMENTS`, `ALL_COL_COMMENTS`: human-written descriptions. *What a domain expert said.*
3. **Relational** — `ALL_CONSTRAINTS`, `ALL_CONS_COLUMNS`: primary keys, foreign keys. *How tables relate.*
4. **Code** — `ALL_SOURCE`, `ALL_VIEWS.TEXT`: definitions of views, packages, triggers. *How derived tables are built.*
5. **Workload** — `V$SQL`: sample of recent queries. *How the database is actually used — the "query inference" layer.*

We deliberately skip hints (`/*+ ... */`) from `V$SQL` for now — they're noisy on a fresh DB — but the scanner is easily extended to mine them.

> **Why store scanned facts as *text* with embeddings, not as normalized rows?** Because the agent retrieves by meaning, not by primary key. When the user asks "which table has the voyage manifest?" we want a cosine search over embedded descriptions to surface `SUPPLYCHAIN.CONTAINERS`, not a JOIN through four catalog views. The structured catalog data is still one SQL away via the `run_sql` tool — but the semantic layer on top is what gives us semantic context retrieval.


In [ ]:
import json
from dataclasses import dataclass


@dataclass
class Fact:
    kind: str # 'table' | 'column' | 'relationship' | 'query_pattern' | 'correction'
    subject: str
    body: str
    metadata: dict

##### Five scanner helpers, defined in the next five cells

The scanner is intentionally small — five focused functions, each mining one
catalog source and emitting `Fact` records:

| Function | Source | Fact `kind` |
|---|---|---|
| `_scan_tables`        | `ALL_TABLES` + `ALL_TAB_COMMENTS` | `table` |
| `_scan_columns`       | `ALL_TAB_COLUMNS` + `ALL_COL_COMMENTS` | `column` |
| `_scan_relationships` | `ALL_CONSTRAINTS` + `ALL_CONS_COLUMNS` | `relationship` |
| `_scan_workload`      | `V$SQL` (last 50 SELECTs touching the schema) | `query_pattern` |
| `scan_schema`         | orchestrator — calls all four, returns one list | n/a |

Each one is short enough that the docstring explains the rest. Read top to bottom
and watch how each turn of the catalog becomes one English sentence the embedder
can reason over.


In [ ]:
def _scan_tables(conn, owner: str) -> list[Fact]:
    sql = (
        "SELECT t.table_name, tc.comments, t.num_rows, t.last_analyzed "
        "  FROM all_tables t "
        "  LEFT JOIN all_tab_comments tc "
        "    ON tc.owner = t.owner AND tc.table_name = t.table_name "
        " WHERE t.owner = :owner "
        " ORDER BY t.table_name"
    )
    facts: list[Fact] = []
    with conn.cursor() as cur:
        cur.execute(sql, owner=owner.upper())
        for table, comment, num_rows, last_analyzed in cur:
            body_parts = [f"Table {owner}.{table}."]
            if comment:
                body_parts.append(f"Documented purpose: {comment}")
            if num_rows is not None:
                body_parts.append(f"Approximate row count: {num_rows:,}.")
            if last_analyzed:
                body_parts.append(f"Statistics last gathered at {last_analyzed}.")
            facts.append(Fact(
                kind="table",
                subject=f"{owner}.{table}",
                body=" ".join(body_parts),
                metadata={"owner": owner, "table": table,
                          "num_rows": num_rows,
                          "has_comment": bool(comment)},
            ))
    return facts

In [ ]:
def _scan_columns(conn, owner: str) -> list[Fact]:
    sql = (
        "SELECT c.table_name, c.column_name, c.data_type, c.data_length, "
        "       c.nullable, cc.comments "
        "  FROM all_tab_columns c "
        "  LEFT JOIN all_col_comments cc "
        "    ON cc.owner = c.owner AND cc.table_name = c.table_name "
        "   AND cc.column_name = c.column_name "
        " WHERE c.owner = :owner "
        " ORDER BY c.table_name, c.column_id"
    )
    facts: list[Fact] = []
    with conn.cursor() as cur:
        cur.execute(sql, owner=owner.upper())
        for table, col, dtype, dlen, nullable, comment in cur:
            dtype_str = dtype + (f"({dlen})" if dlen and dtype in ("VARCHAR2", "CHAR") else "")
            nullstr = "nullable" if nullable == "Y" else "NOT NULL"
            body = f"Column {owner}.{table}.{col} of type {dtype_str} ({nullstr})."
            if comment:
                body += f" Meaning: {comment}"
            facts.append(Fact(
                kind="column",
                subject=f"{owner}.{table}.{col}",
                body=body,
                metadata={"owner": owner, "table": table, "column": col,
                          "data_type": dtype, "nullable": nullable == "Y"},
            ))
    return facts

In [ ]:
def _scan_relationships(conn, owner: str) -> list[Fact]:
    sql = (
        "SELECT c.constraint_name, c.table_name, acc.column_name, "
        "       rc.table_name AS r_table, rcc.column_name AS r_column "
        "  FROM all_constraints c "
        "  JOIN all_cons_columns acc "
        "    ON acc.owner = c.owner AND acc.constraint_name = c.constraint_name "
        "  JOIN all_constraints rc "
        "    ON rc.owner = c.r_owner AND rc.constraint_name = c.r_constraint_name "
        "  JOIN all_cons_columns rcc "
        "    ON rcc.owner = rc.owner AND rcc.constraint_name = rc.constraint_name "
        "   AND rcc.position = acc.position "
        " WHERE c.owner = :owner AND c.constraint_type = 'R'"
    )
    facts: list[Fact] = []
    with conn.cursor() as cur:
        cur.execute(sql, owner=owner.upper())
        for cname, tbl, col, r_tbl, r_col in cur:
            facts.append(Fact(
                kind="relationship",
                subject=f"{owner}.{tbl}.{col}->{owner}.{r_tbl}.{r_col}",
                body=(f"Foreign key {cname}: {owner}.{tbl}.{col} references "
                      f"{owner}.{r_tbl}.{r_col}. Use this edge when joining the two tables."),
                metadata={"owner": owner, "child": tbl, "child_col": col,
                          "parent": r_tbl, "parent_col": r_col},
            ))
    return facts

In [ ]:
def _scan_workload(conn, owner: str, limit: int = 50) -> list[Fact]:
    sql = (
        "SELECT /*+ FIRST_ROWS(:lim) */ "
        "       sql_id, sql_fulltext, executions, rows_processed "
        "  FROM v$sql "
        " WHERE parsing_schema_name = :owner "
        "   AND command_type IN (3, 6, 7, 189) "
        "   AND rownum <= :lim "
        " ORDER BY executions DESC NULLS LAST"
    )
    facts: list[Fact] = []
    with conn.cursor() as cur:
        try:
            cur.execute(sql, owner=owner.upper(), lim=limit)
            for sql_id, text, execs, rows_proc in cur:
                if text is None:
                    continue
                stmt = (text.read() if hasattr(text, "read") else str(text)).strip()
                if len(stmt) > 2000:
                    stmt = stmt[:2000] + " /* ...truncated */"
                facts.append(Fact(
                    kind="query_pattern",
                    subject=f"v$sql:{sql_id}",
                    body=(f"A SQL statement observed in the workload for {owner} "
                          f"(executed {execs or 0} times, {rows_proc or 0} rows processed):\n{stmt}"),
                    metadata={"owner": owner, "sql_id": sql_id,
                              "executions": execs, "rows_processed": rows_proc},
                ))
        except oracledb.DatabaseError as e:
            print("  (workload scan skipped - V$SQL access:", e, ")")
    return facts

In [ ]:

def scan_schema(conn, owner: str) -> list[Fact]:
    """Mine every relevant catalog view and return one list of Facts."""
    facts = []
    facts += _scan_tables(conn, owner)
    facts += _scan_columns(conn, owner)
    facts += _scan_relationships(conn, owner)
    facts += _scan_workload(conn, owner)
    return facts


### 5.3 Writing facts into memory

The scanner produces a list of `Fact` objects (table, column, relationship, query pattern). Each becomes one memory record with `record_type="memory"` and the fact metadata attached. Re-scans are cheap because OAMP deduplicates on identical content; we still keep a `scan_history` row per scan for human-readable bookkeeping ("when did we last refresh `SALES`?").


In [ ]:
import hashlib
import uuid


def _hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:32]


def write_facts(facts: list[Fact], scan_id: str | None = None) -> tuple[int, int, int]:
    """Write scanned facts into OAMP memory. Returns (new, updated, skipped).

    Idempotency: each (kind, subject) pair has at most one current memory row.
    On re-scan we look up existing memories for the same subject; if the body is
    unchanged we skip the embedding call, otherwise we delete the stale row and
    insert the new one. OAMP handles the embedding internally on insert.
    """
    new = updated = skipped = 0
    scan_id = scan_id or str(uuid.uuid4())
    store = memory_client._store   # internal handle for metadata-filtered list()

    for f in facts:
        body_hash = _hash(f.body)
        # Exact lookup by (kind, subject) - no vector search, just metadata filter.
        existing = store.list(
            "memory",
            user_id=USER_ID, agent_id=AGENT_ID,
            metadata_filter={"kind": f.kind, "subject": f.subject},
            limit=1,
        )
        meta = {**f.metadata, "kind": f.kind, "subject": f.subject,
                "body_hash": body_hash, "scan_id": scan_id}

        if not existing:
            memory_client.add_memory(
                f.body,
                user_id=USER_ID, agent_id=AGENT_ID,
                metadata=meta,
            )
            new += 1
        elif (existing[0].metadata or {}).get("body_hash") == body_hash:
            skipped += 1
        else:
            memory_client.delete_memory(existing[0].id)
            memory_client.add_memory(
                f.body,
                user_id=USER_ID, agent_id=AGENT_ID,
                metadata=meta,
            )
            updated += 1
    return new, updated, skipped


def run_scan(conn, owner: str) -> dict:
    """Single-call entry point: scan a schema and persist the facts to OAMP."""
    scan_id = str(uuid.uuid4())
    with conn.cursor() as cur:
        cur.execute(
            "INSERT INTO scan_history (scan_id, target_owner, notes) VALUES (:id, :o, :n)",
            id=scan_id, o=owner.upper(), n="started",
        )
    conn.commit()

    facts = scan_schema(conn, owner)
    new, updated, skipped = write_facts(facts, scan_id=scan_id)

    with conn.cursor() as cur:
        cur.execute(
            "UPDATE scan_history "
            "   SET objects_scanned = :n, facts_written = :w, "
            "       finished_at = CURRENT_TIMESTAMP, notes = :notes "
            " WHERE scan_id = :id",
            n=len(facts), w=new + updated,
            notes=f"new={new} updated={updated} skipped={skipped}",
            id=scan_id)
    conn.commit()
    return {"scan_id": scan_id, "facts_total": len(facts),
            "new": new, "updated": updated, "skipped": skipped}


### 5.4 Try it: plant a small demo schema and scan it

So the rest of the notebook has something real to point at, we create a tiny `DEMO` schema with three tables and a foreign key. Then we run the scanner against it.


In [ ]:
# Pivot: replace the toy DEMO schema with a deeper supply-chain dataset
# (carriers, vessels, ports, voyages, current positions, containers, cargo).
# Spatial columns (SDO_GEOMETRY) on `ports.location` and
# `vessel_positions.position`, with R-tree spatial indexes for SDO_WITHIN_DISTANCE
# queries. Same `DEMO_USER` Python variable so downstream cells don't have to
# move; the actual schema in Oracle is `SUPPLYCHAIN`.

DEMO_USER = "SUPPLYCHAIN"
DEMO_PASS = "SupplyPwd_2025"

with sys_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM all_users WHERE username = :u", u=DEMO_USER)
    if cur.fetchone()[0] == 0:
        cur.execute(f"CREATE USER {DEMO_USER} IDENTIFIED BY {DEMO_PASS}")
    cur.execute(f"GRANT CONNECT, RESOURCE, UNLIMITED TABLESPACE TO {DEMO_USER}")
    cur.execute(f"GRANT CREATE VIEW, CREATE PROCEDURE, CREATE TYPE TO {DEMO_USER}")
    # Spatial: every user that creates SDO_GEOMETRY columns needs this grant.
    try:
        cur.execute(f"GRANT EXECUTE ON MDSYS.SDO_GEOMETRY TO {DEMO_USER}")
    except oracledb.DatabaseError:
        pass
    cur.execute(f"GRANT SELECT ANY TABLE TO {AGENT_USER}")
sys_conn.commit()

demo_conn = connect(DEMO_USER, DEMO_PASS, SYS_DSN)


# ---------- Drop existing objects (idempotent re-run) -----------------------
DROP_VIEWS = ["voyage_dv", "vessel_dv"]
for v in DROP_VIEWS:
    try:
        with demo_conn.cursor() as cur:
            cur.execute(f"DROP VIEW {v}")
    except oracledb.DatabaseError:
        pass

DROP_TABLES = ["cargo_items", "containers", "vessel_positions", "voyages",
               "vessels", "ports", "carriers"]
for t in DROP_TABLES:
    try:
        with demo_conn.cursor() as cur:
            cur.execute(f"DROP TABLE {t} CASCADE CONSTRAINTS PURGE")
    except oracledb.DatabaseError:
        pass


# ---------- DDL --------------------------------------------------------------
DDL = [
    # Carriers
    """CREATE TABLE carriers (
         carrier_id      NUMBER(10) PRIMARY KEY,
         name            VARCHAR2(120) NOT NULL,
         hq_country      VARCHAR2(60),
         founded_year    NUMBER(4)
       )""",

    # Ports — has SDO_GEOMETRY plus denormalised lat/lon for duality view ergonomics
    """CREATE TABLE ports (
         port_code       VARCHAR2(5) PRIMARY KEY,
         name            VARCHAR2(120) NOT NULL,
         country         VARCHAR2(60),
         ocean_region    VARCHAR2(20) NOT NULL,
         terminal_count  NUMBER(3),
         latitude        NUMBER(10,6),
         longitude       NUMBER(10,6),
         location        SDO_GEOMETRY
       )""",

    # Vessels
    """CREATE TABLE vessels (
         vessel_id       NUMBER(10) PRIMARY KEY,
         carrier_id      NUMBER(10) NOT NULL REFERENCES carriers(carrier_id),
         name            VARCHAR2(120) NOT NULL,
         imo_number      VARCHAR2(10) UNIQUE,
         vessel_type     VARCHAR2(20) NOT NULL,
         capacity_teu    NUMBER(7),
         year_built      NUMBER(4),
         flag_country    VARCHAR2(60)
       )""",

    # Voyages — region (ocean) drives the §14.4 row policy.
    """CREATE TABLE voyages (
         voyage_id       NUMBER(10) PRIMARY KEY,
         vessel_id       NUMBER(10) NOT NULL REFERENCES vessels(vessel_id),
         origin_code     VARCHAR2(5) NOT NULL REFERENCES ports(port_code),
         dest_code       VARCHAR2(5) NOT NULL REFERENCES ports(port_code),
         departure_ts    TIMESTAMP,
         eta_ts          TIMESTAMP,
         status          VARCHAR2(20) NOT NULL,
         ocean_region    VARCHAR2(20) NOT NULL
       )""",

    # Current vessel positions — 1:1 with vessels for active fleet
    """CREATE TABLE vessel_positions (
         vessel_id       NUMBER(10) PRIMARY KEY REFERENCES vessels(vessel_id),
         position_ts     TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
         latitude        NUMBER(10,6),
         longitude       NUMBER(11,6),
         speed_knots     NUMBER(5,2),
         heading_deg     NUMBER(5,2),
         position        SDO_GEOMETRY
       )""",

    # Containers — what's loaded, who's shipping it
    """CREATE TABLE containers (
         container_id    NUMBER(10) PRIMARY KEY,
         voyage_id       NUMBER(10) NOT NULL REFERENCES voyages(voyage_id),
         container_no    VARCHAR2(11) UNIQUE NOT NULL,
         container_type  VARCHAR2(20) NOT NULL,
         consignor       VARCHAR2(120),
         consignee       VARCHAR2(120),
         status          VARCHAR2(20) NOT NULL
       )""",

    # Cargo line items per container
    """CREATE TABLE cargo_items (
         cargo_item_id   NUMBER(10) PRIMARY KEY,
         container_id    NUMBER(10) NOT NULL REFERENCES containers(container_id),
         hs_code         VARCHAR2(10),
         description     VARCHAR2(400),
         quantity        NUMBER(10),
         unit_value_cents NUMBER(15),
         weight_kg       NUMBER(10,2)
       )""",

    # Comments — the scanner picks these up as institutional knowledge
    "COMMENT ON TABLE carriers IS 'Shipping lines / freight carriers operating the fleet.'",
    "COMMENT ON TABLE vessels IS 'Individual ships owned/operated by carriers; identified by IMO number.'",
    "COMMENT ON COLUMN vessels.capacity_teu IS 'Cargo capacity in 20-foot equivalent units (TEU); never tons.'",
    "COMMENT ON TABLE ports IS 'Container ports worldwide. location is SDO_GEOMETRY (WGS84, SRID 8307); use SDO_WITHIN_DISTANCE for proximity queries.'",
    "COMMENT ON TABLE voyages IS 'A specific journey of a vessel between two ports. ocean_region = PACIFIC | ATLANTIC | INDIAN | MEDITERRANEAN drives DDS row policies.'",
    "COMMENT ON TABLE vessel_positions IS 'Current AIS-style position for each active vessel. position is SDO_GEOMETRY (WGS84).'",
    "COMMENT ON TABLE containers IS 'Container manifest: which container is on which voyage. status: LOADED, IN_TRANSIT, DISCHARGED, CUSTOMS_HOLD.'",
    "COMMENT ON TABLE cargo_items IS 'Line items inside each container; HS code, description, value (in cents — never dollars).'",
    "COMMENT ON COLUMN cargo_items.unit_value_cents IS 'Per-unit declared value in USD CENTS. Multiply by quantity for line total.'",
]

with demo_conn.cursor() as cur:
    for stmt in DDL:
        cur.execute(stmt)
demo_conn.commit()


# ---------- Spatial metadata + indexes --------------------------------------
SPATIAL = [
    # Wipe any stale metadata from a prior schema
    "DELETE FROM USER_SDO_GEOM_METADATA WHERE table_name IN ('PORTS', 'VESSEL_POSITIONS')",
    """INSERT INTO USER_SDO_GEOM_METADATA (table_name, column_name, diminfo, srid)
       VALUES ('PORTS', 'LOCATION',
               SDO_DIM_ARRAY(
                 SDO_DIM_ELEMENT('LON', -180, 180, 0.005),
                 SDO_DIM_ELEMENT('LAT',  -90,  90, 0.005)
               ),
               8307)""",
    """INSERT INTO USER_SDO_GEOM_METADATA (table_name, column_name, diminfo, srid)
       VALUES ('VESSEL_POSITIONS', 'POSITION',
               SDO_DIM_ARRAY(
                 SDO_DIM_ELEMENT('LON', -180, 180, 0.005),
                 SDO_DIM_ELEMENT('LAT',  -90,  90, 0.005)
               ),
               8307)""",
    "CREATE INDEX ports_loc_sx ON ports(location) INDEXTYPE IS MDSYS.SPATIAL_INDEX_V2",
    "CREATE INDEX vpos_loc_sx  ON vessel_positions(position) INDEXTYPE IS MDSYS.SPATIAL_INDEX_V2",
]

with demo_conn.cursor() as cur:
    for stmt in SPATIAL:
        try:
            cur.execute(stmt)
        except oracledb.DatabaseError as e:
            # Idempotent: index/metadata might already exist on re-run
            if e.args[0].code in (955, 1408, 13223, 13226, 29855):
                continue
            raise
demo_conn.commit()


# ---------- Seed data --------------------------------------------------------
import random
random.seed(42)  # determinism — same seed every run

# 15 carriers (real-world container shipping lines)
carriers = [
    (1,  "Maersk",         "Denmark",       1904),
    (2,  "MSC",            "Switzerland",   1970),
    (3,  "CMA CGM",        "France",        1978),
    (4,  "COSCO",          "China",         1961),
    (5,  "Hapag-Lloyd",    "Germany",       1970),
    (6,  "ONE",            "Japan",         2017),
    (7,  "Evergreen",      "Taiwan",        1968),
    (8,  "Yang Ming",      "Taiwan",        1972),
    (9,  "HMM",            "South Korea",   1976),
    (10, "ZIM",            "Israel",        1945),
    (11, "OOCL",           "Hong Kong",     1969),
    (12, "Wan Hai",        "Taiwan",        1965),
    (13, "PIL",            "Singapore",     1967),
    (14, "Matson",         "USA",           1882),
    (15, "ANL",            "Australia",     1956),
]

# 25 ports — mix of largest/strategic global ports with realistic coords
# (lon, lat) ordering matches SDO_POINT_TYPE(X=lon, Y=lat).
ports = [
    # code,  name,                country,       region,           term,  lat,        lon
    ("SGSIN", "Singapore",         "Singapore",   "INDIAN",         62,    1.264900,  103.832600),
    ("CNSHA", "Shanghai",          "China",       "PACIFIC",        43,   31.235400,  121.473700),
    ("CNSHK", "Shenzhen Yantian",  "China",       "PACIFIC",        20,   22.564200,  114.272100),
    ("CNNGB", "Ningbo-Zhoushan",   "China",       "PACIFIC",        38,   29.868400,  121.836700),
    ("HKHKG", "Hong Kong",         "Hong Kong",   "PACIFIC",        24,   22.319300,  114.169400),
    ("KRPUS", "Busan",             "South Korea", "PACIFIC",        21,   35.102800,  129.040300),
    ("AEJEA", "Jebel Ali (Dubai)", "UAE",         "INDIAN",         15,   25.012300,   55.061700),
    ("MYTPP", "Tanjung Pelepas",   "Malaysia",    "INDIAN",         11,    1.366000,  103.560000),
    ("LKCMB", "Colombo",           "Sri Lanka",   "INDIAN",          5,    6.951900,   79.851800),
    ("OMSLL", "Salalah",           "Oman",        "INDIAN",          4,   17.018300,   54.092400),
    ("USLAX", "Los Angeles",       "USA",         "PACIFIC",        25,   33.732500, -118.262000),
    ("USLGB", "Long Beach",        "USA",         "PACIFIC",        22,   33.760100, -118.214800),
    ("USNYC", "New York/Newark",   "USA",         "ATLANTIC",       16,   40.683800,  -74.137300),
    ("USSAV", "Savannah",          "USA",         "ATLANTIC",        9,   32.083200,  -81.097100),
    ("USHOU", "Houston",           "USA",         "ATLANTIC",       10,   29.730400,  -95.246700),
    ("NLRTM", "Rotterdam",         "Netherlands", "ATLANTIC",       30,   51.924400,    4.477700),
    ("DEHAM", "Hamburg",           "Germany",     "ATLANTIC",       17,   53.551100,    9.993700),
    ("BEANR", "Antwerp",           "Belgium",     "ATLANTIC",       21,   51.221300,    4.401800),
    ("FRLEH", "Le Havre",          "France",      "ATLANTIC",       12,   49.490000,    0.107400),
    ("GBFXT", "Felixstowe",        "UK",          "ATLANTIC",        8,   51.961100,    1.351200),
    ("ESVLC", "Valencia",          "Spain",       "MEDITERRANEAN",  10,   39.469900,   -0.376300),
    ("ESALG", "Algeciras",         "Spain",       "MEDITERRANEAN",   7,   36.131500,   -5.453300),
    ("ITGOA", "Genoa",             "Italy",       "MEDITERRANEAN",   8,   44.405600,    8.946300),
    ("EGSCB", "Suez Canal South",  "Egypt",       "MEDITERRANEAN",   1,   29.967600,   32.549400),
    ("JPYOK", "Yokohama / Tokyo",  "Japan",       "PACIFIC",        18,   35.443900,  139.638100),
]

# 30 vessels distributed across carriers
vessel_types = ["CONTAINER", "CONTAINER", "CONTAINER", "CONTAINER",  # bias container
                "BULK", "TANKER", "RORO"]
flag_country = ["Denmark", "Liberia", "Panama", "Singapore", "Marshall Islands",
                "Hong Kong", "Malta", "Bahamas", "Cyprus"]
def _imo():
    return f"IMO{random.randint(9000000, 9999999)}"

vessels = []
vessel_names_by_carrier = {
    1:  ["Maersk Madrid", "Maersk Edinburgh", "Maersk Hong Kong", "Maersk Tukang"],
    2:  ["MSC Oscar", "MSC Gulsun", "MSC Eloane", "MSC Mia"],
    3:  ["CMA CGM Marco Polo", "CMA CGM Alexander", "CMA CGM Jacques Saade"],
    4:  ["COSCO Shipping Universe", "COSCO Pisces", "COSCO Cancer"],
    5:  ["Hapag-Lloyd Berlin Express", "Hapag-Lloyd Hamburg Express"],
    6:  ["ONE Apus", "ONE Stork"],
    7:  ["Ever Ace", "Ever Given", "Ever Globe"],
    8:  ["YM Witness", "YM Wreath"],
    9:  ["HMM Algeciras", "HMM Oslo"],
    10: ["ZIM Mount Everest", "ZIM Kingston"],
    11: ["OOCL Hong Kong", "OOCL Spain"],
    12: ["Wan Hai 805"],
    13: ["Kota Pekarang"],
    14: ["Matson Daniel K. Inouye"],
    15: ["ANL Wahroonga"],
}
vid = 1
for cid, names in vessel_names_by_carrier.items():
    for name in names:
        vtype = "CONTAINER" if "Express" in name or "MSC" in name or "CMA" in name or "COSCO" in name or "OOCL" in name or "Maersk" in name or "HMM" in name or "ZIM" in name or "Ever" in name or "YM" in name or "ONE" in name or "Wan Hai" in name or "Matson" in name or "Kota" in name or "ANL" in name else random.choice(vessel_types)
        capacity = random.choice([5500, 8800, 11000, 14000, 18000, 20000, 23000, 24000])
        year = random.choice([2014, 2015, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024])
        flag = random.choice(flag_country)
        vessels.append((vid, cid, name, _imo(), vtype, capacity, year, flag))
        vid += 1

# 60 voyages — one each per active route, status mix
voyage_statuses = ["ACTIVE"] * 30 + ["DELAYED"] * 10 + ["SCHEDULED"] * 10 + ["COMPLETED"] * 10

# Map ports to the ocean region a voyage between them will be tagged with.
# When origin/destination span regions, the voyage takes the *transit* region
# (the ocean it's sailing through) — close enough for the demo's row policy.
def _voyage_region(origin_port, dest_port):
    op_region = next(p[3] for p in ports if p[0] == origin_port)
    dp_region = next(p[3] for p in ports if p[0] == dest_port)
    if op_region == dp_region:
        return op_region
    # Cross-ocean — pick the dominant transit ocean
    pairs = {
        ("PACIFIC", "ATLANTIC"): "PACIFIC",   # Asia → US east via Panama
        ("ATLANTIC", "PACIFIC"): "ATLANTIC",
        ("INDIAN", "MEDITERRANEAN"): "INDIAN",
        ("MEDITERRANEAN", "INDIAN"): "MEDITERRANEAN",
        ("PACIFIC", "INDIAN"): "PACIFIC",
        ("INDIAN", "PACIFIC"): "INDIAN",
        ("ATLANTIC", "MEDITERRANEAN"): "ATLANTIC",
        ("MEDITERRANEAN", "ATLANTIC"): "MEDITERRANEAN",
        ("ATLANTIC", "INDIAN"): "INDIAN",
        ("INDIAN", "ATLANTIC"): "INDIAN",
        ("PACIFIC", "MEDITERRANEAN"): "PACIFIC",
        ("MEDITERRANEAN", "PACIFIC"): "MEDITERRANEAN",
    }
    return pairs.get((op_region, dp_region), op_region)

voyages = []
import datetime as _dt
now = _dt.datetime.utcnow()
random.shuffle(voyage_statuses)
for vid_i in range(60):
    voyage_id = vid_i + 1
    vessel_id = (vid_i % len(vessels)) + 1
    o = random.choice(ports)[0]
    d = random.choice([p for p in ports if p[0] != o])[0]
    status = voyage_statuses[vid_i]
    # Times relative to now
    if status == "SCHEDULED":
        dep = now + _dt.timedelta(days=random.randint(2, 30))
        eta = dep + _dt.timedelta(days=random.randint(7, 35))
    elif status == "COMPLETED":
        dep = now - _dt.timedelta(days=random.randint(40, 90))
        eta = dep + _dt.timedelta(days=random.randint(7, 35))
    elif status == "DELAYED":
        dep = now - _dt.timedelta(days=random.randint(10, 30))
        eta = now + _dt.timedelta(days=random.randint(3, 14))
    else:  # ACTIVE
        dep = now - _dt.timedelta(days=random.randint(2, 20))
        eta = now + _dt.timedelta(days=random.randint(5, 25))
    region = _voyage_region(o, d)
    voyages.append((voyage_id, vessel_id, o, d,
                    dep, eta, status, region))

# Vessel positions — one per ACTIVE/DELAYED voyage's vessel.
# Position is interpolated between origin and destination (rough midpoint with jitter).
ports_by_code = {p[0]: p for p in ports}
vessel_positions = []
seen_vessels = set()
for vy in voyages:
    voyage_id, vessel_id, o_code, d_code, dep, eta, status, region = vy
    if status not in ("ACTIVE", "DELAYED"):
        continue
    if vessel_id in seen_vessels:
        continue
    seen_vessels.add(vessel_id)
    op = ports_by_code[o_code]
    dp = ports_by_code[d_code]
    # Linear interpolation 30-70% of the way, with small lat/lon jitter
    t = random.uniform(0.30, 0.70)
    lat = op[5] + (dp[5] - op[5]) * t + random.uniform(-1.5, 1.5)
    lon = op[6] + (dp[6] - op[6]) * t + random.uniform(-1.5, 1.5)
    speed = round(random.uniform(12.0, 22.5), 2)
    heading = round(random.uniform(0, 360), 2)
    vessel_positions.append((vessel_id, lat, lon, speed, heading))

# Containers + cargo items
container_types = ["DRY", "DRY", "DRY", "REEFER", "HAZMAT", "OPEN_TOP", "DRY"]
container_statuses = ["LOADED", "IN_TRANSIT", "DISCHARGED", "CUSTOMS_HOLD"]
consignors = [
    "Foxconn Technology Group", "Bosch Automotive", "Samsung SDI",
    "Tesla Inc.", "Volkswagen AG", "Toyota Motor Corp.",
    "Lenovo Group", "Apple Inc.", "Nestle SA", "IKEA",
    "Maersk Logistics", "Caterpillar Inc.", "BASF SE",
    "Pfizer Inc.", "Glencore Agriculture",
]
consignees = [
    "Walmart Inc.", "Costco Wholesale", "Amazon.com Services",
    "Best Buy", "Home Depot", "Target Corp.",
    "Carrefour", "Tesco Stores", "Aldi Nord",
    "El Corte Ingles", "Loblaws", "Coles Group",
    "Sumitomo Electric Industries", "Pemex", "Petrobras",
]
cargo_catalog = [
    # (hs_code, description, base_value_cents, base_weight_kg, container_type)
    ("8517", "Smartphones, retail-packed", 28000, 0.18, "DRY"),
    ("8528", "LED televisions, 50-inch class", 32000, 18.5, "DRY"),
    ("8703", "Passenger automobiles", 2800000, 1500, "RORO"),
    ("8714", "Bicycle parts and accessories", 1200, 1.4, "DRY"),
    ("8419", "Industrial heat exchangers", 580000, 480, "OPEN_TOP"),
    ("3004", "Pharmaceuticals, packaged", 45000, 0.5, "DRY"),
    ("3004", "Pharmaceuticals, refrigerated", 89000, 0.5, "REEFER"),
    ("0303", "Frozen seafood, salmon", 4500, 1.0, "REEFER"),
    ("0810", "Fresh berries", 2800, 1.0, "REEFER"),
    ("0901", "Roasted coffee beans, bulk", 1800, 1.0, "DRY"),
    ("8506", "Lithium-ion battery cells (HAZ class 9)", 8500, 0.4, "HAZMAT"),
    ("3105", "Fertilizer, ammonium nitrate", 850, 25, "HAZMAT"),
    ("2710", "Petroleum lubricants, drums", 1900, 200, "HAZMAT"),
    ("9018", "Medical instruments and apparatus", 240000, 4.2, "DRY"),
    ("8471", "Personal computers and laptops", 95000, 2.3, "DRY"),
    ("9403", "Office furniture, knock-down", 12000, 35, "DRY"),
    ("4202", "Designer luggage and handbags", 35000, 1.8, "DRY"),
    ("6203", "Men's apparel, woven", 4500, 0.4, "DRY"),
    ("6204", "Women's apparel, woven", 5200, 0.4, "DRY"),
    ("8482", "Industrial ball bearings, assorted", 1900, 0.6, "DRY"),
    ("7308", "Steel structures, prefab", 32000, 200, "OPEN_TOP"),
    ("4011", "Pneumatic tires, passenger car", 9500, 8.5, "DRY"),
    ("8407", "Marine engines, spare parts", 165000, 75, "DRY"),
    ("8501", "Industrial electric motors", 92000, 110, "DRY"),
    ("3923", "Plastic articles for packing", 350, 0.05, "DRY"),
    ("4818", "Toilet paper and tissues, bulk", 280, 0.4, "DRY"),
    ("8504", "Power transformers", 145000, 320, "OPEN_TOP"),
    ("9405", "LED lighting fixtures", 5500, 1.2, "DRY"),
    ("0808", "Apples, fresh", 180, 0.18, "REEFER"),
    ("8703", "Electric vehicles, complete", 4200000, 2200, "RORO"),
]

containers = []
cargo_items = []
container_id = 1
cargo_id = 1
for v in voyages:
    voyage_id, vessel_id, _, _, _, _, status, _ = v
    if status == "COMPLETED":
        n_containers = random.randint(0, 2)
    elif status == "SCHEDULED":
        n_containers = random.randint(1, 3)
    else:
        n_containers = random.randint(2, 5)
    for _ in range(n_containers):
        ctype = random.choice(container_types)
        cstatus = random.choice(container_statuses)
        cno = f"MSCU{random.randint(1000000, 9999999)}"
        consignor = random.choice(consignors)
        consignee = random.choice(consignees)
        containers.append((container_id, voyage_id, cno, ctype, consignor, consignee, cstatus))
        # 1-3 cargo items per container, type-matched
        type_matched = [c for c in cargo_catalog if c[4] == ctype]
        pool = type_matched if type_matched else cargo_catalog
        for _ in range(random.randint(1, 3)):
            hs, desc, val, wt, _ctype = random.choice(pool)
            qty = random.choice([10, 25, 50, 100, 200, 500, 1000])
            cargo_items.append((cargo_id, container_id, hs, desc, qty,
                                int(val * random.uniform(0.85, 1.15)),
                                round(wt * qty, 2)))
            cargo_id += 1
        container_id += 1


# ---------- Inserts ---------------------------------------------------------
with demo_conn.cursor() as cur:
    cur.executemany(
        "INSERT INTO carriers (carrier_id, name, hq_country, founded_year) "
        "VALUES (:1, :2, :3, :4)",
        carriers,
    )

    # Named binds: SDO_POINT_TYPE references :lat/:lon a second time, and
    # the thin driver counts every reference for positional binds (DPY-4009).
    # Switching to dict-style named binds deduplicates by name.
    cur.executemany(
        "INSERT INTO ports (port_code, name, country, ocean_region, terminal_count, "
        "                   latitude, longitude, location) "
        "VALUES (:code, :name, :country, :region, :terminals, :lat, :lon, "
        "        SDO_GEOMETRY(2001, 8307, SDO_POINT_TYPE(:lon, :lat, NULL), NULL, NULL))",
        [
            {"code": p[0], "name": p[1], "country": p[2], "region": p[3],
             "terminals": p[4], "lat": p[5], "lon": p[6]}
            for p in ports
        ],
    )

    cur.executemany(
        "INSERT INTO vessels (vessel_id, carrier_id, name, imo_number, vessel_type, "
        "                     capacity_teu, year_built, flag_country) "
        "VALUES (:1, :2, :3, :4, :5, :6, :7, :8)",
        vessels,
    )

    cur.executemany(
        "INSERT INTO voyages (voyage_id, vessel_id, origin_code, dest_code, "
        "                     departure_ts, eta_ts, status, ocean_region) "
        "VALUES (:1, :2, :3, :4, :5, :6, :7, :8)",
        voyages,
    )

    cur.executemany(
        "INSERT INTO vessel_positions (vessel_id, latitude, longitude, "
        "                              speed_knots, heading_deg, position) "
        "VALUES (:vid, :lat, :lon, :speed, :heading, "
        "        SDO_GEOMETRY(2001, 8307, SDO_POINT_TYPE(:lon, :lat, NULL), NULL, NULL))",
        [
            {"vid": p[0], "lat": p[1], "lon": p[2], "speed": p[3], "heading": p[4]}
            for p in vessel_positions
        ],
    )

    cur.executemany(
        "INSERT INTO containers (container_id, voyage_id, container_no, container_type, "
        "                        consignor, consignee, status) "
        "VALUES (:1, :2, :3, :4, :5, :6, :7)",
        containers,
    )

    cur.executemany(
        "INSERT INTO cargo_items (cargo_item_id, container_id, hs_code, description, "
        "                         quantity, unit_value_cents, weight_kg) "
        "VALUES (:1, :2, :3, :4, :5, :6, :7)",
        cargo_items,
    )

demo_conn.commit()

print(f"SUPPLYCHAIN seeded:")
print(f"  carriers:         {len(carriers):4d}")
print(f"  ports:            {len(ports):4d}  (with SDO_GEOMETRY + spatial index)")
print(f"  vessels:          {len(vessels):4d}")
print(f"  voyages:          {len(voyages):4d}")
print(f"  vessel_positions: {len(vessel_positions):4d}  (with SDO_GEOMETRY + spatial index)")
print(f"  containers:       {len(containers):4d}")
print(f"  cargo_items:      {len(cargo_items):4d}")


First real scanner invocation against the `DEMO` schema we just planted. Walks `ALL_TABLES`, `ALL_TAB_COLUMNS`, `ALL_CONSTRAINTS`, `V$SQL`, turns each finding into a `Fact`, runs the scanner's idempotent upsert. The summary tells you new vs. updated vs. skipped rows.


In [ ]:
summary = run_scan(agent_conn, owner=DEMO_USER)
print(json.dumps(summary, indent=2))

### 5.5 Retrieve by meaning

`retrieve_knowledge` is a two-stage call:

1. **Cosine search** — `memory_client.search` returns `k * 4` candidates ordered by vector distance over `eda_onnx_*` memory embeddings.
2. **Cross-encoder rerank** — those candidates run through `rerank(...)`, which calls `DBMS_VECTOR.RERANK` server-side when an Oracle ONNX reranker is loaded (§3.5). When no reranker is loaded, `rerank` is a pass-through that just slices to top-k — the call site is unchanged either way.

Tool-output memories (kind `tool_output`) are filtered out before reranking so the log of past tool calls doesn't pollute knowledge retrieval.


In [ ]:
def retrieve_knowledge(query: str, k: int = 5,
                        kinds: list[str] | None = None) -> list[dict]:
    """Semantic search over the agent's long-term memory.

    OAMP's search() doesn't take a metadata_filter, so we ask for a wider window
    when a kind filter is set and trim it client-side. For unfiltered queries
    that's a single round-trip; for filtered ones we over-fetch by 4x as a cheap
    way to keep top-k recall acceptable.
    """
    # Stage 1: oversample from the cosine search so the reranker has a
    # genuine candidate set to chew on. ~4x the requested k is a fine default.
    # Defensive: LLMs sometimes pass `kinds` as a comma-separated string
    # ("table,column") instead of a list. Coerce so the membership check works.
    if isinstance(kinds, str):
        kinds = [s.strip() for s in kinds.split(",") if s.strip()]
    if kinds is not None and not isinstance(kinds, list):
        kinds = None
    if kinds == []:
        kinds = None

    cosine_fetch = k * 4
    hits = memory_client.search(
        query,
        user_id=USER_ID, agent_id=AGENT_ID,
        record_types=["memory"],
        max_results=cosine_fetch,
    )

    candidates: list[dict] = []
    for h in hits:
        meta = h.metadata or {}
        kind_value = meta.get("kind")
        if kind_value == "tool_output":
            continue
        # Only filter when kinds is set AND the candidate has a kind to match
        # (avoids `None not in <kinds>` -> TypeError on missing-kind memories).
        if kinds is not None and (kind_value is None or kind_value not in kinds):
            continue
        candidates.append({
            "kind":     meta.get("kind", "?"),
            "subject":  meta.get("subject", ""),
            "body":     h.content,
            "metadata": meta,
            "distance": float(h.distance),
        })

    # Stage 2: cross-encoder rerank (in-DB if a reranker is loaded; otherwise
    # passthrough). Always called - keeps the call site uniform.
    return rerank(query, candidates, top_k=k, content_key="body")



### 5.6 Hybrid retrieval — vector + Oracle Text fused with RRF

§5.5's vector retrieval is great at *meaning*: ask "where do we record vessel current positions?" and the agent finds `SUPPLYCHAIN.VESSEL_POSITIONS` even when the words don't overlap. But pure vector search has a known weakness — it under-weights **exact tokens**. If the user types "TEU" or an `ORA-00904` error code, the agent wants to find the row that *literally contains that string*, not the one that's vaguely similar to it.

The fix is to run a **second** retrieval — a full-text search over the same content — and fuse the two ranked lists. The classic fusion algorithm is **Reciprocal Rank Fusion (RRF)**:

$$\text{score}(d) = \frac{1}{k + r_{\text{vec}}(d)} + \frac{1}{k + r_{\text{text}}(d)}$$

where $r_{\text{vec}}$ and $r_{\text{text}}$ are 1-based ranks from each retriever (with a sentinel large value when a doc is missing from one list), and $k=60$ is the standard smoothing constant. RRF doesn't care about the absolute scores from each retriever — only the relative ranks — so it's robust to whatever scoring scheme each side uses.

Two prerequisites — both already in this Oracle:

| Side | What we use | Where |
|---|---|---|
| **Vector** | `VECTOR_DISTANCE(c.embedding, VECTOR_EMBEDDING(...) USING :q AS DATA, COSINE)` over `eda_onnx_record_chunks.embedding` (HNSW-indexed) | the OAMP memory store, populated by §5.3 every time `add_memory` runs |
| **Full-text** | `CONTAINS(m.content, :kw, 1) > 0` with `SCORE(1)` over a `CTXSYS.CONTEXT` index on `eda_onnx_memory.content` | the next cell creates the index |

The fusion happens in **one SQL statement** — vector CTE, text CTE, FULL OUTER JOIN on memory id, RRF score computed inline, sort + fetch top-k. No round trip to Python. Same `agent_conn`. Same trust boundary.


In [ ]:
# Create the Oracle Text index on the OAMP memory body. Idempotent: drops any
# prior index of the same name first. SYNC (ON COMMIT) keeps the index fresh
# whenever new memories land via add_memory — no separate refresh job needed.
TEXT_INDEX_NAME = "eda_memory_text_idx"
MEMORY_TABLE    = "eda_onnx_memory"

with agent_conn.cursor() as cur:
    try:
        cur.execute(f"DROP INDEX {TEXT_INDEX_NAME}")
        print(f"dropped existing {TEXT_INDEX_NAME}")
    except oracledb.DatabaseError as e:
        if e.args[0].code != 1418:  # ORA-01418 = index doesn't exist
            raise
    try:
        cur.execute(
            f"CREATE INDEX {TEXT_INDEX_NAME} "
            f"  ON {MEMORY_TABLE}(content) "
            f"  INDEXTYPE IS CTXSYS.CONTEXT "
            f"  PARAMETERS ('SYNC (ON COMMIT)')"
        )
        print(f"created Oracle Text index {TEXT_INDEX_NAME} on {MEMORY_TABLE}.content")
    except oracledb.DatabaseError as e:
        code_ = e.args[0].code
        if code_ == 29855:
            # Sometimes ON COMMIT isn't supported on this build — fall back to manual sync.
            cur.execute(
                f"CREATE INDEX {TEXT_INDEX_NAME} "
                f"  ON {MEMORY_TABLE}(content) "
                f"  INDEXTYPE IS CTXSYS.CONTEXT"
            )
            print(f"created {TEXT_INDEX_NAME} (without SYNC ON COMMIT — call ctx_ddl.sync_index "
                  f"manually if memories are added later)")
        elif code_ in (1031, 942):
            print(f"!! cannot create CTXSYS.CONTEXT index (ORA-{code_:05d}). "
                  f"AGENT user needs CTXAPP role: GRANT CTXAPP TO {AGENT_USER};")
        else:
            raise

agent_conn.commit()


In [ ]:
def keyword_search_memories(query: str, k: int = 5) -> list[dict]:
    """Full-text search over OAMP memories using the Oracle Text index from §5.6.
    Returns top-k rows by SCORE(1) — Oracle Text's relevance metric (an
    okapi-BM25-style ranking).

    Tool-output kind memories are filtered out so the search surfaces durable
    facts and corrections, matching the behaviour of search_knowledge.
    """
    sql = f"""
        SELECT
            m.record_id,
            m.metadata,
            DBMS_LOB.SUBSTR(m.content, 4000, 1) AS content,
            SCORE(1) AS score_txt
          FROM {MEMORY_TABLE} m
         WHERE CONTAINS(m.content, :kw, 1) > 0
           AND m.user_id  = :u
           AND m.agent_id = :a
           AND (JSON_VALUE(m.metadata, '$.kind') IS NULL
                OR JSON_VALUE(m.metadata, '$.kind') <> 'tool_output')
         ORDER BY SCORE(1) DESC
         FETCH FIRST :k ROWS ONLY
    """
    with agent_conn.cursor() as cur:
        cur.execute(sql, kw=query, u=USER_ID, a=AGENT_ID, k=k)
        rows = []
        for record_id, metadata, content, score in cur:
            meta = metadata or {}
            if hasattr(content, "read"):
                content = content.read()
            rows.append({
                "record_id": record_id,
                "kind": (meta or {}).get("kind", "memory"),
                "subject": (meta or {}).get("subject", ""),
                "content": str(content or "")[:500],
                "score_txt": float(score) if score is not None else 0.0,
            })
    return rows


# Smoke test — should find the cents correction by literal keyword match
print("keyword_search_memories(\"TEU\"):")
for h in keyword_search_memories("TEU", k=3):
    print(f"  [{h['score_txt']:6.2f}] {h['kind']:12s} {h['subject'][:40]:40s}  {h['content'][:80]}")


In [ ]:
import math

def hybrid_rrf_search_memories(
    query: str,
    k: int = 5,
    per_list: int = 30,   # how many candidates from each retriever before fusion
    rrf_k: int = 60,      # RRF smoothing constant — 60 is the canonical value
) -> list[dict]:
    """Hybrid retrieval over OAMP memories: vector (semantic) + Oracle Text
    (lexical), fused via Reciprocal Rank Fusion.

    Issued as one SQL statement so the JOIN happens server-side. Both retrievers
    use the same `user_id` / `agent_id` scope and skip tool_output memories.
    """
    sql = f"""
        WITH q_emb AS (
            -- OAMP stores embeddings as FLOAT64; the ONNX model returns FLOAT32.
            -- Coerce the query vector to FLOAT64 so VECTOR_DISTANCE doesn't
            -- raise ORA-51812 (mismatched dimension formats).
            SELECT TO_VECTOR(VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :q AS DATA), 384, FLOAT64) AS emb
              FROM dual
        ),
        vec AS (
            SELECT
                m.record_id,
                DBMS_LOB.SUBSTR(m.content, 4000, 1) AS content,
                m.metadata,
                1 - VECTOR_DISTANCE(c.embedding, q_emb.emb, COSINE) AS sim_vec,
                ROW_NUMBER() OVER (
                    ORDER BY VECTOR_DISTANCE(c.embedding, q_emb.emb, COSINE)
                ) AS r_vec
              FROM {MEMORY_TABLE} m
              JOIN eda_onnx_record_chunks c ON c.source_id = m.record_id
              CROSS JOIN q_emb
             WHERE m.user_id  = :u
               AND m.agent_id = :a
               AND (JSON_VALUE(m.metadata, '$.kind') IS NULL
                    OR JSON_VALUE(m.metadata, '$.kind') <> 'tool_output')
             FETCH FIRST :n ROWS ONLY
        ),
        txt AS (
            SELECT
                m.record_id,
                DBMS_LOB.SUBSTR(m.content, 4000, 1) AS content,
                m.metadata,
                SCORE(1) AS score_txt,
                ROW_NUMBER() OVER (ORDER BY SCORE(1) DESC) AS r_txt
              FROM {MEMORY_TABLE} m
             WHERE CONTAINS(m.content, :kw, 1) > 0
               AND m.user_id  = :u
               AND m.agent_id = :a
               AND (JSON_VALUE(m.metadata, '$.kind') IS NULL
                    OR JSON_VALUE(m.metadata, '$.kind') <> 'tool_output')
             FETCH FIRST :n ROWS ONLY
        ),
        fused AS (
            SELECT
                COALESCE(v.record_id, t.record_id)            AS record_id,
                COALESCE(v.content, t.content)                AS content,
                COALESCE(v.metadata, t.metadata)              AS metadata,
                NVL(v.sim_vec, 0)                             AS sim_vec,
                NVL(t.score_txt, 0)                           AS score_txt,
                NVL(v.r_vec, 999999)                          AS r_vec,
                NVL(t.r_txt, 999999)                          AS r_txt,
                ( 1.0 / (:rrf_k + NVL(v.r_vec, 999999))
                + 1.0 / (:rrf_k + NVL(t.r_txt, 999999)) )      AS rrf_score
              FROM vec v
              FULL OUTER JOIN txt t ON v.record_id = t.record_id
        )
        SELECT * FROM fused
         ORDER BY rrf_score DESC
         FETCH FIRST :k ROWS ONLY
    """

    with agent_conn.cursor() as cur:
        # Phrase-quote multi-word queries so Oracle Text treats them as a phrase
        # rather than a boolean expression.
        kw = f'"{query}"' if " " in query.strip() else query
        cur.execute(sql, q=query, kw=kw, u=USER_ID, a=AGENT_ID,
                    n=per_list, rrf_k=rrf_k, k=k)
        rows = []
        for rec_id, content, meta, sim_vec, score_txt, r_vec, r_txt, rrf in cur:
            if hasattr(content, "read"):
                content = content.read()
            rows.append({
                "record_id": rec_id,
                "kind": (meta or {}).get("kind", "memory"),
                "subject": (meta or {}).get("subject", ""),
                "content": str(content or "")[:500],
                "sim_vec":   float(sim_vec)   if sim_vec   is not None else 0.0,
                "score_txt": float(score_txt) if score_txt is not None else 0.0,
                "r_vec":     int(r_vec),
                "r_txt":     int(r_txt),
                "rrf_score": float(rrf),
            })
    return rows


print("hybrid_rrf_search_memories(\"TEU capacity unit\"):")
for h in hybrid_rrf_search_memories("TEU capacity unit", k=3):
    print(f"  rrf={h['rrf_score']:.4f} (r_vec={h['r_vec']:>3}, r_txt={h['r_txt']:>3})  "
          f"[{h['kind']:12s}] {h['subject'][:40]}")
    print(f"      {h['content'][:150]}")


In [ ]:
# Three-way comparison: same query, three retrievers.
# A query that mixes a precise token ("TEU") with a fuzzier semantic concept
# ("how vessel capacity is measured") gives RRF something to do — vector and
# keyword should disagree on the top result, and fusion should rank the
# best-matching memory above either alone.

probe_q = "TEU 20-foot equivalent capacity unit for vessels"

print("=" * 80)
print(f"QUERY: {probe_q!r}")
print("=" * 80)

print("\n--- A) VECTOR ONLY (search_knowledge, §5.5) ---")
for h in retrieve_knowledge(probe_q, k=3):
    subject = (h.get("metadata") or {}).get("subject", "")
    print(f"  [{h.get('kind','memory'):12s}] {subject[:40]:40s}  {h['body'][:100]}")

print("\n--- B) KEYWORD ONLY (Oracle Text CONTAINS + SCORE) ---")
for h in keyword_search_memories(probe_q, k=3):
    print(f"  [{h['kind']:12s}] score={h['score_txt']:6.2f}  {h['subject'][:40]:40s}  {h['content'][:100]}")

print("\n--- C) HYBRID via RRF ---")
for h in hybrid_rrf_search_memories(probe_q, k=3):
    print(f"  [{h['kind']:12s}] rrf={h['rrf_score']:.4f}  "
          f"r_vec={h['r_vec']:>3}  r_txt={h['r_txt']:>3}  "
          f"{h['subject'][:40]:40s}  {h['content'][:100]}")

print("\n" + "=" * 80)
print("Note the r_vec / r_txt columns in the hybrid output: a row whose r_vec is")
print("low (top of the vector list) BUT r_txt is 999999 (missing from the keyword")
print("list) still gets a fair RRF score from its vector half — and vice versa.")
print("Memories that show up in BOTH lists get the highest combined score.")
print("=" * 80)


> ### 💡 Key takeaways — Part 5
>
> - **Catalog views are training data.** `ALL_TABLES` / `ALL_TAB_COLUMNS` / `ALL_CONSTRAINTS` / `V$SQL` mined into prose facts is how you teach an agent your schema without fine-tuning a model.
> - **Vector search alone misses exact tokens.** A user typing "TEU" or `ORA-00904` wants the row that *literally contains the string*. Pure cosine retrieval underweights this. Hybrid (vector + Oracle Text via RRF) closes the gap server-side in one SQL.
> - **`body_hash` makes re-scans free.** The scanner only re-embeds facts whose underlying text changed. Hourly re-scans become viable when the dedup is content-based, not time-based.
> - **Reranking is one SQL primitive.** `DBMS_VECTOR.RERANK` runs the cross-encoder server-side; the call gracefully degrades to cosine ordering when no reranker is loaded. No Python detour, no second service.


## Part 7 — DBFS: the short-term scratchpad

*Part 6 closed out durable memory. Part 7 adds the missing piece — a file-like scratchpad for mid-task drafts (SQL the agent will edit before running, intermediate calculations) that doesn't deserve a row in the long-term store.*

DBFS is Oracle's POSIX-like filesystem layered on SecureFile LOBs in a table. The agent sees files and directories; the database sees rows. Same backups, same audit, same security model as everything else in §4 — but with `open()`/`read()`/`write()` ergonomics.

Why a filesystem at all when we already have tables?

1. **Mid-task scratch.** "Write a SQL draft, read it back, edit, run it" is a filesystem workload, not OLTP.
2. **Path-addressable handles.** Tools can pass `/scratch/draft.sql` between calls without serializing a row id.
3. **Cheap rewrite.** Overwriting a CLOB row works but isn't idiomatic; DBFS gives you the file semantics directly.

We use DBFS only as the agent's scratchpad — the long-term, search-heavy data stays in §4's tables.


### 7.1 Create a DBFS filesystem owned by `AGENT`

One-time setup: create a tablespace (one datafile), then create a DBFS store inside it. `DBMS_DBFS_SFS.CREATEFILESYSTEM` creates the backing table; `DBMS_DBFS_CONTENT.REGISTERSTORE` + `MOUNTSTORE` register it under a path the agent will use.


In [ ]:
import os.path

DBFS_TABLESPACE = "AGENT_DBFS_TS"
DBFS_STORE      = "AGENT_SCRATCH"
DBFS_MOUNT      = "/scratch"

with sys_conn.cursor() as cur:
    # Tablespace (idempotent guard)
    cur.execute("SELECT COUNT(*) FROM dba_tablespaces WHERE tablespace_name = :n",
                n=DBFS_TABLESPACE)
    if cur.fetchone()[0] == 0:
        # `DATAFILE SIZE ...` (with no path) only works when DB_CREATE_FILE_DEST
        # is set (Oracle Managed Files). For Free in Docker it usually isn't,
        # so we derive the datafile directory from an existing tablespace and
        # name a file ourselves.
        cur.execute("SELECT name FROM v$datafile WHERE rownum = 1")
        sample_path = cur.fetchone()[0]
        dbfs_dir = os.path.dirname(sample_path)
        dbfs_path = f"{dbfs_dir}/agent_dbfs01.dbf"
        cur.execute(
            f"CREATE TABLESPACE {DBFS_TABLESPACE} "
            f"DATAFILE '{dbfs_path}' SIZE 100M AUTOEXTEND ON NEXT 50M MAXSIZE 2G")
        print(f"created tablespace {DBFS_TABLESPACE} at {dbfs_path}")
    else:
        print(f"tablespace {DBFS_TABLESPACE} already exists")

    cur.execute(f"ALTER USER {AGENT_USER} QUOTA UNLIMITED ON {DBFS_TABLESPACE}")
sys_conn.commit()


DBFS isn't ready until you create a tablespace, create the filesystem inside it, register a content store, and mount it. All four steps in this cell, idempotent. After it runs, `/scratch` is a real path inside the database that supports `read`/`write` semantics on top of SecureFile LOBs.


In [ ]:
# Create and mount the DBFS store under the AGENT user.
# We tolerate "already exists" style errors so the cell is re-runnable.

_create_fs = (
    "BEGIN "
    "  DBMS_DBFS_SFS.CREATEFILESYSTEM("
    f"    store_name => '{DBFS_STORE}',"
    f"    tbl_name   => '{DBFS_STORE}_T',"
    f"    tbl_tbs    => '{DBFS_TABLESPACE}',"
    "    use_bf     => FALSE); "
    "END;"
)
_register = (
    "BEGIN "
    "  DBMS_DBFS_CONTENT.REGISTERSTORE("
    f"    store_name => '{DBFS_STORE}',"
    "    provider_name => 'sample1',"
    "    provider_package => 'DBMS_DBFS_SFS'); "
    "END;"
)
_mount = (
    "BEGIN "
    "  DBMS_DBFS_CONTENT.MOUNTSTORE("
    f"    store_name => '{DBFS_STORE}',"
    f"    store_mount => '{DBFS_MOUNT.lstrip('/')}'); "
    "END;"
)

_DBFS_EXISTS_CODES = (955, 64007, 64008, 1)  # 1 = unique constraint on DBFS$_STORES/DBFS$_MOUNTS
with agent_conn.cursor() as cur:
    for label, stmt in [("createfilesystem", _create_fs),
                        ("registerstore",    _register),
                        ("mountstore",       _mount)]:
        try:
            cur.execute(stmt)
            print(f"{label}: ok")
        except oracledb.DatabaseError as e:
            err_code = e.args[0].code if e.args else None
            err_text = str(e)
            if err_code in _DBFS_EXISTS_CODES or "already exists" in err_text.lower():
                print(f"{label}: (already exists)")
            else:
                print(f"{label}: {e}")

agent_conn.commit()


### 7.2 A minimal Python wrapper around DBFS

The native API is PL/SQL — `DBMS_DBFS_CONTENT.PUTPATH` / `GETPATH`. We wrap it so the rest of the notebook can treat the scratchpad as if it were `open()`/`read()`. This is the only place we touch DBFS; everything else uses this thin façade.

> **Why not just `open()` on the notebook kernel's filesystem?** Because we want the scratchpad to live inside the database. It survives container rebuilds (as long as the datafile survives), it's inside the same transactional boundary as OAMP's memory tables, and in an enterprise deployment it's covered by the same backups, replication, and audit as the rest of the database. No separate filesystem to secure.


In [ ]:
class DBFS:
    """Minimal file-like wrapper over Oracle DBFS."""

    def __init__(self, conn, mount: str = DBFS_MOUNT):
        self.conn = conn
        self.mount = mount.rstrip("/")

    def _full(self, path: str) -> str:
        """Resolve to the absolute DBFS path. Accepts both bare and
        mount-qualified forms ('foo.sql' OR '/scratch/foo.sql') so
        callers don't accidentally double the prefix into
        '/scratch/scratch/foo.sql' (ORA-64001).
        """
        if not path.startswith("/"):
            path = "/" + path
        if path == self.mount or path.startswith(self.mount + "/"):
            return path
        return f"{self.mount}{path}"

    def write(self, path: str, content) -> None:
        """Create-or-overwrite the file at `path` under the scratchpad mount."""
        data = content.encode("utf-8") if isinstance(content, str) else content
        full = self._full(path)
        # DELETEFILE(path) is the path-based file delete in DBMS_DBFS_CONTENT;
        # the other arguments (filter, soft_delete, store_name, principal) all
        # default. The exception handler swallows "no such path" so the call
        # is a no-op when the file doesn't exist yet (i.e. "create or overwrite").
        delete_plsql = (
            "BEGIN "
            "  DBMS_DBFS_CONTENT.DELETEFILE(:p); "
            "EXCEPTION WHEN OTHERS THEN NULL; "
            "END;"
        )
        # CREATEFILE(path, properties, content) — properties and content are
        # IN/OUT, so they must be assignment-targets (locals), not literals.
        # We declare them inside the PL/SQL block and seed l_blob from the
        # bound bytes (forced to BLOB via setinputsizes below).
        create_plsql = (
            "DECLARE "
            "  l_props DBMS_DBFS_CONTENT_PROPERTIES_T := DBMS_DBFS_CONTENT_PROPERTIES_T(); "
            "  l_blob  BLOB := :b; "
            "BEGIN "
            "  DBMS_DBFS_CONTENT.CREATEFILE("
            "    path       => :p,"
            "    properties => l_props,"
            "    content    => l_blob); "
            "  COMMIT; "
            "END;"
        )
        with self.conn.cursor() as cur:
            cur.execute(delete_plsql, p=full)
            cur.setinputsizes(b=oracledb.DB_TYPE_BLOB)
            cur.execute(create_plsql, p=full, b=data)
        self.conn.commit()

    def append(self, path: str, content) -> None:
        """Append `content` to `path`. If the file doesn't exist yet,
        behaves like `write`. Use for running findings logs that should
        grow without overwriting prior entries.
        """
        new = content.encode("utf-8") if isinstance(content, str) else content
        try:
            existing = self.read(path).encode("utf-8")
            sep = b"" if existing.endswith(b"\n") or not existing else b"\n"
            self.write(path, existing + sep + new)
        except FileNotFoundError:
            self.write(path, new)

    def read(self, path: str) -> str:
        full = self._full(path)
        # GETPATH(path, properties, content, item_type, ...) - first four are
        # mandatory in this version. We pass locals for all the OUT params and
        # only return content via :out.
        read_plsql = (
            "DECLARE "
            "  l_props     DBMS_DBFS_CONTENT_PROPERTIES_T := DBMS_DBFS_CONTENT_PROPERTIES_T(); "
            "  l_blob      BLOB; "
            "  l_item_type NUMBER; "
            "BEGIN "
            "  DBMS_DBFS_CONTENT.GETPATH("
            "    path       => :p,"
            "    properties => l_props,"
            "    content    => l_blob,"
            "    item_type  => l_item_type); "
            "  :out := l_blob; "
            "END;"
        )
        try:
            with self.conn.cursor() as cur:
                out = cur.var(oracledb.DB_TYPE_BLOB)
                cur.execute(read_plsql, p=full, out=out)
                blob = out.getvalue()
            if blob is None:
                raise FileNotFoundError(full)
            return blob.read().decode("utf-8", errors="replace")
        except oracledb.DatabaseError as e:
            # SFS provider raises ORA-64002 for non-existent paths instead
            # of returning NULL — translate so callers (esp. append) can
            # treat both the same way via FileNotFoundError.
            if e.args and e.args[0].code in (64002, 64007):
                raise FileNotFoundError(full) from e
            raise

    def list(self, path: str = "/") -> list[str]:
        """List every file under `path` in DBFS.

        Two strategies, in order:

        1. ``DBMS_DBFS_CONTENT.LIST(path, '*', 1)`` — the documented API. Works
           on some providers; the SecureFile (SFS) provider on Oracle Free 26ai
           raises ``ORA-64003`` ("unsupported operation") instead.

        2. Direct query on the SFS storage table (``AGENT_SCRATCH_T``).
           Important subtlety: SFS stores pathnames *without* the mount prefix
           (just '/comment.txt', not '/scratch/comment.txt'). We translate the
           caller's mount-qualified `full` path into an SFS-relative prefix
           for the LIKE filter, then prepend the mount back onto each result.
        """
        full = self._full(path)
        out: list[str] = []

        # Strategy 1 — DBMS_DBFS_CONTENT.LIST
        try:
            with self.conn.cursor() as cur:
                cur.execute(
                    "SELECT * FROM TABLE(DBMS_DBFS_CONTENT.LIST(:p, '*', 1))",
                    p=full)
                descs = cur.description or []
                path_idx = next(
                    (i for i, d in enumerate(descs) if "PATH" in (d[0] or "").upper()),
                    0,
                )
                for row in cur:
                    v = row[path_idx]
                    if v: out.append(v)
            if out:
                return out
        except oracledb.DatabaseError:
            pass

        # Strategy 2 — query the SFS storage table directly.
        store_table = f"{DBFS_STORE}_T"
        try:
            with self.conn.cursor() as cur:
                if full == self.mount or full == self.mount + "/":
                    sfs_prefix = "/"
                elif full.startswith(self.mount + "/"):
                    sfs_prefix = full[len(self.mount):]
                    if not sfs_prefix.endswith("/"):
                        sfs_prefix += "/"
                else:
                    sfs_prefix = "/"

                # pathtype = 1 -> file; std_deleted = 0 -> not tombstoned.
                cur.execute(
                    f"SELECT pathname FROM {store_table} "
                    f" WHERE pathtype = 1 AND std_deleted = 0 "
                    f"   AND pathname LIKE :p "
                    f" ORDER BY pathname",
                    p=f"{sfs_prefix}%")
                for (n,) in cur:
                    if n:
                        out.append(f"{self.mount}{n}")
        except oracledb.DatabaseError:
            pass

        return out


# Module-level instance every later cell uses — DBFS is just a thin wrapper
# around `agent_conn`, so it's safe to instantiate once at notebook scope.
scratch = DBFS(agent_conn)
print(f"DBFS scratch ready at {scratch.mount}")


Instantiate the Python wrapper from §7.2 and round-trip a small file. Confirms the four PL/SQL calls (`PUTPATH`, `GETPATH`, etc.) actually work end-to-end on this build before we depend on them in the agent loop.


> ### 💡 Key takeaways — Part 7
>
> - **The scratchpad is a real filesystem.** DBFS persists across tool calls AND across turns on the same thread — like `/tmp` on a server, except transactional and inside Oracle.
> - **`scratch_write` for drafts, `scratch_append` for logs.** Write replaces (SQL drafts, plan revisions); append grows (findings logs, transcripts). Batch your appends — one append per row of data is wasteful and burns the iteration budget.
> - **Multi-step reasoning without context bloat.** The agent reasons over a long task by *writing* intermediate state to the scratchpad, not by inflating the prompt. "Fits in context" stops being a hard limit.


## Part 9 — Code execution: in-database JavaScript via Oracle MLE

*Parts 4–8 gave the agent memory. Part 9 gives it compute. JavaScript snippets run inside the Oracle process via `DBMS_MLE` — no Python, Node, or GraalVM on the host. The harness needs this because LLMs are unreliable at percentile/median/aggregation math; we want the model to fetch the values via SQL and hand the math to a deterministic engine.*

The agent needs to run code: slice a result set, reshape JSON, format a number, compute an aggregate the model would rather not do in its head. We route those snippets through Oracle's **Multilingual Engine (MLE)** — JavaScript that runs *inside* the Oracle process, called via `DBMS_MLE`.

> **Why JavaScript and not Python?** Oracle AI Database 26ai Free ships MLE with JavaScript. Python MLE is a separate package not enabled in this build, which surfaces as `ORA-04101: Multilingual Engine does not support the language PYTHON`. The agent doesn't care which language we expose — it'll happily emit JS for the kinds of computation we need (arithmetic, string formatting, JSON reshaping).

Why MLE rather than a subprocess sandbox or a remote VM:

- **MLE is a *language* sandbox, not a *privilege* sandbox.** GraalVM polyglot blocks the things you'd expect a sandbox to block — filesystem, network, native code, OS shell — but **JS-side code inherits the caller's database grants** via `mle-js-oracledb`. So the real trust boundary is `AGENT_USER`'s grants plus whatever the kernel enforces at SQL-execution time (DDS/VPD/audit/redaction — §14.4). Heads-up: the only enforcement that survives an MLE-issued `oracledb.defaultConnection().execute(...)` is the kernel's — DDS / DBMS_RLS policies (§14.4), DB grants, audit. Anything you implement in Python (harness-level wrappers, regex checks on tool args) is bypassable by SQL the JS sandbox issues directly. If you care, narrow `AGENT`'s grants or push policy into the kernel.
- **No exfiltration channel by construction.** The MLE polyglot engine doesn't expose filesystem, network, or native libs, so agent-authored code can't open sockets or read `~/.ssh/`.
- **Code runs next to the data.** When the agent's snippet operates over rows it just retrieved, those rows don't have to round-trip back out to a separate process.
- **Zero local install.** GraalVM ships *inside* the database. No Python, Java, or GraalVM on the laptop running the harness.
- **No default time bound.** A `while(true){}` snippet will hold the session until you (or DBA) kill it. If `exec_js` is exposed to anything semi-trusted, gate it behind a Resource Manager consumer group with a `MAX_EST_EXEC_TIME` / `IDLE_TIME` cap; the harness here doesn't do that.

The whole sandbox is one helper, `exec_js(code)`, called by the §10 tool.

**Why the agent will actually reach for it.** The system prompt in §11 has an explicit rule (*"For numeric work over fetched rows ... you MUST fetch with `run_sql` and then call `exec_js`"*) that pushes percentile / mean / max / unit-conversion / reshaping work through MLE rather than letting the model do arithmetic in its head or rely solely on SQL aggregation. Without that rule GPT-5-class models often skip the JS hop and answer from a single `run_sql` aggregate — convenient but harder to audit when the math gets non-trivial.


In [ ]:
def exec_js(code: str) -> dict:
    """Evaluate a snippet of JavaScript inside Oracle MLE.

    Returns {"stdout": str, "stderr": str, "ok": bool}.
    """
    # Wrap user code so we can capture console.log output and route the
    # result back through mle-js-bindings (the EVAL `result` CLOB doesn't
    # auto-fill from an IIFE on this build, so we use the explicit binding
    # API and read it back via DBMS_MLE.IMPORT_FROM_MLE).
    wrapper = (
        '(function() {\n'
        '  let _stdout = "";\n'
        '  let _stderr = "";\n'
        '  let _ok = true;\n'
        '  const _origLog = console.log;\n'
        '  console.log = function() {\n'
        '    _stdout += Array.from(arguments).map(String).join(" ") + "\\n";\n'
        '  };\n'
        '  try {\n'
        + code + '\n'
        '  } catch (e) {\n'
        '    _stderr = String(e && e.message ? e.message : e) + "\\n" + (e && e.stack ? e.stack : "");\n'
        '    _ok = false;\n'
        '  } finally {\n'
        '    console.log = _origLog;\n'
        '  }\n'
        '  const bindings = require("mle-js-bindings");\n'
        '  bindings.exportValue("result", JSON.stringify({stdout: _stdout, stderr: _stderr, ok: _ok}));\n'
        '})();'
    )

    plsql = (
        "DECLARE "
        "  l_ctx    RAW(16); "
        "  l_result CLOB; "
        "BEGIN "
        "  l_ctx := DBMS_MLE.CREATE_CONTEXT(); "
        "  DBMS_MLE.EVAL(l_ctx, 'JAVASCRIPT', :source); "
        "  DBMS_MLE.IMPORT_FROM_MLE(l_ctx, 'result', l_result); "
        "  :result := l_result; "
        "  DBMS_MLE.DROP_CONTEXT(l_ctx); "
        "EXCEPTION WHEN OTHERS THEN "
        "  BEGIN DBMS_MLE.DROP_CONTEXT(l_ctx); EXCEPTION WHEN OTHERS THEN NULL; END; "
        "  RAISE; "
        "END;"
    )

    with agent_conn.cursor() as cur:
        out = cur.var(oracledb.DB_TYPE_CLOB)
        try:
            cur.setinputsizes(source=oracledb.DB_TYPE_CLOB)
            cur.execute(plsql, source=wrapper, result=out)
        except oracledb.DatabaseError as e:
            return {"stdout": "", "stderr": str(e), "ok": False}

    clob = out.getvalue()
    text = clob.read() if hasattr(clob, "read") else (clob or "")
    if not text:
        return {"stdout": "", "stderr": "MLE returned no output", "ok": False}
    try:
        return json.loads(text)
    except (ValueError, TypeError):
        return {"stdout": text, "stderr": "", "ok": True}


In [ ]:
# Imagine `run_sql` just returned this list of order totals (in cents):
order_totals = [199, 4999, 12999, 599, 8999, 24999, 1499, 89999, 3499, 599, 19999, 11999]

# The agent inlines the values and asks MLE to compute summary stats.
snippet = f"""
const totals = {json.dumps(order_totals)}.slice().sort((a, b) => a - b);
const n = totals.length;
function pct(p) {{
    const k = (n - 1) * p;
    const f = Math.floor(k);
    const c = Math.min(f + 1, n - 1);
    return f === c ? totals[f] : totals[f] + (totals[c] - totals[f]) * (k - f);
}}
const sum = totals.reduce((a, b) => a + b, 0);
console.log(
    "n=" + n + "  sum=" + sum.toLocaleString() +
    "  mean=" + Math.floor(sum / n).toLocaleString() +
    "  p50=" + Math.round(pct(0.50)).toLocaleString() +
    "  p95=" + Math.round(pct(0.95)).toLocaleString() +
    "  max=" + totals[n - 1].toLocaleString()
);
"""

r = exec_js(snippet)
print(r["stdout"], end="")
if not r["ok"]:
    print("stderr:", r["stderr"])


> ### 💡 Key takeaways — Part 9
>
> - **LLMs are unreliable at math.** Percentiles, weighted means, post-fetch reshaping — anything quantitative — should run in a deterministic engine, not in the model's head.
> - **MLE is a *language* sandbox, not a *privilege* sandbox.** GraalVM blocks I/O and arbitrary syscalls; same trust boundary as `run_sql`. No subprocess, no network, no extra install.
> - **The audit trail is the actual computation.** Routing math through `exec_js` means the trace shows the JS source, the inputs, and the result — not a number the LLM claims is correct.


## Part 10 — Tools: what the agent can actually *do*

*Parts 4–9 gave the agent memory + compute. Part 10 wires up the dispatchable interface — every Python callable the model can invoke as a tool. Each tool gets a row in a vector-indexed `toolbox` table, retrieved per turn by similarity to the user's query so the per-turn prompt stays lean even when the registry grows past 30 tools.*

A **tool** in our harness is two things:

1. A Python callable that does the real work. Its name, docstring, type hints, and signature defaults *are* the tool's public spec.
2. A row in the `toolbox` table — name, description, parameters JSON, embedding — so we can do **vector retrieval** over tools at agent-loop time.

The decorator is argument-less: `@register` introspects the function and writes both the in-memory entry and the `toolbox` row.

### How the index works — HNSW in Oracle AI Database

The `toolbox` table is created with a **`VECTOR(384, FLOAT32)`** column and an **HNSW** (Hierarchical Navigable Small World) index over it:

```sql
CREATE VECTOR INDEX toolbox_emb_v ON toolbox(embedding)
  ORGANIZATION INMEMORY NEIGHBOR GRAPH
  DISTANCE COSINE
```

`ORGANIZATION INMEMORY NEIGHBOR GRAPH` is Oracle 26ai's HNSW implementation: a multi-layer proximity graph held in the **vector memory pool** (the SGA region we sized in §3.2.1 with `vector_memory_size`). Lookups walk the graph rather than scanning every row, so retrieval cost grows with `log(N)` rather than `N`.

That's why §3.2.1's pool allocation matters: with `vector_memory_size = 0`, the `CREATE VECTOR INDEX` raises `ORA-51962` and we fall back to a full-table cosine scan. Cheap at 7 tools, slow at 7000. The cell below verifies the index landed; if it didn't, you get a loud message telling you to bounce and re-run §3.1+.


### Why pull tools by vector search?

If the registry has 6 tools, it's harmless to put them all in every LLM call. Once you have 30+ tools (per-system MCP servers, per-team helpers), the model starts confusing them and the per-turn token bill grows linearly with the registry. Indexing tools by an embedding of `name + description + arg names` lets us pass *only the relevant top-k* for a given user query.

We still always include a small **always-on** set (`run_sql`, `search_knowledge`, `remember`) — they're cheap and the agent calls them on almost every turn.

### The toolset (after registration)

| Tool | What it does |
|---|---|
| `scan_database(owner)` | Run the §5 scanner against a schema; append facts to OAMP memory. |
| `search_knowledge(query, k, kinds)` | Semantic search over the agent's long-term memory. |
| `run_sql(sql, max_rows)` | Read-only SQL against the target database. |
| `exec_js(code)` | Run JavaScript inside Oracle MLE. |
| `scratch_write(path, content)` / `scratch_read(path)` | DBFS scratchpad I/O (§7.2). |
| `remember(subject, body, kind)` | Persist a correction or learning into memory. |
| `fetch_tool_output(tool_call_id)` | Recover the full bytes of a previously-truncated tool output (§14.2). |


**Toolbox flow at a glance** — toolbox does *two* distinct things at *two* different times. Confusing them is the most common pitfall reading the §10 code.

![Toolbox flow — registration vs per-turn retrieval](images/cover-toolbox-flow.png)



In [ ]:
import inspect
import re
import typing

TOOLS: dict[str, tuple] = {}            # name -> (callable, openai_schema)
ALWAYS_ON_TOOLS = {"search_knowledge", "run_sql", "remember", "exec_js"}
TOOL_EMBEDDING_DIM = ONNX_EMBED_DIM      # 384 for ALL_MINILM_L12_V2


# ---- toolbox table (DDL) -------------------------------------------------
_TOOLBOX_DDL = [
    (
        "CREATE TABLE toolbox ("
        "  name        VARCHAR2(128) PRIMARY KEY,"
        "  description CLOB NOT NULL,"
        "  parameters  JSON,"
        f" embedding   VECTOR({TOOL_EMBEDDING_DIM}, FLOAT32),"
        "  updated_at  TIMESTAMP DEFAULT CURRENT_TIMESTAMP"
        ")"
    ),
    (
        "CREATE VECTOR INDEX toolbox_emb_v ON toolbox(embedding) "
        "ORGANIZATION INMEMORY NEIGHBOR GRAPH DISTANCE COSINE"
    ),
]
with agent_conn.cursor() as cur:
    # If a previous run created `toolbox` with a different vector dim,
    # drop it cleanly so the new DDL takes.
    try:
        cur.execute("SELECT data_length FROM user_tab_columns "
                    " WHERE table_name = 'TOOLBOX' AND column_name = 'EMBEDDING'")
        row = cur.fetchone()
        if row and row[0] and row[0] not in (TOOL_EMBEDDING_DIM * 4, None):
            cur.execute("DROP TABLE toolbox PURGE")
            print(f"dropped toolbox (old vector dim) so we can rebuild at {TOOL_EMBEDDING_DIM}")
    except oracledb.DatabaseError:
        pass

    for stmt in _TOOLBOX_DDL:
        try:
            cur.execute(stmt)
        except oracledb.DatabaseError as e:
            code_ = e.args[0].code
            if code_ in (955, 1408):
                continue
            if code_ == 51962 and "VECTOR INDEX" in stmt.upper():
                print("!! ORA-51962: vector_memory_size is 0. HNSW index NOT created.")
                print("   The agent will fall back to linear cosine scan over the toolbox.")
                print("   Fix: run §3.2.1 to set vector_memory_size, `docker restart oracle-free`,")
                print("        restart the kernel, and re-run from §3.1.")
                continue
            raise

agent_conn.commit()


# ---- HNSW verification (loud, explicit) ----------------------------------
# We create the toolbox so the agent can do *vector retrieval* over its own
# tools at runtime - matters once you have 30+ tools and the per-turn prompt
# would otherwise carry every schema. The retrieval is only fast if there's
# an HNSW (Hierarchical Navigable Small World) graph index on `embedding`;
# without it, every turn does a full table scan with VECTOR_DISTANCE.
with agent_conn.cursor() as cur:
    cur.execute(
        "SELECT index_name, index_type "
        "  FROM user_indexes "
        " WHERE table_name = 'TOOLBOX' "
        "   AND index_type LIKE 'VECTOR%'"
    )
    vec_indexes = list(cur)

if vec_indexes:
    for name, kind in vec_indexes:
        print(f"OK: HNSW vector index in place -> {name} ({kind})")
else:
    print("!! HNSW vector index is MISSING on toolbox.embedding.")
    print("   Tool retrieval still works (linear cosine over a few rows is fine),")
    print("   but you'll lose the index benefit at scale. To enable: §3.2.1 + bounce + re-run §3.1+.")


# ---- type-hint -> JSON-schema helper -------------------------------------
_PRIMS = {int: "integer", float: "number", bool: "boolean", str: "string"}


def _hint_to_json(hint) -> dict:
    """Map a Python type hint into a JSON-schema fragment (best effort)."""
    origin = typing.get_origin(hint)
    if hint in _PRIMS:
        return {"type": _PRIMS[hint]}
    if origin in (list, typing.List):
        args = typing.get_args(hint) or (str,)
        return {"type": "array", "items": _hint_to_json(args[0])}
    if origin in (dict, typing.Dict):
        return {"type": "object"}
    if origin is typing.Union:
        non_none = [a for a in typing.get_args(hint) if a is not type(None)]
        if len(non_none) == 1:
            return _hint_to_json(non_none[0])
    return {"type": "string"}


def _build_schema(fn) -> tuple[str, str, dict, dict]:
    raw_name = fn.__name__
    name = raw_name[5:] if raw_name.startswith("tool_") else raw_name
    description = (inspect.getdoc(fn) or "").strip()
    if not description:
        raise ValueError(f"tool {name!r} has no docstring; @register needs one for retrieval")
    sig = inspect.signature(fn)
    hints = typing.get_type_hints(fn)
    properties: dict = {}
    required: list[str] = []
    for pname, param in sig.parameters.items():
        prop = _hint_to_json(hints.get(pname, str))
        if param.default is not inspect.Parameter.empty and param.default is not None:
            prop["default"] = param.default
        else:
            required.append(pname)
        properties[pname] = prop
    parameters = {"type": "object", "properties": properties, "required": required}
    openai_schema = {"type": "function",
                     "function": {"name": name,
                                  "description": description,
                                  "parameters": parameters}}
    return name, description, parameters, openai_schema


# ---- @register ------------------------------------------------------------
def register(fn):
    """Argument-less decorator: name from __name__, description from __doc__,
    parameters schema from the signature + type hints. Embedding for the
    `toolbox` row is computed in-DB via VECTOR_EMBEDDING - no Python-side
    embedder, no network call.
    """
    name, description, parameters, openai_schema = _build_schema(fn)
    TOOLS[name] = (fn, openai_schema)

    arg_text = " ".join(parameters["properties"].keys())
    embed_text = f"{name}: {description}\nargs: {arg_text}"

    with agent_conn.cursor() as cur:
        cur.execute(
            "MERGE INTO toolbox t USING (SELECT :tn AS n FROM dual) s ON (t.name = s.n) "
            "WHEN MATCHED THEN UPDATE SET description = :td, parameters = :tp, "
            f"                              embedding = VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :etext AS DATA), "
            "                              updated_at = CURRENT_TIMESTAMP "
            "WHEN NOT MATCHED THEN INSERT (name, description, parameters, embedding) "
            f"                       VALUES (:tn, :td, :tp, VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :etext AS DATA))",
            tn=name, td=description, tp=json.dumps(parameters), etext=embed_text)
    agent_conn.commit()
    return fn


# ---- retrieval ------------------------------------------------------------
def retrieve_tools(query: str, k: int = 6) -> list[dict]:
    """Top-k tool schemas for `query`, plus the always-on set.

    Stage 1: cosine search over `toolbox.embedding` (in-DB; HNSW when present).
    Stage 2: cross-encoder rerank via `rerank()` - in-DB when a reranker is
    loaded (§3.5), pass-through otherwise. Always-on tools are merged in last
    so they're guaranteed in the schema list regardless of rerank ordering.
    """
    # Stage 1: oversample so the reranker has signal to work with.
    cosine_fetch = k * 4
    rows: list[dict] = []
    with agent_conn.cursor() as cur:
        cur.execute(
            "SELECT name, description FROM toolbox "
            f" ORDER BY VECTOR_DISTANCE(embedding, VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :q AS DATA), COSINE) "
            " FETCH FIRST :k ROWS ONLY",
            q=query, k=cosine_fetch)
        for name, desc in cur:
            desc_text = desc.read() if hasattr(desc, "read") else str(desc or "")
            rows.append({"name": name, "content": desc_text})

    # Stage 2: rerank by description against the user query.
    ranked = rerank(query, rows, top_k=k, content_key="content")

    schemas: dict[str, dict] = {}
    for r in ranked:
        if r["name"] in TOOLS:
            schemas[r["name"]] = TOOLS[r["name"]][1]

    # Always-on set merges in last - guaranteed inclusion.
    for name in ALWAYS_ON_TOOLS:
        if name in TOOLS:
            schemas[name] = TOOLS[name][1]
    return list(schemas.values())


Every tool the agent can dispatch, defined as a plain Python function with type hints and a docstring. `@register` (defined in §10's prior cell) introspects the signature, builds an OpenAI-style JSON schema, embeds the description, and writes a row to the `toolbox` table. The decorator is the *only* place we describe each tool — there is no separate JSON schema file to keep in sync.


Two new tools: `load_skill(name)` returns the full markdown body of a named skill, and `list_skills(query)` vector-searches the skillbox for the top-k closest names + descriptions. We add `load_skill` to `ALWAYS_ON_TOOLS` so the manifest in the system prompt always has a working dispatch path — the model can always call it without depending on cosine retrieval surfacing it.


In [ ]:
# ============== Tool implementations =====================================

@register
def tool_scan_database(owner: str) -> str:
    """Scan the specified schema of the target Oracle AI Database and update institutional knowledge.
    Run this when the user asks about a schema you have never seen or when you suspect the knowledge store is stale.
    `owner` is the schema owner (e.g. 'DEMO').
    """
    return json.dumps(run_scan(agent_conn, owner=owner))


@register
def tool_search_knowledge(query: str, k: int = 5, kinds: list[str] | None = None) -> str:
    """Search institutional knowledge (what the agent has learned about the target database) by semantic similarity.
    Use this BEFORE running SQL to discover which tables and columns are relevant.
    `kinds` is an optional filter: table, column, relationship, query_pattern, correction.
    """
    hits = retrieve_knowledge(query, k=k, kinds=kinds)
    for h in hits:
        h["body"] = h["body"][:500]
    return json.dumps(hits)


_READ_ONLY = re.compile(r"^\s*(select|with)\b", re.IGNORECASE)


@register
def tool_run_sql(sql: str, max_rows: int = 50) -> str:
    """Execute a READ-ONLY SQL statement (SELECT/WITH only) against the target Oracle AI Database
    and return up to `max_rows` rows as JSON. Reject any statement that isn't read-only.
    """
    if not _READ_ONLY.match(sql.strip()):
        return json.dumps({"error": "only SELECT / WITH statements are allowed in run_sql"})
    try:
        with agent_conn.cursor() as cur:
            cur.execute(sql)
            cols = [d[0] for d in cur.description]
            rows = []
            for i, r in enumerate(cur):
                if i >= max_rows: break
                rows.append([(v.read() if hasattr(v, "read") else v) for v in r])
        return json.dumps({"columns": cols, "rows": rows, "row_count": len(rows)},
                          default=str)
    except Exception as e:
        return json.dumps({"error": str(e)})


@register
def tool_exec_js(code: str) -> str:
    """Execute JavaScript inside Oracle MLE (no filesystem, no network).
    Good for arithmetic, string formatting, JSON reshaping, simple aggregations.
    `console.log(...)` output comes back as `stdout`.
    """
    return json.dumps(exec_js(code))


@register
def tool_scratch_write(path: str, content: str) -> str:
    """Write a short-term note to the DBFS scratchpad.
    Use for draft SQL, intermediate results, running calculations you want to reference later in the same turn.
    """
    scratch.write(path, content)
    return json.dumps({"ok": True, "path": path, "bytes": len(content)})


@register
def tool_scratch_append(path: str, content: str) -> str:
    """Append text to the end of a DBFS scratchpad file (or create it).
    Use this instead of `scratch_write` when you want to ADD to a running
    log without losing prior entries — e.g. /scratch/findings.md as you
    discover facts across multiple turns, /scratch/transcript.md, etc.
    For SQL drafts or "latest is truth" content, prefer scratch_write.
    """
    scratch.append(path, content)
    return json.dumps({"ok": True, "path": path, "appended_bytes": len(content)})


@register
def tool_scratch_read(path: str) -> str:
    """Read from the DBFS scratchpad."""
    try:
        return json.dumps({"content": scratch.read(path)})
    except FileNotFoundError:
        return json.dumps({"error": f"not found: {path}"})


@register
def tool_remember(subject: str, body: str, kind: str = "correction") -> str:
    """Persist a correction or learning into institutional knowledge so future turns benefit.
    Use when the user corrects you, or when you discover a non-obvious fact that future retrievals should surface.
    `subject` is a short label (e.g. 'SALES.ORDERS.total_cents'); `body` is the fact written as a full sentence.
    """
    fact = Fact(kind=kind, subject=subject, body=body, metadata={"source": "agent_remember"})
    new, updated, _ = write_facts([fact])
    return json.dumps({"ok": True, "new": new, "updated": updated})


print(f"registered {len(TOOLS)} tools: {sorted(TOOLS)}")


> ### 💡 Key takeaways — Part 10
>
> - **Tools are Python callables with embeddings.** The `@register` decorator introspects the function and writes a vector-indexed row. Function name + docstring + arg names *are* the public spec.
> - **Vector retrieval keeps the prompt lean.** With 30+ tools, including all of them every turn confuses the model. Top-k by cosine over the user query exposes only what's relevant — registry size grows without per-turn cost growing.
> - **Always-on vs retrieved.** Cheap-and-frequent tools (`run_sql`, `search_knowledge`, `remember`) ship in every prompt. Specialised tools (`scan_database`, `get_document`) come from the toolbox lookup.


## Part 11.5 — Skillbox: procedural memory for *how* to do things

*Part 10's `toolbox` answers "what can the agent call?" — function specs, dispatched as `tool_calls`. Part 11.5 answers a different question: "what does the agent know how to do?" — prose playbooks the model reads as part of its context. Same in-DB vector retrieval pattern as `toolbox`, different consumption model (read on demand instead of dispatch).*

The §10 `toolbox` answers *"what can the agent call?"* — function specs, dispatched as `tool_calls`. The **`skillbox`** answers a different question: *"what does the agent know how to do?"* — prose playbooks the model reads as part of its context.

The pattern follows [Anthropic's SKILL.md convention](https://platform.claude.com/docs/en/agents-and-tools/agent-skills/overview): each skill is a markdown document with a `name` and a `description`. At turn start the agent sees a **manifest** (top-k names + one-line descriptions, surfaced by vector retrieval) prepended to the system prompt. When a skill is relevant, the agent calls `load_skill(name)` and the body becomes a tool output it can read and follow.

**Two procedural-memory tables, parallel structures:**

| | `toolbox` (§10) | `skillbox` (here) |
|---|---|---|
| Holds | Callable function specs | Prose markdown playbooks |
| Consumed as | `tools=[...]` parameter — dispatched | Text in context — read |
| Per-turn injection | Top-k schemas + always-on | Top-k *names + 1-line desc* (manifest) |
| Full content | n/a (functions just run) | `load_skill(name)` returns the body |

**Source: [`oracle/skills/db`](https://github.com/oracle/skills/tree/main/db).** Oracle publishes a curated library (100+ guides, organized by category — `agent`, `performance`, `security`, `plsql`, `sqlcl`, …). Each `.md` file is a skill: an H1 title, a first-paragraph description, and a body of prose + SQL examples. We mirror them into `skillbox` with their content SHA so re-ingestion is idempotent.

**Why a separate table, not extending `toolbox`?** Different content (executable spec vs prose), different retrieval contract (always-list vs surface-on-demand), different lifecycle. Conflating them would force every consumer to filter by `kind`.

**Why CLOB inline, not DBFS?** Skills are 1–30 KB. A single `SELECT` returns metadata + body in one round trip; DBFS would buy `open()` semantics for nothing — the agent doesn't need a file handle, it needs the bytes.

**Why two-stage (manifest + on-demand load), not always-inject?** ~135 skills × ~5 KB → 15 KB per turn for top-3 bodies. The manifest is ~200 tokens; the full body only loads when the model decides to act on a skill.


**Skillbox flow at a glance** — the asymmetry that separates it from the toolbox is *which part of the row* gets surfaced when. Skim the diagram before reading the code.

![Skillbox flow — manifest always-on, body on-demand](images/cover-skillbox-flow.png)



In [ ]:
SKILLBOX_DDL = [
    """
    CREATE TABLE skillbox (
      name        VARCHAR2(160) PRIMARY KEY,
      category    VARCHAR2(64),
      description VARCHAR2(2000) NOT NULL,
      body        CLOB NOT NULL,
      source_url  VARCHAR2(2000),
      source_sha  VARCHAR2(64),
      embedding   VECTOR(384, FLOAT32),
      metadata    JSON,
      updated_at  TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """,
    """
    CREATE VECTOR INDEX skillbox_emb_v ON skillbox(embedding)
      ORGANIZATION INMEMORY NEIGHBOR GRAPH DISTANCE COSINE
    """,
]

with agent_conn.cursor() as cur:
    for stmt in SKILLBOX_DDL:
        try:
            cur.execute(stmt)
        except oracledb.DatabaseError as e:
            code_ = e.args[0].code
            if code_ in (955, 1408):  # already exists
                continue
            if code_ == 51962 and "VECTOR INDEX" in stmt.upper():
                print("!! ORA-51962: vector_memory_size = 0 — HNSW index on skillbox NOT created.")
                print("   See §3.2.1; skillbox vector retrieval will linear-scan until you bounce.")
                continue
            raise
agent_conn.commit()

# Loud verification (mirrors §10/cell 73's pattern)
with agent_conn.cursor() as cur:
    cur.execute(
        "SELECT index_name, index_type FROM user_indexes "
        " WHERE table_name = 'SKILLBOX' AND index_type LIKE 'VECTOR%'"
    )
    vec = list(cur)
if vec:
    for n, t in vec:
        print(f"OK: HNSW vector index in place -> {n} ({t})")
else:
    print("!! HNSW vector index missing on skillbox.embedding.")


Markdown parser for the skill files we'll fetch. Extracts the first paragraph after the H1 (the file's natural "description") and trims to a budget of 1800 chars. Handles the case where a skill grows YAML frontmatter in the future. The resulting `description` is the only thing we embed for retrieval — bodies are loaded on demand by `load_skill`, never inlined.


In [ ]:
import re as _re_skill

def _parse_skill_md(text: str, fallback_name: str) -> str:
    """Pull a one-paragraph description out of a markdown skill file.

    Looks for the first non-empty paragraph after the H1 (and after any
    horizontal-rule frontmatter delimiters). Falls back to fallback_name when
    nothing useful is in the body.
    """
    lines = text.splitlines()
    n, i = len(lines), 0

    # Skip any YAML frontmatter ("---" ... "---") at the top.
    if i < n and lines[i].strip() == "---":
        i += 1
        while i < n and lines[i].strip() != "---":
            i += 1
        i += 1  # skip closing ---

    # Skip leading blank lines.
    while i < n and not lines[i].strip():
        i += 1
    # Skip the H1 line if present.
    if i < n and lines[i].lstrip().startswith("# "):
        i += 1
    # Skip blank lines after H1.
    while i < n and not lines[i].strip():
        i += 1
    # Collect first paragraph (until a blank line or another heading).
    para = []
    while i < n and lines[i].strip() and not lines[i].lstrip().startswith("#"):
        para.append(lines[i].strip())
        i += 1

    description = " ".join(para).strip()
    if not description:
        description = fallback_name
    if len(description) > 1800:
        description = description[:1797].rsplit(" ", 1)[0] + "..."
    return description


# Smoke test on a synthetic markdown blob.
_test = """# Schema Discovery

Before composing a SQL query against an unfamiliar table, an agent should mine
the catalog views (ALL_TAB_COLUMNS, ALL_CONS_COLUMNS) to confirm column names
and key relationships.

## Steps

1. Query ALL_TABLES.
"""
print(_parse_skill_md(_test, fallback_name="agent/schema-discovery"))


##### Chunking long text before we ingest

We're about to ingest ~150 skill files. Some are 30+ KB of markdown — each one a single, dense doc of "how to diagnose ORA-00904", "how to design a partitioning strategy", etc.

If we embed an entire 30 KB doc as one vector, the vector represents an *average* of every concept inside it. A user query that's specifically about *partitioning by hash* gets diluted against everything else in the partitioning skill — table shapes, parallelism, online operations, examples. Retrieval recall drops measurably the longer the document.

The fix is **chunking**: split each document into ~1KB pieces at natural boundaries (paragraph breaks), embed each chunk separately, and let retrieval pick the most-on-point chunk rather than the most-on-average doc. Two things make chunking work:

| Choice | Why |
|---|---|
| Split at `\n\n` (paragraphs), not fixed character offsets | Keeps semantic units intact — sentences shouldn't break mid-thought across chunks |
| Slide ~200 chars of overlap between adjacent chunks | A fact straddling a boundary appears in both chunks, so retrieval doesn't miss it because we cut at exactly the wrong place |

The next cell defines `chunk_text(text, max_chars, overlap)` — a paragraph-aware splitter we can drop in before any future "embed-and-store-long-text" pipeline. We don't actually rebuild the skillbox to chunk every body (the skillbox embeds the *description* and loads the body on demand via `load_skill`, which sidesteps the dilution problem for our use case). But this primitive is here for the moment you DO want chunked retrieval — over PDF-derived text, long docstrings, transcripts, anything where the body itself should be searchable.


In [ ]:
def chunk_text(text: str, max_chars: int = 1200, overlap: int = 200) -> list[str]:
    """Split `text` into overlapping chunks at paragraph boundaries.

    - `max_chars`: target maximum size of each chunk (soft — paragraphs aren't
      split mid-paragraph, so a single very long paragraph can exceed this).
    - `overlap`: trailing characters of the previous chunk that get prepended
      to the next, so a fact near a chunk boundary appears in both.

    Returns a list with one chunk minimum (the original text if it fits).
    """
    if len(text) <= max_chars:
        return [text]

    paragraphs = text.split("\n\n")
    chunks: list[str] = []
    current: list[str] = []
    current_len = 0

    for p in paragraphs:
        p_len = len(p) + 2  # +2 for the \n\n separator
        if current and current_len + p_len > max_chars:
            # Flush the current chunk
            chunk = "\n\n".join(current)
            chunks.append(chunk)
            # Slide forward with `overlap` chars of context from the previous chunk
            tail = chunk[-overlap:] if overlap > 0 else ""
            current = [tail, p] if tail else [p]
            current_len = len(tail) + p_len
        else:
            current.append(p)
            current_len += p_len

    if current:
        chunks.append("\n\n".join(current))
    return chunks


# Demo — pick the longest skill we have and show how it splits
with agent_conn.cursor() as cur:
    cur.execute(
        # Select the CLOB directly: DBMS_LOB.SUBSTR returns VARCHAR2, capped at
        # MAX_STRING_SIZE (4000 by default) — asking for 32000 chars triggers ORA-06502.
        # Reading the CLOB and slicing in Python avoids the SQL VARCHAR2 limit entirely.
        "SELECT name, DBMS_LOB.GETLENGTH(body) AS sz, body "
        "  FROM skillbox ORDER BY DBMS_LOB.GETLENGTH(body) DESC FETCH FIRST 1 ROW ONLY"
    )
    row = cur.fetchone()

if row:
    name, size_bytes, body_clob = row
    body = body_clob.read() if hasattr(body_clob, "read") else str(body_clob)
    pieces = chunk_text(body, max_chars=1200, overlap=200)
    print(f"Longest skill: {name}  ({size_bytes} bytes)")
    print(f"Split into {len(pieces)} chunks of max ~1200 chars with ~200-char overlap.")
    for i, ch in enumerate(pieces[:3]):
        head = ch.split("\n", 1)[0][:90]
        print(f"  chunk {i + 1}/{len(pieces)} ({len(ch)} chars): {head}...")
    if len(pieces) > 3:
        print(f"  ... ({len(pieces) - 3} more chunks)")
else:
    print("(skillbox is empty — chunking demo will run after the next cell ingests)")


Ingestion pipeline. Single tarball download (one HTTP request, no GitHub auth needed, no rate limit risk), in-memory `tarfile.open` walk, idempotent MERGE keyed on a content SHA. Re-running this cell is cheap because rows whose source SHA hasn't changed are skipped. Note `setinputsizes(body=CLOB, md=JSON)` — markdown bodies exceed the 32K VARCHAR2 ceiling routinely so we have to bind them as proper CLOBs.


**Ingestion pipeline at a glance** — the same shape applies to the §5.2 schema scanner and the skillbox ingestion below. Read the column on the left as the abstract pipeline; the column on the right is what each step looks like in this notebook.

```
DOCUMENT INGESTION PIPELINE — generic pattern (used by §5 scanner + §11.5 skillbox)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  abstract step                              this notebook
  ──────────────────────                     ─────────────────────────────────────
  ┌──────────────────────┐
  │ external source      │                   oracle/skills tarball (§11.5)
  │ doc / file / API     │                   ALL_TABLES, ALL_TAB_COLUMNS, V$SQL (§5)
  └──────────┬───────────┘
             │  1. fetch
             ▼
  ┌──────────────────────┐
  │ in-memory blob       │                   tarfile.open(BytesIO).extractfile()
  │ (bytes / str)        │                   cur.execute("SELECT … FROM ALL_TABLES")
  └──────────┬───────────┘
             │  2. parse — extract structure
             ▼
  ┌──────────────────────┐                   _parse_skill_md(text) →
  │ structured record    │                       (name, description, body, sha)
  │ name + meta + body   │                   _scan_tables(conn, owner) →
  │                      │                       Fact(kind, subject, body, metadata)
  └──────────┬───────────┘
             │  3. chunk (when body is long)
             ▼
  ┌──────────────────────┐
  │ chunks: list[str]    │                   chunk_text(body, max_chars=1200,
  │  ┌──┐ ┌──┐ ┌──┐      │                              overlap=200)
  │  │  │ │  │ │  │      │
  │  └──┘ └──┘ └──┘      │                   why chunk? long-doc embeddings
  │   ◄──overlap──►      │                     average too many concepts together;
  │                      │                     focused chunks → better recall
  └──────────┬───────────┘
             │  4. embed (server-side — no host round-trip)
             ▼
  ┌──────────────────────┐                   VECTOR_EMBEDDING(
  │ vectors:             │                       ALL_MINILM_L12_V2
  │  [0.12, -0.45, …]    │                       USING :text AS DATA)
  │  384-dim FLOAT32     │                   → VECTOR(384, FLOAT32)
  └──────────┬───────────┘
             │  5. MERGE  (idempotent on content SHA)
             ▼
╔═══════════ Oracle 26ai ═══════════════════════════════════╗
║                                                            ║
║  ┌─ table rows (name, body, metadata, embedding) ──────┐  ║
║  │   skillbox  /  eda_onnx_memory  /  scan_history     │  ║
║  └────────────────────┬────────────────────────────────┘  ║
║                       │                                    ║
║                       ▼                                    ║
║  ┌─ HNSW vector index ────────────────────────────────┐   ║
║  │  ORGANIZATION INMEMORY NEIGHBOR GRAPH              │   ║
║  │  DISTANCE COSINE                                    │   ║
║  │  served from vector_memory_size SGA pool (§3.2.1)   │   ║
║  └────────────────────┬────────────────────────────────┘   ║
║                       │                                    ║
║                       │  6. retrieve at query time:        ║
║                       │     VECTOR_DISTANCE + cosine       ║
║                       │     + Oracle Text + RRF (§5.6)     ║
║                       ▼                                    ║
║                 top-k rows → agent context                 ║
╚════════════════════════════════════════════════════════════╝
```


In [ ]:
import urllib.request as _urlreq
import tarfile as _tarfile
import io as _io
import hashlib as _hashlib
import json as _json_skill
import datetime as _dt_skill

ORACLE_SKILLS_REPO = "oracle/skills"
ORACLE_SKILLS_BASE = "db"
SKILLS_TARBALL_URL = f"https://api.github.com/repos/{ORACLE_SKILLS_REPO}/tarball/main"


def _download_repo_tarball(url: str = SKILLS_TARBALL_URL) -> dict[str, bytes]:
    """One unauthenticated request — returns {repo_relative_path: bytes} for every
    .md file under db/. Avoids the 60-req/hr GitHub API limit on per-file fetches.
    """
    req = _urlreq.Request(url, headers={"User-Agent": "eda-skillbox-ingester"})
    with _urlreq.urlopen(req, timeout=60) as resp:
        data = resp.read()

    files: dict[str, bytes] = {}
    with _tarfile.open(fileobj=_io.BytesIO(data), mode="r:gz") as tar:
        for member in tar:
            if not member.isfile():
                continue
            # Tarball paths look like "oracle-skills-<sha7>/db/agent/schema-discovery.md".
            parts = member.name.split("/", 1)
            if len(parts) < 2:
                continue
            rel = parts[1]
            if not rel.startswith(f"{ORACLE_SKILLS_BASE}/") or not rel.endswith(".md"):
                continue
            f = tar.extractfile(member)
            if f is not None:
                files[rel] = f.read()
    return files


def ingest_skills_from_oracle(verbose: bool = True) -> dict:
    """Pull every db/<category>/<skill>.md from oracle/skills and MERGE into skillbox.

    Idempotent: rows with unchanged source_sha are skipped. Run repeatedly without
    side effects.
    """
    if verbose:
        print(f"downloading tarball from {SKILLS_TARBALL_URL} ...")
    raw_files = _download_repo_tarball()
    if verbose:
        print(f"  fetched {len(raw_files)} .md files under db/")

    new = updated = skipped = 0
    skipped_overview = 0

    for rel_path, body_bytes in sorted(raw_files.items()):
        # rel_path = "db/<category>/<file>.md"
        parts = rel_path.split("/")
        if len(parts) != 3:
            # nested deeper than db/<cat>/file.md; skip — the repo isn't deeply nested
            # but be defensive in case future versions add subfolders
            continue
        _, category, filename = parts
        # Skip category-level overview file; only mirror leaf skills
        if filename == "SKILL.md":
            skipped_overview += 1
            continue

        body = body_bytes.decode("utf-8", errors="replace")
        file_stem = filename[:-3]  # drop ".md"
        full_name = f"{category}/{file_stem}"
        sha = _hashlib.sha256(body.encode("utf-8")).hexdigest()[:32]

        # Idempotency: skip if same SHA already in DB
        with agent_conn.cursor() as cur:
            cur.execute(
                "SELECT source_sha FROM skillbox WHERE name = :n",
                n=full_name,
            )
            row = cur.fetchone()
        if row and row[0] == sha:
            skipped += 1
            continue

        description = _parse_skill_md(body, fallback_name=full_name)
        embed_text = f"{full_name}\n{description}"
        source_url = (
            f"https://raw.githubusercontent.com/{ORACLE_SKILLS_REPO}/main/{rel_path}"
        )
        meta = _json_skill.dumps({
            "category": category,
            "ingested_at": _dt_skill.datetime.now(_dt_skill.timezone.utc).isoformat(timespec="seconds"),
            "source_repo": ORACLE_SKILLS_REPO,
            "rel_path": rel_path,
        })

        with agent_conn.cursor() as cur:
            # body is CLOB and skill files routinely exceed VARCHAR2 limits;
            # metadata is JSON. Bind both with explicit types so the thin driver
            # doesn't try to send them as oversized VARCHAR2 payloads (ORA-03146).
            cur.setinputsizes(body=oracledb.DB_TYPE_CLOB, md=oracledb.DB_TYPE_JSON)
            cur.execute(
                "MERGE INTO skillbox t "
                "USING (SELECT :n AS name FROM dual) s ON (t.name = s.name) "
                "WHEN MATCHED THEN UPDATE SET "
                "  category = :cat, "
                "  description = :dsc, "
                "  body = :body, "
                "  source_url = :url, "
                "  source_sha = :sha, "
                f" embedding = VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :etext AS DATA), "
                "  metadata = :md, "
                "  updated_at = CURRENT_TIMESTAMP "
                "WHEN NOT MATCHED THEN INSERT "
                "  (name, category, description, body, source_url, source_sha, embedding, metadata) "
                "  VALUES (:n, :cat, :dsc, :body, :url, :sha, "
                f"          VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :etext AS DATA), :md)",
                n=full_name, cat=category, dsc=description, body=body,
                url=source_url, sha=sha, etext=embed_text, md=meta,
            )
            if row is None:
                new += 1
                marker = "+"
            else:
                updated += 1
                marker = "~"
        agent_conn.commit()
        if verbose:
            print(f"  {marker} {full_name}")

    return {
        "new": new,
        "updated": updated,
        "skipped": skipped,
        "skipped_overview": skipped_overview,
        "total_in_repo": len(raw_files),
    }

# Run the ingest. Idempotent — re-runs MERGE on changed source_sha and skip
# unchanged files, so this is safe to leave at the bottom of the cell.
_skill_ingest_result = ingest_skills_from_oracle(verbose=False)
print(f"skillbox ingest: {_skill_ingest_result}")


Actually populates the skillbox. ~155 skills across 17 categories from `oracle/skills/db/**/*.md`. The summary tells you how many rows were new, updated, or skipped on this run. Re-running it after a repo update is the way you refresh — the SHA-based dedup means only changed files re-embed.


In [ ]:
summary = ingest_skills_from_oracle(verbose=True)
print(f"\nIngestion summary: {summary}")

# Show category distribution
with agent_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM skillbox")
    total = cur.fetchone()[0]
    cur.execute("SELECT category, COUNT(*) FROM skillbox GROUP BY category ORDER BY COUNT(*) DESC")
    print(f"\nskillbox total: {total}")
    print("by category:")
    for cat, n in cur:
        print(f"  {cat:20s}  {n}")


In [ ]:
@register
def tool_load_skill(name: str) -> str:
    """Load the full content of a named skill from the skillbox.
    Use this when the system prompt's "Available skills" manifest lists a skill
    relevant to the current task. The full markdown guide is returned and you
    should follow its instructions for the duration of the task.
    `name` is the full namespace, e.g. "agent/schema-discovery".
    """
    with agent_conn.cursor() as cur:
        cur.execute(
            "SELECT description, body, source_url, category "
            "  FROM skillbox WHERE name = :n",
            n=name,
        )
        row = cur.fetchone()
    if not row:
        return json.dumps({"error": f"no skill named {name!r}; call list_skills(query) to find available skills"})
    desc, body, url, category = row
    body_text = body.read() if hasattr(body, "read") else str(body or "")
    return json.dumps({
        "name": name,
        "category": category,
        "description": desc,
        "source_url": url,
        "body": body_text,
    })


@register
def tool_list_skills(query: str, k: int = 5) -> str:
    """Search the skillbox semantically. Returns top-k skills (name + description).
    Use when the system prompt's manifest didn't surface the right skill for the
    current task. Then call load_skill(name) on the most relevant one.
    """
    with agent_conn.cursor() as cur:
        cur.execute(
            "SELECT name, category, description FROM skillbox "
            f" ORDER BY VECTOR_DISTANCE(embedding, VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :q AS DATA), COSINE) "
            " FETCH FIRST :k ROWS ONLY",
            q=query, k=k,
        )
        hits = [{"name": n, "category": c, "description": d} for n, c, d in cur]
    return json.dumps(hits)


# load_skill is always-on so the manifest in the system prompt is never a dead reference.
ALWAYS_ON_TOOLS.add("load_skill")
print(f"registered: load_skill, list_skills  (TOOLS total: {len(TOOLS)}; always-on: {sorted(ALWAYS_ON_TOOLS)})")


Two pieces wire skillbox into the loop without rewriting `agent_turn`. `build_skill_manifest()` does the vector retrieval and formats top-k as one line per skill. The wrap of `build_context` prepends that manifest to every turn's context. The sentinel attribute (`build_context._skillbox_wrapped`) makes re-running this cell idempotent — the wrap doesn't stack.


In [ ]:
def build_skill_manifest(query: str, k: int = 3) -> str:
    """Top-k skills relevant to `query` formatted as a one-line-per-skill manifest.
    Returns an empty string when the skillbox is empty (graceful degradation —
    the rest of the prompt is unaffected).

    `build_context` (defined in §11) calls this directly, so the agent loop's
    user message starts with the manifest whenever §11.5 has been run.
    """
    try:
        with agent_conn.cursor() as cur:
            cur.execute(
                "SELECT name, description FROM skillbox "
                f" ORDER BY VECTOR_DISTANCE(embedding, VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :q AS DATA), COSINE) "
                " FETCH FIRST :k ROWS ONLY",
                q=query, k=k,
            )
            rows = list(cur)
    except oracledb.DatabaseError:
        return ""
    if not rows:
        return ""
    lines = [f"  - {n} — {(d.read() if hasattr(d, 'read') else d)[:240]}" for n, d in rows]
    return (
        "Available skills (call load_skill(name) to read the full guide and follow it):\n"
        + "\n".join(lines) + "\n\n"
    )


> ### 💡 Key takeaways — Part 11.5
>
> - **Tools answer "what can I call?". Skills answer "what do I know how to do?".** Tools are dispatched as function calls; skills are prose playbooks the model reads. Same retrieval pattern, different consumption model.
> - **Manifest now, body on demand.** Top-3 skill names + descriptions go into every prompt (~200 tokens). Full body is one `load_skill(name)` tool call away. The model sees the menu without paying for the meal.
> - **Chunking matters for long docs.** Embedding a 30 KB skill as one vector dilutes the signal. Paragraph-aware chunks with overlap let retrieval pick the most-on-point fragment.


## Part 11 — The agent loop: the heart of the harness

*Parts 4–10 were plumbing — every component the loop needs. Part 11 is the agent itself: a while-loop that builds context (Part 6 memory + Part 5 schema facts), calls the LLM with the retrieved tools (Part 10), dispatches each `tool_call`, persists outputs to OAMP (Part 4), and stops on iteration count or wall-clock budget.*

Everything we've built so far is plumbing. *This* is the agent. Read this section twice.


In [ ]:
SYSTEM_PROMPT = (
    "You are an Enterprise Data Agent operating against an Oracle AI Database.\n"
    "\n"
    "Your job is to answer natural-language questions about the data by reasoning over:\n"
    "  - the institutional knowledge you have accumulated about the database\n"
    "    (tables, columns, relationships, observed query patterns, past corrections,\n"
    "    AND episodic memories of prior conversations on this or other threads);\n"
    "  - the live database, via read-only SQL when you need runtime facts;\n"
    "  - a scratchpad for intermediate notes;\n"
    "  - an in-database JavaScript sandbox via Oracle MLE (`exec_js`) for computation.\n"
    "\n"
    "How to work:\n"
    "  1. ALWAYS call `search_knowledge` first with a paraphrase of the user's question.\n"
    "     Use the results to pick the right tables before writing any SQL.\n"
    "  2. If the user references a schema you have no facts about, call `scan_database`\n"
    "     on that owner to build institutional knowledge, THEN search again.\n"
    "  3. Keep SQL read-only. Use `run_sql` only; never attempt DDL or DML through it.\n"
    "  4. For numeric work over fetched rows - mean / median / percentile / max /\n"
    "     custom aggregation / unit conversion / formatting - you MUST fetch the raw\n"
    "     values with `run_sql` and then call `exec_js` to compute. Do not compute\n"
    "     statistics in your head, and do not lean only on SQL aggregation when JS is\n"
    "     more honest about the math (e.g. percentiles, weighted means, post-fetch\n"
    "     reshaping). The MLE sandbox runs inside Oracle - same trust boundary as\n"
    "     `run_sql`, no network egress, no separate install - so prefer it over\n"
    "     in-head arithmetic every time.\n"
    "  5. For non-trivial SQL (more than ~3 lines, or involving multiple tables /\n"
    "     joins), draft it first to the DBFS scratchpad via `scratch_write` to a\n"
    "     path like `/scratch/<task>.sql`, then `scratch_read` it before passing to\n"
    "     `run_sql`. The scratchpad is a real file system inside Oracle - files\n"
    "     persist across tool calls AND across turns on the same thread.\n"
    "     - `scratch_write(path, content)` REPLACES the file. Use for SQL drafts,\n"
    "       plan revisions, anything where 'latest is the truth'.\n"
    "     - `scratch_append(path, content)` ADDS to the end. Use for running\n"
    "       findings logs (`/scratch/findings.md`), transcripts, anything you\n"
    "       want to grow across multiple steps or turns without overwriting.\n"
    "       BATCH your appends: one `scratch_append` per row of data is wasteful\n"
    "       and burns the iteration budget. Combine many rows into ONE call.\n"
    "  6. When you discover a non-obvious fact, or the user corrects you, call `remember`\n"
    "     so future turns benefit.\n"
    "  7. Prefer short, direct answers. Quote table and column names verbatim.\n"
    "  8. If a tool fails, read the error, adjust, and try once more - don't spin.\n"
    "\n"
    "Never fabricate a table or column. If you're unsure, say so and propose a scan."
)


Three small helpers the loop will use: a per-process cache of OAMP `Thread` objects (so each user turn doesn't reopen the thread), `build_context()` which assembles the per-turn user message from the OAMP context card plus retrieved schema facts, and `chat()` — a thin retry wrapper around the LLM call.


In [ ]:
import concurrent.futures

# Cache of OAMP thread objects keyed by the harness-level thread_id.
THREADS: dict[str, object] = {}


def get_thread(thread_id: str):
    """Look up or create the OAMP thread for this harness-level thread_id."""
    if thread_id in THREADS:
        return THREADS[thread_id]
    try:
        thread = memory_client.get_thread(thread_id)
    except Exception:
        thread = memory_client.create_thread(
            thread_id=thread_id,
            user_id=USER_ID,
            agent_id=AGENT_ID,
            memory_extraction_frequency=2,
            memory_extraction_window=4,
            enable_context_summary=True,
            context_summary_update_frequency=4,
        )
    THREADS[thread_id] = thread
    return thread


def build_context(thread_id: str, user_query: str, k_knowledge: int = 3) -> str:
    """Assemble the prompt context block for one user turn.

    Three layers stack into one user message:
      1. Skill manifest (top-3 from §11.5's `skillbox`, only if defined yet —
         graceful fallback when the cell hasn't been run).
      2. OAMP context card — relevant memories + recent turns + running summary.
      3. Top-k schema fact memories matching the current question.

    `k_knowledge=3` keeps the per-call prompt small enough that LLM round-trip
    latency stays under the agent_turn budget. Bump it for richer context
    once you've wired prompt caching.
    """
    parts: list[str] = []

    # Layer 1 — skill manifest (defined in §11.5 above). The `try` keeps Part 11
    # runnable even if §11.5 hasn't been executed yet on this kernel.
    try:
        manifest = build_skill_manifest(user_query, k=3)
    except NameError:
        manifest = ""
    if manifest:
        parts.append(manifest.rstrip())

    thread = get_thread(thread_id)

    # Layer 2 — OAMP context card
    card = thread.get_context_card()
    card_text = str(card) if card else ""
    if card_text:
        parts.append("## Memory context (from OAMP)")
        parts.append(card_text)

    # Layer 3 — institutional knowledge top-k
    hits = retrieve_knowledge(user_query, k=k_knowledge)
    if hits:
        parts.append("\n## Institutional knowledge (top matches)")
        for h in hits:
            parts.append(f"- ({h['kind']}) {h['subject']} - {h['body'][:280]}")

    parts.append("\n## User question")
    parts.append(user_query)
    return "\n".join(parts)


# Fire-and-forget logger. OAMP's `add_messages` runs synchronous
# extraction-LLM work when the message-extraction window fills — that's a
# 10–30 s LLM round-trip we don't want to block the agent loop on. We
# submit the write to a small pool and return immediately; any error
# surfaces on the next pool flush via `_drain_log_errors`. Persistence is
# best-effort by design — losing a log line is recoverable; blocking the
# loop on every turn is not.
_LOG_EXECUTOR = concurrent.futures.ThreadPoolExecutor(
    max_workers=4, thread_name_prefix="oamp-log")
_LOG_PENDING: list[concurrent.futures.Future] = []


def _drain_log_errors() -> None:
    """Pop any completed log futures and surface their errors. Cheap;
    runs synchronously but only over already-finished work."""
    still_pending = []
    for fut in _LOG_PENDING:
        if not fut.done():
            still_pending.append(fut)
            continue
        exc = fut.exception()
        if exc is not None:
            print(f"  ! background log_message failed: {type(exc).__name__}: {exc}")
    _LOG_PENDING[:] = still_pending


def log_message(thread_id: str, role: str, content: str) -> None:
    """Persist one chat turn to the OAMP thread without blocking.

    The submit returns immediately; the actual `add_messages` call runs on
    a background thread. We drain completed futures opportunistically so
    failures surface on the next call rather than vanishing.
    """
    from oracleagentmemory.apis.thread import Message

    def _do_log():
        get_thread(thread_id).add_messages([Message(role=role, content=content)])

    _drain_log_errors()
    _LOG_PENDING.append(_LOG_EXECUTOR.submit(_do_log))


The whole agent, in one function. Reads the prose in §11.1 (above) before this cell — it walks through the loop step-by-step. Notable design choices: every tool output is offloaded to OAMP as a `tool_output` memory and replaced inline with a compact reference, the loop has both an iteration cap and a wall-clock budget, and the guarded-stop path forces a non-tool response if budget runs out so the user always gets a final answer.


In [ ]:
def agent_turn(user_query: str, thread_id: str = "default",
                max_iterations: int = 8, budget_seconds: float = 360.0,
                verbose: bool = True) -> str:
    """Run one user turn through the loop and return the assistant's final answer.

    This is the *minimal* loop — tool outputs are inlined verbatim into the
    next message. §14.2 redefines this function to add OAMP-backed tool-output
    offload and a truncation marker so big results don't blow the context.
    """
    started = time.time()
    log_message(thread_id, "user", user_query)

    context = build_context(thread_id, user_query)
    messages: list[dict] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": context},
    ]
    tool_schemas = retrieve_tools(user_query, k=6)

    final = ""
    step = 0
    DEDUPE_WINDOW = 3
    recent_calls: list[tuple[str, str]] = []
    for step in range(max_iterations):
        if time.time() - started > budget_seconds:
            if verbose: print(f"  ! budget exhausted at iteration {step}")
            break

        resp = chat(messages, tools=tool_schemas)
        msg = resp.choices[0].message

        if not msg.tool_calls:
            final = msg.content or ""
            if verbose: print(f"  step {step}: final answer")
            break

        messages.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name,
                              "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ],
        })

        for tc in msg.tool_calls:
            name = tc.function.name
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            if verbose: print(f"  step {step}: -> {name}({args})")

            # Dedupe: short-circuit if the LLM is dispatching the same
            # (tool, args) it called within the last DEDUPE_WINDOW steps.
            # Common pathology: agent loops re-reading the same scratchpad
            # path or re-issuing the same search_knowledge query, never
            # converging until budget exhausts.
            call_key = (name, json.dumps(args, sort_keys=True))
            if call_key in recent_calls[-DEDUPE_WINDOW:]:
                output = json.dumps({
                    "note": f"duplicate call to {name} with identical args within "
                            f"the last {DEDUPE_WINDOW} dispatches — output unchanged; "
                            f"reuse the prior tool result above instead of re-dispatching."
                })
                if verbose: print(f"          ↳ duplicate dispatch — short-circuited")
            elif name not in TOOLS:
                output = json.dumps({"error": f"unknown tool: {name}"})
            else:
                fn, _ = TOOLS[name]
                try:
                    output = fn(**args)
                except Exception as e:
                    output = json.dumps({"error": f"{type(e).__name__}: {e}"})
            recent_calls.append(call_key)

            # Minimal version: full output inlined. §14.2 swaps this for the
            # offload-plus-truncation-marker pattern.
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": output})

    if not final:
        messages.append({"role": "user",
                         "content": "Budget exhausted. Provide your best answer now, no more tools."})
        resp = chat(messages, tools=None)
        final = resp.choices[0].message.content or "(no answer produced)"

    log_message(thread_id, "assistant", final)

    # Episodic / conversational memory — store the (user, assistant) pair so
    # future turns can recall this interaction via search_knowledge.
    try:
        memory_client.add_memory(
            f"User: {user_query}\n\nAssistant: {final}",
            user_id=USER_ID, agent_id=AGENT_ID,
            thread_id=thread_id,
            metadata={"kind": "episodic", "thread_id": thread_id,
                      "user_query": user_query[:240],
                      "elapsed_seconds": round(time.time() - started, 2)},
        )
    except Exception as _e:
        if verbose: print(f"  ! episodic add_memory failed: {type(_e).__name__}: {_e}")

    if verbose: print(f"  [{time.time() - started:.1f}s, {step + 1} steps]")
    return final


> ### 💡 Key takeaways — Part 11
>
> - **The whole agent is ~90 lines.** `build_context → call LLM → if tool_calls dispatch else final → log`. No framework. Read `agent_turn` once and you understand the entire control flow.
> - **Both budgets matter.** Iteration count alone lets a fast model burn money; wall-clock alone lets a slow LLM stall the cell. Use both, every loop.
> - **Tool-call dedupe prevents pathological loops.** A 3-deep recent-calls check short-circuits identical `(tool, args)` dispatches without running the tool. Cheap insurance against an LLM that loops on the same scratchpad re-read.
> - **Persist fire-and-forget.** OAMP `add_messages` runs synchronous extraction LLM work — never block the agent loop on a logging write. Submit to a pool and move on.


## Part 11.6 — JSON Relational Duality Views

*Part 10 gave the agent a vector-indexed `toolbox` (dispatchable functions). Part 11.5 added a `skillbox` (prose playbooks). Part 11.6 adds a third procedural-memory shape: **document-shaped reads of the relational schema** via Oracle 23ai/26ai's [JSON Relational Duality Views](https://docs.oracle.com/en/database/oracle/oracle-database/26/jsnvu/creating-duality-views.html).*

A duality view is a JSON projection over a set of tables joined by PK/FK/UK relationships. The same row in `voyages` is accessible as a relational tuple **and** as a nested JSON document that includes its `vessel`, `carrier`, `origin`/`destination` ports, and the array of `containers` (with their `cargo_items` nested inside). One read, no JOINs, no client-side reshaping.

**Why this matters for an LLM agent specifically.** GPT-class models reason about JSON markedly better than they reason about tabular join results. Every time the agent asks *"tell me everything about voyage 7"*, the choice today is: write a 4-table JOIN, parse the rows, mentally reassemble the document. With a duality view the agent runs `SELECT JSON_SERIALIZE(data) FROM voyage_dv WHERE _id = 7` and gets the document back already shaped. Fewer tool turns, fewer hallucinated columns.

**Why this matters for the harness specifically.** The §14.4 row policy on `voyages.ocean_region` is enforced **inside** the duality view by the kernel. An `analyst` reading `voyage_dv` for a voyage in an unauthorized region simply gets no row back — the document filtering is identical to row filtering. Same goes for the column mask on `cargo_items.unit_value_cents`: it shows up as `null` inside the nested cargo array. Trust boundary stays in the kernel, document shape is just sugar on top.

**Two duality views in this Part:**

| View | Shape | When the agent reaches for it |
|---|---|---|
| `voyage_dv` | voyage → vessel → carrier; voyage → origin/dest port; voyage → containers[] → cargo_items[] | Whole-voyage queries: "what's on voyage X?", "all cargo bound for Rotterdam" |
| `vessel_dv` | vessel → carrier; vessel → current position; vessel → current voyage | Fleet/vessel queries: "where is vessel Y?", "which Maersk vessels are at sea?" |

We then register `tool_get_document(view, key)` so the agent can fetch a full document by primary key, and `tool_query_documents(view, sql_predicate)` so it can filter the view with a SQL predicate. Both are read-only — the views are defined without `WITH UPDATE`, so DML through them is rejected by the kernel.


**Three lenses, one source of truth.** Before walking the DDL: who actually consumes a duality view, and what shape do they each want? The diagram below answers that — same `voyage_dv`, same underlying rows, three radically different access patterns and yet every byte served from one definition.

![JSON Relational Duality — three lenses, one source of truth](images/cover-duality-view.png)


**The mental model in one sentence:** the relational schema is the *truth*, the duality view is the *projection*, and which projection you read from — REST, SQL*Plus, or a tool dispatch — is just a UI choice. The kernel keeps all three in sync; your app never has to.


In [ ]:
DV_DDL = [
    "DROP VIEW IF EXISTS voyage_dv",
    "DROP VIEW IF EXISTS vessel_dv",

    # voyage_dv — the headline document. Read-only (no WITH UPDATE clause).
    """
    CREATE OR REPLACE JSON RELATIONAL DUALITY VIEW voyage_dv AS
    SELECT JSON {
      '_id'         : v.voyage_id,
      'status'      : v.status,
      'oceanRegion' : v.ocean_region,
      'departureTs' : v.departure_ts,
      'etaTs'       : v.eta_ts,
      'origin'      : (
        SELECT JSON {
          'portCode' : po.port_code,
          'name'     : po.name,
          'country'  : po.country,
          'latitude' : po.latitude,
          'longitude': po.longitude
        } FROM ports po WHERE po.port_code = v.origin_code
      ),
      'destination' : (
        SELECT JSON {
          'portCode' : pd.port_code,
          'name'     : pd.name,
          'country'  : pd.country,
          'latitude' : pd.latitude,
          'longitude': pd.longitude
        } FROM ports pd WHERE pd.port_code = v.dest_code
      ),
      'vessel'      : (
        SELECT JSON {
          'vesselId'    : ve.vessel_id,
          'name'        : ve.name,
          'imoNumber'   : ve.imo_number,
          'vesselType'  : ve.vessel_type,
          'capacityTeu' : ve.capacity_teu,
          'flagCountry' : ve.flag_country,
          'carrier'     : (
            SELECT JSON {
              'carrierId' : ca.carrier_id,
              'name'      : ca.name,
              'hqCountry' : ca.hq_country
            } FROM carriers ca WHERE ca.carrier_id = ve.carrier_id
          )
        } FROM vessels ve WHERE ve.vessel_id = v.vessel_id
      ),
      'containers'  : [
        SELECT JSON {
          'containerId'   : c.container_id,
          'containerNo'   : c.container_no,
          'containerType' : c.container_type,
          'consignor'     : c.consignor,
          'consignee'     : c.consignee,
          'status'        : c.status,
          'cargo'         : [
            SELECT JSON {
              'cargoItemId'    : ci.cargo_item_id,
              'hsCode'         : ci.hs_code,
              'description'    : ci.description,
              'quantity'       : ci.quantity,
              'unitValueCents' : ci.unit_value_cents,
              'weightKg'       : ci.weight_kg
            } FROM cargo_items ci WHERE ci.container_id = c.container_id
          ]
        } FROM containers c WHERE c.voyage_id = v.voyage_id
      ]
    } FROM voyages v
    """,

    # vessel_dv — fleet-centric. Includes current position + active voyage if any.
    """
    CREATE OR REPLACE JSON RELATIONAL DUALITY VIEW vessel_dv AS
    SELECT JSON {
      '_id'          : v.vessel_id,
      'name'         : v.name,
      'imoNumber'    : v.imo_number,
      'vesselType'   : v.vessel_type,
      'capacityTeu'  : v.capacity_teu,
      'yearBuilt'    : v.year_built,
      'flagCountry'  : v.flag_country,
      'carrier'      : (
        SELECT JSON {
          'carrierId' : c.carrier_id,
          'name'      : c.name,
          'hqCountry' : c.hq_country
        } FROM carriers c WHERE c.carrier_id = v.carrier_id
      ),
      'position'     : (
        SELECT JSON {
          'positionTs' : p.position_ts,
          'latitude'   : p.latitude,
          'longitude'  : p.longitude,
          'speedKnots' : p.speed_knots,
          'headingDeg' : p.heading_deg
        } FROM vessel_positions p WHERE p.vessel_id = v.vessel_id
      )
    } FROM vessels v
    """,
]

# DROP VIEW IF EXISTS isn't valid Oracle syntax — translate manually.
for stmt in DV_DDL:
    if stmt.strip().upper().startswith("DROP VIEW IF EXISTS"):
        view_name = stmt.split()[-1]
        try:
            with demo_conn.cursor() as cur:
                cur.execute(f"DROP VIEW {view_name}")
        except oracledb.DatabaseError:
            pass
        continue
    try:
        with demo_conn.cursor() as cur:
            cur.execute(stmt)
        # Identify which view we just created from the DDL
        head = stmt.strip().split("\n", 1)[0]
        print(f"OK: {head[:80]}")
    except oracledb.DatabaseError as e:
        code_ = e.args[0].code
        head = stmt.strip().split("\n", 1)[0][:80]
        if code_ in (900, 901, 922, 2000):
            print(f"!! {head!r}: duality-view syntax not on this image (ORA-{code_:05d}).")
            print("   The tool_get_document fallback below will return an error pointing")
            print("   the agent at run_sql instead.")
        else:
            raise
demo_conn.commit()

# Verify the views exist
with agent_conn.cursor() as cur:
    cur.execute(
        "SELECT view_name FROM all_views "
        " WHERE owner = :o AND view_name IN ('VOYAGE_DV', 'VESSEL_DV')",
        o=DEMO_USER,
    )
    print("duality views in", DEMO_USER, ":", [r[0] for r in cur])


#### 11.6.1 Tools: `get_document` and `query_documents`

Two tools so the agent can use the views. Both go through `agent_conn` so the §14.4 DDS / DBMS_RLS row and column policies apply transparently.


In [ ]:
@register
def tool_get_document(view: str, key: str) -> str:
    """Read one full document from a JSON Relational Duality View by primary key.
    Use this instead of writing JOINs whenever you need the full shape of an entity
    (a voyage with its vessel/carrier/ports/containers/cargo, or a vessel with its
    carrier/position). Returns a JSON document.

    `view` must be one of: voyage_dv, vessel_dv.
    `key` is the value of the document _id (numeric voyage_id or vessel_id, as a string).
    """
    allowed = {"voyage_dv", "vessel_dv"}
    if view not in allowed:
        return json.dumps({"error": f"unknown view {view!r}; allowed: {sorted(allowed)}"})
    try:
        with agent_conn.cursor() as cur:
            cur.execute(
                f"SELECT JSON_SERIALIZE(data PRETTY) FROM {DEMO_USER}.{view} WHERE data.\"_id\" = :k",
                k=int(key) if str(key).isdigit() else key,
            )
            row = cur.fetchone()
        if not row:
            return json.dumps({"error": f"no document with _id={key} in {view}"})
        body = row[0].read() if hasattr(row[0], "read") else str(row[0])
        return body
    except Exception as e:
        return json.dumps({"error": f"{type(e).__name__}: {e}"})


@register
def tool_query_documents(view: str, where: str = "1=1", max_rows: int = 10) -> str:
    """Filter a JSON Relational Duality View with a SQL predicate.
    Use when you want a list of documents matching some condition without writing
    JOINs by hand. The predicate references underlying-table columns of the view's
    root table (e.g. status, ocean_region for voyage_dv; vessel_type for vessel_dv).

    `view` must be one of: voyage_dv, vessel_dv.
    `where` is a SQL boolean expression on the root table's columns (default '1=1').
    `max_rows` caps the result set.
    """
    allowed = {"voyage_dv", "vessel_dv"}
    if view not in allowed:
        return json.dumps({"error": f"unknown view {view!r}; allowed: {sorted(allowed)}"})
    sql = f"SELECT JSON_SERIALIZE(data) FROM {DEMO_USER}.{view} WHERE {where} FETCH FIRST :n ROWS ONLY"
    try:
        with agent_conn.cursor() as cur:
            cur.execute(sql, n=max_rows)
            docs = [(r[0].read() if hasattr(r[0], "read") else str(r[0])) for r in cur]
        return json.dumps({"count": len(docs), "documents": [json.loads(d) for d in docs]}, default=str)
    except Exception as e:
        return json.dumps({"error": f"{type(e).__name__}: {e}", "sql": sql})


print(f"registered: get_document, query_documents  (TOOLS total: {len(TOOLS)})")


#### 11.6.2 Demo: same question, three paths

Same NL question — *"give me everything about voyage 7"* — answered three ways so you can see what the duality view buys. Watch the trace.


In [ ]:
demo_thread_dv = "demo-dv-1"

q_dv = (
    "Give me a complete picture of voyage_id 7: which carrier and vessel is running it, "
    "what ports it's between, what's loaded on it (containers and cargo items), and the "
    "vessel's current position. Use the duality view if you have one — that's why we built it."
)

print("=" * 70)
print(q_dv)
print("=" * 70)
print(agent_turn(q_dv, thread_id=demo_thread_dv))


#### 11.6.3 DDS interaction — the document inherits the row policy

The §14.4 row policy on `voyages.ocean_region` filters which voyages are visible. That filter applies *inside* the duality view too — query `voyage_dv` as a region-restricted user and you simply don't see voyages outside your authorization. Same for the cargo-value column mask: declared values inside the nested `cargo` array come back as `null` for non-EXECUTIVE clearance.

Below: same `get_document(voyage_dv, K)` for some `K` whose voyage is in `MEDITERRANEAN`, asked as a CFO (allowed) and an APAC fleet manager (denied). The denied case gets back an error, not a leaked document.


In [ ]:
# Find a voyage in MEDITERRANEAN
with agent_conn.cursor() as cur:
    cur.execute(
        "SELECT voyage_id FROM SUPPLYCHAIN.voyages WHERE ocean_region = 'MEDITERRANEAN' "
        " AND status IN ('ACTIVE','DELAYED') FETCH FIRST 1 ROW ONLY"
    )
    row = cur.fetchone()
mediterranean_voyage = row[0] if row else None
print(f"Mediterranean voyage chosen: {mediterranean_voyage}")

# This demo depends on §14.4 already having run (it defines set_identity, the
# EDA_CTX namespace, and the voyages_region_policy DBMS_RLS row policy).
# If you haven't reached §14.4 yet, skip — and re-run this cell after §14.4.4
# to see the kernel-level row filtering applied through the duality view.
if "set_identity" not in globals():
    print()
    print("(skipped DDS interaction — §14.4 hasn't been run yet on this kernel.")
    print(" After running §14.4.3 (set_identity helper) and §14.4.4 (DBMS_RLS")
    print(" policies), come back and re-run this cell.)")
elif mediterranean_voyage is not None:
    print()
    print("--- as CFO (regions=ALL) ---")
    set_identity("cfo@acme.com", "EXECUTIVE")
    print(tool_get_document(view="voyage_dv", key=str(mediterranean_voyage))[:1500])

    print("\n--- as APAC fleet (regions=PACIFIC+INDIAN, no Mediterranean) ---")
    set_identity("apac.fleet@acme.com", "STANDARD")
    print(tool_get_document(view="voyage_dv", key=str(mediterranean_voyage))[:1500])

    set_identity(None)  # don't leak identity to next caller


#### 11.6.4 Writable duality views with ETag-based concurrency

Everything we've done with `voyage_dv` so far has been **read** — fetch a document by `_id`, filter with a SQL predicate, watch the §14.4 row policy propagate into the nested arrays. That's half of what duality views are for. The other half is **write**: the same row that you read as a JSON document is *updatable* through the document — `UPDATE voyage_dv SET data = …` writes back to the underlying tables, atomically, with the row policy still enforced on the resulting rows.

Two pieces make that safe in a multi-writer world:

| Piece | Purpose |
|---|---|
| `WITH UPDATE` clause on the DV definition | Tells the kernel the view is writable; without it, an UPDATE through the view raises `ORA-42692`. Granular forms (`WITH INSERT`, `WITH DELETE`, per-column annotations) restrict further |
| **ETag** — `_metadata.etag` field on every retrieved document | Optimistic concurrency control. Each row has an ETag (a hash of its current state). When you PUT a modified doc back, you include the ETag you originally read; if the row's current ETag doesn't match (because someone else wrote first), the kernel raises `ORA-42699` and your update is rejected |

This is the same model HTTP uses for `If-Match` headers — but it's enforced **inside the database**, by the SQL engine, on every UPDATE through the duality view. No application-layer locking, no SELECT-FOR-UPDATE fan-out, no client-side cache reconciliation logic.

**Why we don't expose this to the agent.** The agent harness is deliberately read-only: `tool_run_sql` rejects DDL/DML, and `tool_get_document` / `tool_query_documents` only `SELECT`. Adding an `update_document` tool would re-open the write path we closed — a hostile prompt could rewrite voyage status or cargo manifests. The cells below are SQL-level demonstrations of the duality mechanic, not agent capabilities. If you decide to give your agent narrow write authority later (e.g., status updates only, gated by an identity check), this is the foundation you'd build on.

Three demos below: the DDL that opens up writes; a clean read-modify-PUT round trip; and a deliberate two-reader conflict where the stale writer's ETag mismatch surfaces as `ORA-42699`.


In [ ]:
# A SECOND duality view, dedicated to the writable demo.
# Only voyage_id (the key) and status are exposed — keeping the surface narrow
# limits what an UPDATE through this view can change. ocean_region stays
# read-only so the §14.4 row policy can't be circumvented by mutating it
# through the JSON layer.
DV_WRITABLE_DDL = """
CREATE OR REPLACE JSON RELATIONAL DUALITY VIEW voyage_status_dv AS
SELECT JSON {
  '_id'         : v.voyage_id,
  'status'      : v.status,
  'oceanRegion' : v.ocean_region,
  'departureTs' : v.departure_ts,
  'etaTs'       : v.eta_ts
} FROM voyages v WITH UPDATE
"""

try:
    with demo_conn.cursor() as cur:
        cur.execute(DV_WRITABLE_DDL)
        # AGENT was granted SELECT ANY TABLE in §4 so reads work, but UPDATE
        # through a duality view requires the explicit UPDATE privilege on
        # the view itself (ORA-41900 otherwise). Owner side grants it here.
        cur.execute(f"GRANT SELECT, UPDATE ON voyage_status_dv TO {AGENT_USER}")
    demo_conn.commit()
    print(f"OK: created voyage_status_dv WITH UPDATE; granted UPDATE to {AGENT_USER}")
except oracledb.DatabaseError as e:
    code_ = e.args[0].code
    if code_ in (900, 901, 922, 2000):
        print(f"!! duality-view DDL not supported on this image (ORA-{code_:05d}). "
              "The §11.6.4 demos below will skip if voyage_status_dv isn't present.")
    else:
        raise

# Quick check that it landed
with agent_conn.cursor() as cur:
    cur.execute(
        "SELECT view_name FROM all_views WHERE owner = :o AND view_name = 'VOYAGE_STATUS_DV'",
        o=DEMO_USER)
    rows = list(cur)
print(f"voyage_status_dv exists: {bool(rows)}")


In [ ]:
import json as _json_dv

# Pick a voyage to round-trip. We capture the initial status so the cell is
# idempotent — we restore it at the end and the rest of the notebook sees no
# side effect.
TARGET_VOYAGE = 1


def _read_doc(voyage_id):
    """Fetch one document from voyage_status_dv. The result includes
    _metadata.etag, which the kernel sets per-row."""
    with agent_conn.cursor() as cur:
        cur.execute(
            # Aliasing the view as `t` is required so Oracle parses
            # `t.data."_id"` as a JSON path step rather than ambiguous
            # <table>.<column> syntax (which raises ORA-00904).
            f"SELECT JSON_SERIALIZE(t.data PRETTY) "
            f"  FROM {DEMO_USER}.voyage_status_dv t "
            f' WHERE t.data."_id" = :k',
            k=voyage_id,
        )
        row = cur.fetchone()
    if not row:
        return None
    raw = row[0]
    if hasattr(raw, "read"):
        raw = raw.read()
    return _json_dv.loads(raw)


def _put_doc(voyage_id, doc):
    """PUT a modified document back through the DV. The kernel compares the
    doc's _metadata.etag against the row's current etag and rejects the UPDATE
    with ORA-42699 if they differ."""
    with agent_conn.cursor() as cur:
        cur.execute(
            f"UPDATE {DEMO_USER}.voyage_status_dv t "
            f"   SET t.data = :new_doc "
            f' WHERE t.data."_id" = :k',
            new_doc=_json_dv.dumps(doc), k=voyage_id,
        )
        rc = cur.rowcount
    agent_conn.commit()
    return rc


# -- Read --
doc = _read_doc(TARGET_VOYAGE)
if doc is None:
    print(f"voyage {TARGET_VOYAGE} not found — nothing to demo")
else:
    initial_status = doc["status"]
    initial_etag = doc.get("_metadata", {}).get("etag")
    print(f"Initial state of voyage {TARGET_VOYAGE}:")
    print(f"  status: {initial_status}")
    print(f"  etag:   {initial_etag}")

    # -- Modify in memory --
    doc["status"] = "DELAYED" if initial_status != "DELAYED" else "ACTIVE"
    new_status = doc["status"]
    print(f"\nFlipping status -> {new_status} and PUTting back with the matching etag...")

    # -- PUT back --
    try:
        rc = _put_doc(TARGET_VOYAGE, doc)
        print(f"  rows updated: {rc}")
    except oracledb.DatabaseError as e:
        print(f"  PUT failed: {e}")
        rc = 0

    # -- Verify --
    after = _read_doc(TARGET_VOYAGE)
    if after:
        print(f"\nAfter PUT — status: {after['status']}, "
              f"new etag: {after.get('_metadata', {}).get('etag')}")
        print(f"  (etag changed: {after.get('_metadata', {}).get('etag') != initial_etag})")

    # -- Restore (so the rest of the notebook sees no drift) --
    after["status"] = initial_status
    _put_doc(TARGET_VOYAGE, after)
    print(f"\nRestored status to {initial_status} (cell is idempotent on re-run).")


In [ ]:
# Conflict demo: two readers grab the same doc. First writer commits, second
# writer's ETag is now stale and the kernel rejects the UPDATE.
TARGET_VOYAGE = 2

original = _read_doc(TARGET_VOYAGE)
if original is None:
    print(f"voyage {TARGET_VOYAGE} not found — nothing to demo")
else:
    initial_status = original["status"]
    print(f"Initial state of voyage {TARGET_VOYAGE}: status={initial_status}, "
          f"etag={original.get('_metadata', {}).get('etag')}")

    # Two readers, same etag (because they read at the same logical state).
    reader_a = _read_doc(TARGET_VOYAGE)
    reader_b = _read_doc(TARGET_VOYAGE)
    print(f"\nReader A and Reader B both have etag={reader_a.get('_metadata',{}).get('etag')}")

    # Writer A commits first — succeeds.
    reader_a["status"] = "DELAYED" if initial_status != "DELAYED" else "ACTIVE"
    print(f"\nWriter A: PUT status={reader_a['status']}")
    try:
        rc = _put_doc(TARGET_VOYAGE, reader_a)
        print(f"  rows updated: {rc}  (success — etag matched)")
    except oracledb.DatabaseError as e:
        print(f"  unexpected failure: {e}")

    # Writer B tries to PUT with the *original* (now stale) etag — should fail.
    reader_b["status"] = "COMPLETED"
    print(f"\nWriter B: PUT status={reader_b['status']} (with stale etag)")
    try:
        rc = _put_doc(TARGET_VOYAGE, reader_b)
        print(f"  rows updated: {rc}  (this should NOT have succeeded)")
    except oracledb.DatabaseError as e:
        code_ = e.args[0].code
        if code_ == 42699:
            print(f"  REJECTED with ORA-42699 — stale etag, exactly as expected.")
            print(f"  Writer B would now read again, re-apply its change, and retry.")
        else:
            print(f"  REJECTED with ORA-{code_:05d}: {e}")

    # Restore for cell idempotency
    final = _read_doc(TARGET_VOYAGE)
    final["status"] = initial_status
    _put_doc(TARGET_VOYAGE, final)
    print(f"\nRestored status of voyage {TARGET_VOYAGE} to {initial_status}.")


**ETag conflict sequence** — the demo cell below scripts exactly what the timeline shows. Two readers, two writers, second writer's stale ETag is rejected by the kernel.

```
ETAG-BASED OPTIMISTIC CONCURRENCY — TWO-WRITER CONFLICT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Time   Reader A                        Reader B                        Database row
────   ──────────────────              ──────────────────              ─────────────────
T0     SELECT data FROM dv             SELECT data FROM dv             status=ACTIVE
       WHERE _id=2                     WHERE _id=2                     etag=E1
            ↓                                ↓
       doc_a:                          doc_b:
       { _id: 2,                       { _id: 2,
         status: "ACTIVE",               status: "ACTIVE",
         _metadata: {                    _metadata: {
           etag: E1 } }                    etag: E1 } }


T1     doc_a.status = "DELAYED"
       UPDATE dv
         SET data = doc_a   ───────────────────────────────────────►   match: E1 == E1
         WHERE _id = 2                                                  ✓ apply: DELAYED
                                                                        ✓ rotate etag → E2
            ◄─── ROWCOUNT=1                                            status=DELAYED
                                                                        etag=E2


T2                                     doc_b.status = "COMPLETED"
                                       UPDATE dv
                                         SET data = doc_b   ─────►     mismatch: E1 ≠ E2
                                         WHERE _id = 2                  ✗ ORA-42699
                                                                        (stale etag)
                                            ◄─── exception              row unchanged:
                                                                        status=DELAYED
                                                                        etag=E2


T3                                     retry path:
                                         SELECT again → doc_b' has E2
                                         re-apply change → "COMPLETED"
                                         UPDATE again with fresh etag
                                       (or merge & resolve in-app)


WHAT MAKES THIS WORK:
   • _metadata.etag is set BY the kernel on every retrieved document.
   • UPDATE compares the etag in the new doc to the row's current etag.
   • ETags rotate atomically with the row's data on every successful write.
   • Mismatch raises ORA-42699 — no half-update, no overwriting newer state.
```


**Where this earns its keep over plain JSON views:**

- `JSON_OBJECT`-based views are read-only by construction; you'd write to the underlying tables directly, which means the *application* has to know which JSON path maps to which `(table, column)`. Duality views invert that: the *kernel* knows.
- Optimistic concurrency through ETags is server-side. With plain JSON views you'd need `SELECT … FOR UPDATE`, version columns, or application-level locks — all of which break down across stateless services.
- The §14.4 row policy on `voyages.ocean_region` filters writes too, not just reads. A regional fleet manager can't accidentally PUT a status update on a voyage outside their authorized oceans because their `SELECT` doesn't return the row in the first place.

**What we still don't expose to the agent.** The harness keeps `tool_run_sql` SELECT-only and there's no `tool_update_document`. If you wanted to give the agent narrow write authority — say, "let `cfo@acme.com` mark voyages as DELAYED" — you'd register a tool that calls `_put_doc` above, gated by an identity check that reads `EDA_CTX.CLEARANCE`. The duality view + ETag + DDS row policy stack is what makes that *safe* without writing application-layer security code.


> ### 💡 Key takeaways — Part 11.6
>
> - **One row, two equally-valid shapes.** A duality view serves the SAME data as a relational tuple AND a JSON document — kernel-managed, no app-layer mapping. The relational schema is the truth; the JSON shape is the contract.
> - **Three personas, one source.** A DBA, an AI developer, and an AI agent all consume the same row through their preferred lens (SQL, REST/JSON, tool dispatch). Defining the projection once kills the per-consumer mapping code.
> - **Writable views need `WITH UPDATE`.** Without it, `UPDATE …` raises `ORA-42692`. With it, you get server-side optimistic concurrency for free — every doc carries `_metadata.etag`, stale writes raise `ORA-42699`.
> - **DDS row policies still apply through DVs.** A user can't escape the kernel's row filter by writing JSON instead of SQL — the kernel re-checks at the base table on every PUT.


## Part 12 — Continuous scans: keeping institutional knowledge fresh

*Part 5 ran the scanner once on demand. Part 12 moves it onto a schedule — `DBMS_SCHEDULER` queues scan requests, the harness drains them when it's running. Schemas drift; this Part keeps the agent's institutional knowledge from going stale.*

An enterprise database is a moving target. Tables get added, columns get renamed, comments get written *after* launch. A one-shot scan on day zero is not enough. We need the agent's institutional knowledge to track reality, and we'd prefer not to beat up the primary database with hourly catalog sweeps while we're at it.

Two moves in this section:

1. **Incremental scans** — already cheap, thanks to the `body_hash` check from §5.3.
2. **Periodic scans via `DBMS_SCHEDULER`** — let the database itself drive a cadence so re-scans keep running when the harness is gone.ainer; we wire the harness's read path through it so scans and user SQL don't touch the primary.
3. **Periodic scheduler** — a plain background thread that calls `run_scan()` on an interval. In production you'd use `dbms_scheduler` or a K8s CronJob; the shape is the same.


### 12.1 Incremental scans are already cheap

The `body_hash` check from §5.3 means a re-scan only re-embeds facts whose underlying text changed. So running `run_scan(conn, "SALES")` every hour is viable — the vast majority of calls will hash-check and skip. That's the minimum you need for continuous knowledge; everything below makes it *operationally* nicer.


### 12.2 Periodic scans via `DBMS_SCHEDULER`

A scan you only run manually isn't very institutional — schemas drift, new tables land, query patterns change. We want re-scans on a cadence. The natural place for that schedule is **inside the database** via `DBMS_SCHEDULER`.

Two reasons to put the schedule there rather than in an outside loop:

- **Lifecycle.** The DB scheduler keeps running when the harness kernel is gone, when your laptop is asleep, when you reboot.
- **Operational surface.** `DBA_SCHEDULER_JOB_RUN_DETAILS` is a real audit trail; you can query it, alert on failures, and suspend the job during a freeze.

**Implementation choice.** `DBMS_SCHEDULER` runs PL/SQL, not our Python `scan_schema`. Two ways to bridge:

1. **Rewrite the scanner in PL/SQL.** Mine the catalog views, build the fact body in PL/SQL, INSERT directly into the OAMP memory table with `VECTOR_EMBEDDING(...)` for the embedding column. Zero Python in the loop — fully autonomous. It's the right end state, but it couples to OAMP's internal table layout (the `eda_onnx_*` schema).
2. **Use the scheduler as a *trigger*.** A tiny PL/SQL proc just *requests* a scan by writing a queued `scan_history` row. The harness, when next running, polls for queued rows and runs `run_scan(owner)` on each. This keeps the scan logic in one place (`scan_schema`), gives you a durable schedule, and stays decoupled from OAMP's internals.

We wire #2 below as the default. The cell is opt-in via `RUN_SCHEDULER=1` so it doesn't create a job on every kernel run. The PL/SQL-rewrite path is a one-day exercise once you've decided the trade is worth it.


In [ ]:
os.environ["RUN_SCHEDULER"] = "1"

The classic decision when scheduling agent work: rewrite in PL/SQL, or use the scheduler as a *trigger* and keep the logic in Python. We pick #2 — `DBMS_SCHEDULER` inserts a queued row into `scan_history`, and the harness's `drain_queued_scans()` consumes it when it's running. The Python scanner stays in one place; the database owns the cadence.

First, a one-time housekeeping migration on `scan_history.notes` (the next cell). The original DDL made it a `CLOB`, but the scheduler's queue check filters `WHERE notes = 'queued-by-scheduler'` and ORA-22848 forbids CLOB equality comparisons. Idempotent — only runs if the column is still CLOB.


In [ ]:
# Migrate existing scan_history.notes (CLOB) -> VARCHAR2(4000) so the
# `WHERE notes = 'queued-by-scheduler'` lookup below works (ORA-22848: cannot
# use CLOB type as comparison key).  Idempotent: only runs if the column is
# still CLOB.
with agent_conn.cursor() as cur:
    cur.execute(
        "SELECT data_type FROM user_tab_columns "
        " WHERE table_name = 'SCAN_HISTORY' AND column_name = 'NOTES'"
    )
    row = cur.fetchone()
    if row and row[0] == 'CLOB':
        cur.execute("ALTER TABLE scan_history RENAME COLUMN notes TO notes_clob")
        cur.execute("ALTER TABLE scan_history ADD notes VARCHAR2(4000)")
        cur.execute("UPDATE scan_history SET notes = DBMS_LOB.SUBSTR(notes_clob, 4000, 1)")
        cur.execute("ALTER TABLE scan_history DROP COLUMN notes_clob")
        agent_conn.commit()
        print("migrated scan_history.notes  CLOB -> VARCHAR2(4000)")
    else:
        print(f"scan_history.notes already {row[0] if row else '?'}; no migration needed")


With the migration done, install the scheduler. The cell is opt-in via `RUN_SCHEDULER=1` so it doesn't create a job on every kernel run. `DBMS_SCHEDULER` invokes `AGENT_REQUEST_SCAN`, which queues a row; the harness drains queued rows on its own cadence.


In [ ]:
import os

if os.environ.get("RUN_SCHEDULER", "0") != "1":
    print("Skipping DBMS_SCHEDULER setup. Set RUN_SCHEDULER=1 to install it.")
else:
    SCAN_PROC = "AGENT_REQUEST_SCAN"
    SCAN_JOB  = "AGENT_PERIODIC_SCAN"
    SCAN_INTERVAL_MIN = int(os.environ.get("SCAN_INTERVAL_MIN", "60"))
    SCAN_OWNER        = os.environ.get("SCAN_OWNER", DEMO_USER)

    # 1. Create the stored proc that records a "scan requested" row in scan_history.
    #    The harness side drains queued rows below; once consumed, run_scan rewrites the
    #    same row with the actual outcome.
    proc_sql = f"""
    CREATE OR REPLACE PROCEDURE {SCAN_PROC}(p_owner IN VARCHAR2) AS
    BEGIN
      INSERT INTO scan_history (target_owner, notes)
      VALUES (UPPER(p_owner), 'queued-by-scheduler');
      COMMIT;
    END;
    """
    with agent_conn.cursor() as cur:
        cur.execute(proc_sql)
    print(f"created procedure {SCAN_PROC}")

    # 2. (Re-)create the scheduler job.
    with agent_conn.cursor() as cur:
        # Drop existing if any (idempotent).
        try:
            cur.execute("BEGIN DBMS_SCHEDULER.DROP_JOB(:n, force => TRUE); END;",
                        n=SCAN_JOB)
        except oracledb.DatabaseError:
            pass

        cur.execute(
            "BEGIN "
            "  DBMS_SCHEDULER.CREATE_JOB("
            "    job_name        => :job_name, "
            "    job_type        => 'STORED_PROCEDURE', "
            "    job_action      => :proc, "
            "    number_of_arguments => 1, "
            "    start_date      => SYSTIMESTAMP, "
            "    repeat_interval => :rep, "
            "    enabled         => FALSE); "
            "  DBMS_SCHEDULER.SET_JOB_ARGUMENT_VALUE("
            "    job_name => :job_name, argument_position => 1, argument_value => :owner); "
            "  DBMS_SCHEDULER.ENABLE(:job_name); "
            "END;",
            job_name=SCAN_JOB,
            proc=SCAN_PROC,
            rep=f"FREQ=MINUTELY; INTERVAL={SCAN_INTERVAL_MIN}",
            owner=SCAN_OWNER,
        )
    agent_conn.commit()
    print(f"scheduled job {SCAN_JOB} -> {SCAN_PROC}({SCAN_OWNER!r}) every {SCAN_INTERVAL_MIN} min")


# Harness-side drain: run scans for any queued requests since last call.
# Call this from your agent loop, a worker process, or just inline before the demo.
def drain_queued_scans(verbose: bool = True) -> int:
    """Process every 'queued-by-scheduler' row in scan_history.

    For each queued row, run the Python scanner (`run_scan`) and update the row's
    notes/finished_at fields. Returns the number of scans actually run.
    """
    with agent_conn.cursor() as cur:
        cur.execute(
            "SELECT scan_id, target_owner FROM scan_history "
            " WHERE notes = 'queued-by-scheduler' "
            "   AND finished_at IS NULL "
            " ORDER BY started_at"
        )
        queued = list(cur)

    ran = 0
    for scan_id, owner in queued:
        try:
            summary = run_scan(agent_conn, owner=owner)
            with agent_conn.cursor() as cur:
                cur.execute(
                    "UPDATE scan_history SET notes = :n WHERE scan_id = :id",
                    n=f"drained: new={summary['new']} updated={summary['updated']} "
                      f"skipped={summary['skipped']}",
                    id=scan_id,
                )
            agent_conn.commit()
            ran += 1
            if verbose: print(f"[drain] scanned {owner}: {summary}")
        except Exception as e:
            print(f"[drain] {owner} failed: {e}")
    return ran


# Sanity: how many queued requests are pending right now?
with agent_conn.cursor() as cur:
    cur.execute(
        "SELECT COUNT(*) FROM scan_history "
        " WHERE notes = 'queued-by-scheduler' AND finished_at IS NULL"
    )
    pending = cur.fetchone()[0]
print(f"queued scans pending: {pending} (run drain_queued_scans() to process)")


> ### 💡 Key takeaways — Part 12
>
> - **Schemas drift; institutional knowledge must drift with them.** A one-shot scan on day zero rots. The agent's "what tables exist" answer must track reality, not training-day reality.
> - **Scheduler-as-trigger beats rewriting in PL/SQL.** `DBMS_SCHEDULER` queues a row, the harness drains it on its own cadence — Python logic stays in one place, the database owns the schedule.
> - **Lifecycle outlives the kernel.** A `dbms_scheduler` job keeps running when your laptop sleeps, when the kernel restarts, when you bounce the container. `DBA_SCHEDULER_JOB_RUN_DETAILS` is the audit trail.


## Part 13 — End-to-end demo

*Parts 1–12 built every component. Part 13 runs the agent against the real DEMO schema — five turns that exercise institutional knowledge retrieval (Part 5), live SQL (Part 10), MLE compute (Part 9), persisted corrections (Part 4), and the full agent loop (Part 11). Read the trace from the model's perspective: each step is a deliberate choice that maps to one of the earlier Parts.*

We now put every piece together:

1. We talk to the agent in natural language.
2. It retrieves from institutional knowledge, runs read-only SQL, writes intermediate notes to the scratchpad, and commits corrections back.
3. Between turns, nothing resets — conversation, tool log, and institutional knowledge all persist in Oracle.


In [ ]:
thread = "demo-session-1"

q1 = "What's in the SUPPLYCHAIN schema? Briefly - list the entities and how they relate to each other."
print("USER:", q1)
print("\nASSISTANT:", agent_turn(q1, thread_id=thread))


In [ ]:
q2 = "How many active voyages does each carrier currently have? Show me a small table sorted by count desc."
print("USER:", q2)
print("\nASSISTANT:", agent_turn(q2, thread_id=thread))


In [ ]:
q3 = ("Important: in the SUPPLYCHAIN schema, vessels.capacity_teu is always in 20-foot equivalent units (TEU) - "
      "never tons, never cubic meters. And cargo_items.unit_value_cents is always USD CENTS, never dollars. "
      "Save EACH of these as a separate 'correction' memory by calling the remember tool BEFORE you respond, "
      "then confirm with the memory IDs you got back.")
print("USER:", q3)
print("\nASSISTANT:", agent_turn(q3, thread_id=thread))


In [ ]:
q4 = "What's the total declared cargo value, in USD, currently in transit (across all ACTIVE and DELAYED voyages)?"
print("USER:", q4)
print("\nASSISTANT:", agent_turn(q4, thread_id=thread))


In [ ]:
q5 = (
    "Pull the current latitude/longitude of every vessel that's in transit. Then use exec_js to compute "
    "the centroid (mean lat, mean lon) of the fleet and tell me which vessel is the FARTHEST from that "
    "centroid (compute distance in nautical miles via the haversine formula). Show the JS in your trace."
)
print("USER:", q5)
print("\nASSISTANT:", agent_turn(q5, thread_id=thread))


### 13.0.1 Scratchpad — proving the agent uses it

The §11 system prompt now mandates the DBFS scratchpad for any non-trivial SQL or multi-step analysis (rule #5). Two cells below: a multi-part prompt that should drive several `scratch_write` / `scratch_read` calls, then a verification block that:

1. Pulls every tool output for that thread out of OAMP and counts `scratch_*` calls.
2. Reads each scratched path back from DBFS directly — independent of the agent — to prove the bytes actually landed in the database.

If `scratch_write` calls show up but the DBFS read returns nothing, the agent is fabricating its trace. If the DBFS read shows the same bytes the trace claimed, the scratchpad is in use end-to-end.


In [ ]:
scratch_thread = "demo-scratch-1"

scratch_q = (
    "I need a step-by-step analysis of our top-5 most valuable in-transit voyages, "
    "broken out by ocean region. "
    "Use the scratchpad: "
    "(1) write a short plan to /scratch/voyages-plan.md before doing anything, "
    "(2) draft your final SQL to /scratch/voyages-q.sql, then scratch_read it before "
    "passing it to run_sql, "
    "(3) keep findings in /scratch/voyages-findings.md as you discover them. "
    "Then summarise."
)
print("USER:", scratch_q[:200] + "...")
print()
print("ASSISTANT:", agent_turn(scratch_q, thread_id=scratch_thread, max_iterations=12))


#### Proof: trace + on-disk DBFS bytes

Two independent checks below.

**Check A — trace.** OAMP holds every tool call's output as a memory tagged `kind=tool_output` with `tool_name` and `tool_args`. We pull those for the thread and count `scratch_*`.

**Check B — DBFS bytes.** Independent of the trace, we read each scratched path back through the §7.2 DBFS wrapper. If the path exists with content, the write actually committed to SecureFile LOBs in the database — not just claimed by the model.


In [ ]:
import json as _json_proof

# Check A: pull every tool output on this thread, count scratchpad calls.
recent = memory_client._store.list(
    "memory",
    user_id=USER_ID, agent_id=AGENT_ID,
    thread_id=scratch_thread,
    metadata_filter={"kind": "tool_output"},
    limit=50,
)

scratch_writes, scratch_reads, other_tools = [], [], 0
for m in recent or []:
    meta = m.metadata or {}
    name = meta.get("tool_name", "")
    raw_args = meta.get("tool_args", "")
    try:
        args = _json_proof.loads(raw_args) if raw_args.startswith("{") else {}
    except Exception:
        args = {}
    if name == "scratch_write":
        scratch_writes.append(args)
    elif name == "scratch_read":
        scratch_reads.append(args)
    else:
        other_tools += 1

print("=== Check A — trace ===")
print(f"  scratch_write calls: {len(scratch_writes)}")
for w in scratch_writes:
    p = w.get("path", "?")
    n = len(w.get("content", "")) if "content" in w else "?"
    print(f"    -> wrote {p}  ({n} bytes)")
print(f"  scratch_read  calls: {len(scratch_reads)}")
for r in scratch_reads:
    print(f"    -> read  {r.get('path', '?')}")
print(f"  other tool calls:    {other_tools}")
print()

# Check B: read each scratched path back from DBFS directly.
print("=== Check B — bytes on disk in DBFS ===")
written_paths = sorted({w.get("path", "") for w in scratch_writes if w.get("path")})
if not written_paths:
    print("  (no paths to verify - agent didn't use scratch_write)")
for path in written_paths:
    try:
        content = scratch.read(path)
        print(f"  {path}  [{len(content)} bytes on disk]")
        preview = content[:300].replace("\n", " | ")
        print(f"    preview: {preview}")
    except FileNotFoundError:
        print(f"  {path}  !! NOT FOUND in DBFS - write didn't commit")
    except Exception as e:
        print(f"  {path}  !! error: {type(e).__name__}: {e}")

print()
if scratch_writes and written_paths:
    print(f"VERIFIED: agent used scratchpad ({len(scratch_writes)} writes, {len(scratch_reads)} reads) "
          f"and {len(written_paths)} files persist in DBFS.")
elif not scratch_writes:
    print("NOT VERIFIED: agent did not call scratch_write on this thread.")


> ### 💡 Key takeaways — Part 13
>
> - **Persistence between turns is what makes it an agent, not a chat.** Conversation history, tool outputs, schema facts, corrections — all of it survives in Oracle and reappears via the context card on the next turn.
> - **A correction at turn N changes the answer at turn N+1.** The `remember` tool writes to the same OAMP store the agent retrieves from — no separate "training" step, no model fine-tune, no app deploy.
> - **Watch the trace, not just the answer.** A correct answer for the wrong reasons (LLM guessed) is more dangerous than a wrong answer with a clear trace (caught the model and corrected). The trace is the supervision signal.
> - **Independent verification beats agent self-report.** The §13.0.1 scratchpad demo reads bytes back from DBFS *directly*, not via the agent — proves writes actually committed, not that the agent claimed they did.


## Part 14 — Extensions: making the harness production-grade

*Part 13 proved the harness works on the demo schema. Part 14 hardens it for real deployments. Three sharp corners get smoothed: context compaction so long threads don't blow the window (14.1), tool-output offload so big results don't blow the context (14.2), and identity-aware authorization so the agent's results follow the end-user's grants, not the DB user's (14.4 — Deep Data Security).*

The loop in §11 works, but there are three sharp corners that become painful the moment you use it seriously:

1. **Conversations get long.** Token costs grow linearly and reasoning quality degrades. OAMP handles this when configured with an extraction LLM (which we did in §4.1) — §14.1 shows the result.
2. **Tool outputs get large.** We offload to OAMP memories tagged with the `tool_call_id`, but the agent needs a way to *retrieve* the full bytes when its truncated copy isn't enough. §14.2 adds the retrieval tool.

Each fix is additive. The loop is untouched.


### 14.1 Context compaction — handed off to OAMP

Long threads cause context rot — token cost grows linearly and reasoning quality degrades. OAMP solves this when configured with an extraction LLM (which we did in §4.1): it watches each thread, periodically extracts durable facts as `record_type="memory"` rows, and (with `enable_context_summary=True`) maintains a rolling summary surfaced via `thread.get_context_card()`. The cells below confirm both effects on this thread — an LLM-written summary instead of a raw concatenation, and memories the agent didn't write explicitly.


In [ ]:
# Without an LLM attached, get_summary() returns a plain concatenation of
# recent messages rather than an LLM-written compression. Useful for sanity-
# checking that the thread state is what you expect.
thread_obj = get_thread(thread)
summary = thread_obj.get_summary()
# get_summary() returns an OracleSummary (or None). It has a .content field
# (plain text) and a .formatted_content field (XML-style); __str__ delegates
# to formatted_content.
if summary is not None and summary.content:
    text = summary.content
    print(text[:600], "..." if len(text) > 600 else "")
else:
    print("(no summary - thread may be too short, or no LLM is attached for compression)")


Direct read of the OAMP store — what's actually stored on this thread, by kind. Shows that scanner facts, user corrections, and tool outputs all coexist as memories with different `metadata.kind` tags. The agent loop fishes from this same store every turn through `thread.get_context_card()`.


In [ ]:
# Memories landed via explicit add_memory calls (the scanner, the `remember`
# tool, log_tool). With extract_memories=False there are no LLM-extracted
# memories - just the ones we wrote on purpose.
recorded = memory_client.search(
    "what has the agent recorded for this thread",
    user_id=USER_ID, agent_id=AGENT_ID,
    thread_id=thread,
    record_types=["memory"],
    max_results=10,
)
print(f"OAMP has {len(recorded)} memories on this thread:")
for r in recorded:
    kind = r.metadata.get("kind", "?") if r.metadata else "?"
    print(f"  [{kind}] {r.content[:200]}")


In [ ]:
# How many messages OAMP currently has on this thread, and how big the
# context card is now (vs. raw transcript).
thread_obj = get_thread(thread)
msgs = thread_obj.get_messages()
card = thread_obj.get_context_card()
card_text = str(card) if card else ""
raw  = "\n".join(f"[{m.role}] {m.content}" for m in msgs)
ratio = len(card_text) / max(len(raw), 1)
print(f"messages on thread:    {len(msgs)}")
print(f"raw transcript:        {len(raw)} chars")
print(f"context card:          {len(card_text)} chars")
print(f"enrichment vs raw:     {ratio:.0%}  (>=100% means the card surfaces more relevant")
print(f"                                  memory than the raw transcript alone holds)")


Closing test of §14.1 — same thread, fresh question that requires recall from much earlier. The context card now includes the OAMP-extracted memories *and* the rolling summary, so the model can answer without re-reading the full transcript.


In [ ]:
# New turn on the same thread. build_context will now include the summary
# description (because all the old messages are linked), and the agent should
# recall the earlier correction (cell 134) about cargo_items.unit_value_cents
# via the OAMP context card — without us re-stating it in the prompt.
q5 = ("Remind me what unit cargo_items.unit_value_cents is stored in, "
      "and how we established that — cite the memory you retrieved.")
print("USER:", q5)
print("\nASSISTANT:", agent_turn(q5, thread_id=thread))


> ### 💡 Key takeaways — §14.1
>
> - **Long threads cause context rot.** Every turn re-reads the whole history; tokens grow linearly; the model drops signal from the front. Compaction isn't an optimisation, it's a correctness fix.
> - **Hand it to OAMP.** With `extract_memories=True` and `enable_context_summary=True`, OAMP runs background extraction passes and maintains a rolling summary. The agent loop reads `thread.get_context_card()` and gets compaction for free — no bespoke summarizer.
> - **The summary is LLM-written.** Without an LLM attached to OAMP, `get_summary()` is plain message concatenation. With one, you get an actual compression — same model the agent uses, no second credential.


### 14.2 Tool-output offload — moving big results out of context

Part 11's `agent_turn` inlines every tool result verbatim. That's fine for short outputs but blows the context window on a 50-row `run_sql`, a multi-KB skill body, or a long `exec_js` log. §14.2 fixes that with three glued-together pieces:

1. **`log_tool`** — every dispatch persists the full output as an OAMP memory tagged `kind=tool_output` with the LLM's `tool_call_id`.
2. **Truncation marker** — outputs over 600 bytes are replaced in the message list with a compact preview ending in `...[+N bytes. full output: fetch_tool_output(tool_call_id='call_…')]`. The model knows where to find the rest.
3. **`fetch_tool_output(tool_call_id)`** — registered tool that recovers the full bytes by id when the agent decides it needs them.

The cell below does all three in one go: defines `log_tool`, registers `fetch_tool_output`, and **redefines** `agent_turn` to call `log_tool` and emit the truncation marker. Re-run cell §11's `agent_turn` to revert to the minimal version. The change is purely additive — every input/output the agent has ever seen still works.


In [ ]:
# §14.2 — tool-output offload (the missing piece in §11's minimal agent_turn).
#
# Three changes wired together:
#   (a) `log_tool` — persist the full tool output as an OAMP memory tagged
#       kind=tool_output, indexed by the LLM's tool_call_id;
#   (b) `tool_fetch_tool_output` — the agent-side retrieval tool that pulls
#       the bytes back by id when its inlined preview was truncated;
#   (c) redefined `agent_turn` — calls log_tool for each dispatch and replaces
#       large outputs in the message list with a truncation marker that points
#       the agent at fetch_tool_output.
#
# Once you run this cell, all subsequent turns offload + truncate. Re-run
# cell §11's `agent_turn` to revert to the minimal verbatim-inline version.

def log_tool(thread_id: str, tool_call_id: str, tool_name: str,
             tool_args: dict, tool_output: str) -> None:
    """Offload one tool result to OAMP as a kind=tool_output memory."""
    memory_client.add_memory(
        tool_output,
        user_id=USER_ID, agent_id=AGENT_ID,
        thread_id=thread_id,
        metadata={
            "kind": "tool_output",
            "tool_call_id": tool_call_id,
            "tool_name": tool_name,
            "tool_args": json.dumps(tool_args),
        },
    )


@register
def tool_fetch_tool_output(tool_call_id: str) -> str:
    """Retrieve the full, untruncated output of a previous tool call.
    Use this when a prior tool result in your context was truncated with
    '...[+N bytes. full output: fetch_tool_output(tool_call_id=...)]' and you need
    the missing bytes to answer.
    """
    records = memory_client._store.list(
        "memory",
        user_id=USER_ID, agent_id=AGENT_ID,
        metadata_filter={"kind": "tool_output", "tool_call_id": tool_call_id},
        limit=1,
    )
    if not records:
        return json.dumps({"error": f"no tool call with id {tool_call_id}"})
    r = records[0]
    meta = r.metadata or {}
    return json.dumps({
        "tool_name":   meta.get("tool_name"),
        "tool_args":   meta.get("tool_args"),
        "tool_output": r.content,
    })


# Redefine agent_turn to add the offload + truncation marker.
def agent_turn(user_query: str, thread_id: str = "default",     # noqa: F811
                max_iterations: int = 8, budget_seconds: float = 360.0,
                verbose: bool = True) -> str:
    started = time.time()
    log_message(thread_id, "user", user_query)
    context = build_context(thread_id, user_query)
    messages: list[dict] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": context},
    ]
    tool_schemas = retrieve_tools(user_query, k=6)

    final = ""
    step = 0
    DEDUPE_WINDOW = 3
    recent_calls: list[tuple[str, str]] = []
    for step in range(max_iterations):
        if time.time() - started > budget_seconds:
            if verbose: print(f"  ! budget exhausted at iteration {step}")
            break

        resp = chat(messages, tools=tool_schemas)
        msg = resp.choices[0].message
        if not msg.tool_calls:
            final = msg.content or ""
            if verbose: print(f"  step {step}: final answer")
            break

        messages.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name,
                              "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ],
        })

        for tc in msg.tool_calls:
            name = tc.function.name
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            if verbose: print(f"  step {step}: -> {name}({args})")

            # Dedupe: short-circuit if the LLM is dispatching the same
            # (tool, args) it called within the last DEDUPE_WINDOW steps.
            # Common pathology: agent loops re-reading the same scratchpad
            # path or re-issuing the same search_knowledge query, never
            # converging until budget exhausts.
            call_key = (name, json.dumps(args, sort_keys=True))
            if call_key in recent_calls[-DEDUPE_WINDOW:]:
                output = json.dumps({
                    "note": f"duplicate call to {name} with identical args within "
                            f"the last {DEDUPE_WINDOW} dispatches — output unchanged; "
                            f"reuse the prior tool result above instead of re-dispatching."
                })
                if verbose: print(f"          ↳ duplicate dispatch — short-circuited")
            elif name not in TOOLS:
                output = json.dumps({"error": f"unknown tool: {name}"})
            else:
                fn, _ = TOOLS[name]
                try:
                    output = fn(**args)
                except Exception as e:
                    output = json.dumps({"error": f"{type(e).__name__}: {e}"})
            recent_calls.append(call_key)

            # Offload full bytes, inline a compact preview with a recovery hint.
            log_tool(thread_id, tc.id, name, args, output)
            if len(output) <= 600:
                preview = output
            else:
                preview = (
                    output[:600] +
                    f" ...[+{len(output)-600} bytes. "
                    f"full output: fetch_tool_output(tool_call_id='{tc.id}')]"
                )
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": preview})

    if not final:
        messages.append({"role": "user",
                         "content": "Budget exhausted. Provide your best answer now, no more tools."})
        resp = chat(messages, tools=None)
        final = resp.choices[0].message.content or "(no answer produced)"

    log_message(thread_id, "assistant", final)

    try:
        memory_client.add_memory(
            f"User: {user_query}\n\nAssistant: {final}",
            user_id=USER_ID, agent_id=AGENT_ID,
            thread_id=thread_id,
            metadata={"kind": "episodic", "thread_id": thread_id,
                      "user_query": user_query[:240],
                      "elapsed_seconds": round(time.time() - started, 2)},
        )
    except Exception as _e:
        if verbose: print(f"  ! episodic add_memory failed: {type(_e).__name__}: {_e}")

    if verbose: print(f"  [{time.time() - started:.1f}s, {step + 1} steps]")
    return final


# Sanity: ask for the full output of the most recent tool call on `thread`.
recent = memory_client._store.list(
    "memory",
    user_id=USER_ID, agent_id=AGENT_ID,
    thread_id=thread,
    metadata_filter={"kind": "tool_output"},
    limit=1,
)
if recent:
    last_id = (recent[0].metadata or {}).get("tool_call_id", "")
    print(f"fetching tool_call_id={last_id}")
    print(tool_fetch_tool_output(tool_call_id=last_id)[:400])
else:
    print("(no tool_output memories yet on this thread — run a turn first)")


> ### 💡 Key takeaways — §14.2
>
> - **Inlining every tool output blows the window.** A 50-row `run_sql`, a multi-KB skill body, or a long `exec_js` log can each consume more context than the rest of the turn combined.
> - **Offload + truncation marker is the pattern.** Full output → OAMP memory tagged with `tool_call_id`. Compact preview → message list. The marker tells the model where the rest is: `fetch_tool_output(tool_call_id='call_…')`.
> - **The agent decides what to retrieve.** Most truncated outputs are never refetched — the model only pulls full bytes when its preview isn't enough. Bandwidth follows attention.


### 14.4 Identity-aware authorization with Deep Data Security (DDS)

By default the agent sees every row and column the `AGENT` DB user has been granted — it inherits the database user's privileges, period. Real deployments need the trust boundary to follow the **end-user on whose behalf the agent is acting**, not the DB user the agent runs as. A jailbroken prompt that gets the model to issue `SELECT * FROM SUPPLYCHAIN.cargo_items` should still come back filtered for the *human* in the loop.

**Oracle Deep Data Security (DDS)** in Oracle AI Database 26ai moves that boundary into the database kernel. You declare row, column, and cell-level rules in **declarative SQL**, and the engine enforces them transparently — no matter what SQL the agent constructs. The trust model becomes:

```
end_user (real human or upstream caller)
       │  identity propagated via DBMS_SESSION.SET_CONTEXT
       ▼
AGENT (database user)  ──── DDS policies evaluated by kernel ────▶ rows/cols filtered before tool sees them
```

The agent doesn't decide what it can see — the database does. A jailbroken prompt that synthesizes `SELECT * FROM SUPPLYCHAIN.cargo_items` still gets back a DDS-filtered result set.

We'll demo this with two end-users on the `SUPPLYCHAIN` schema:

| End-user | Authorized oceans | Clearance | Should see |
|---|---|---|---|
| `apac.fleet@acme.com` | PACIFIC + INDIAN | STANDARD | only voyages in those oceans; `unit_value_cents` masked to `NULL` |
| `cfo@acme.com` | ALL | EXECUTIVE | every voyage; declared cargo values visible |

Three policies on `SUPPLYCHAIN`:

1. **Row policy** on `voyages` — only rows whose `ocean_region` is in the user's authorized oceans.
2. **Column policy** on `cargo_items` — `unit_value_cents` is hidden (returned as `NULL`) unless clearance is `EXECUTIVE`.
3. **Legacy bypass** — when `EDA_CTX.END_USER` is `NULL`, all rows/columns are visible. This keeps the §13 demos and the §14.1 follow-up turn working unchanged.

> **Image note.** The `CREATE DATA SECURITY POLICY` syntax is shipped with Oracle AI Database 26ai. If your image predates this and the DDL fails with `ORA-00900`/`ORA-02000`, the cells catch the error, print a clear message, and the rest of the notebook still runs — you just won't see filtering. Confirm your build with `SELECT BANNER_FULL FROM v$version`.


**Identity propagation at a glance** — where the persona name in the front-end ends up in the kernel's WHERE clause. The same diagram covers row policies *and* column masks; only the policy predicate changes.

```
IDENTITY PROPAGATION — front-end persona to kernel filter
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  END-USER picks persona in app:  "Use As: cfo@acme.com"
       │
       │ POST /api/chat   { as_user: "cfo@acme.com", message: "…" }
       ▼
  APPLICATION  (Flask events handler)
       identity = get_identity("cfo@acme.com")
                  → regions    = ["ALL"]
                  → clearance  = "EXECUTIVE"
       │
       │ agent_turn(identity=identity, …)
       ▼
  AGENT  (calls set_eda_ctx() before any SQL fires this turn)
       AGENT.set_eda_ctx(
           p_end_user  = 'cfo@acme.com',
           p_clearance = 'EXECUTIVE')
       │
       ▼
╔═══════════════════════ ORACLE KERNEL ═══════════════════════════════╗
║                                                                      ║
║   DBMS_SESSION.SET_CONTEXT('EDA_CTX','END_USER',  'cfo@acme.com')   ║
║   DBMS_SESSION.SET_CONTEXT('EDA_CTX','CLEARANCE', 'EXECUTIVE')      ║
║                                                                      ║
║   Namespace EDA_CTX has a TRUSTED PROCEDURE clause: only            ║
║   AGENT.set_eda_ctx can write it. The agent cannot self-elevate     ║
║   by calling DBMS_SESSION.SET_CONTEXT directly.                     ║
║                              │                                       ║
║                              ▼                                       ║
║   Every SELECT in this session is rewritten by the kernel:          ║
║                                                                      ║
║   ┌─ DDS row policy on SUPPLYCHAIN.voyages ──────────────────────┐  ║
║   │  USING (                                                      │  ║
║   │    EXISTS (                                                   │  ║
║   │      SELECT 1 FROM AGENT.agent_authorizations                 │  ║
║   │       WHERE end_user = SYS_CONTEXT('EDA_CTX','END_USER')      │  ║
║   │         AND (auth_region = 'ALL'                              │  ║
║   │              OR auth_region = voyages.ocean_region))          │  ║
║   │  )                                                            │  ║
║   └──────────────────────────────────────────────────────────────┘  ║
║                                                                      ║
║   ┌─ DDS column mask on cargo_items.unit_value_cents ─────────────┐ ║
║   │  HIDE COLUMNS (unit_value_cents)                                │ ║
║   │  WHEN COALESCE(SYS_CONTEXT('EDA_CTX','CLEARANCE'),              │ ║
║   │                'STANDARD') <> 'EXECUTIVE'                       │ ║
║   └────────────────────────────────────────────────────────────────┘ ║
║                              │                                       ║
║                              ▼                                       ║
║   filtered rows + masked columns returned                           ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
       │
       ▼
  tool_run_sql receives the filtered result set. Same prompt, same
  generated SQL, *different* result per identity — and the agent
  cannot bypass: DDS is enforced at parse/execute time regardless
  of how the SQL was constructed (handwritten in the cell, agent-
  generated, even MLE-issued from inside an exec_js sandbox).
```


#### 14.4.1 `voyages.ocean_region` — what the row policy filters on

DDS needs *something to filter on*. The supply-chain schema already has a first-class `ocean_region` column on `voyages` (`PACIFIC` / `ATLANTIC` / `INDIAN` / `MEDITERRANEAN`), populated at seed time. Below we just confirm the distribution before attaching policies, plus a smoke-test of `SDO_WITHIN_DISTANCE` to show the spatial side is wired up too.


In [ ]:
# In the new SUPPLYCHAIN schema, ocean_region is already a first-class column on
# voyages (PACIFIC | ATLANTIC | INDIAN | MEDITERRANEAN), populated at seed time.
# Confirm the distribution before we attach DDS policies in §14.4.4.
with demo_conn.cursor() as cur:
    cur.execute("SELECT ocean_region, COUNT(*) FROM voyages GROUP BY ocean_region ORDER BY ocean_region")
    print("voyages by ocean_region:", list(cur))

# Bonus smoke test: SDO_WITHIN_DISTANCE — vessels currently within 1500 km of Singapore.
with demo_conn.cursor() as cur:
    cur.execute("""
        SELECT v.name, vp.speed_knots, vp.latitude, vp.longitude
          FROM vessel_positions vp
          JOIN vessels v ON v.vessel_id = vp.vessel_id
         WHERE SDO_WITHIN_DISTANCE(
                 vp.position,
                 SDO_GEOMETRY(2001, 8307,
                   SDO_POINT_TYPE(103.8326, 1.2649, NULL), NULL, NULL),
                 'distance=1500 unit=km'
               ) = 'TRUE'
         ORDER BY v.name
    """)
    rows = list(cur)
print(f"vessels within 1500km of Singapore: {len(rows)}")
for r in rows[:5]:
    print(f"  {r[0]} — {r[2]:.2f}, {r[3]:.2f}, {r[1]} kts")


#### 14.4.2 Authorization & clearance tables

These live in the `AGENT` schema (the security model belongs to the agent's owner, not the data's owner). `DEMO` gets `SELECT` on them so DDS policies on `DEMO.*` can reference them at evaluation time.


In [ ]:
# Tables that DDS policies will join against. End-users authorised by
# ocean region (PACIFIC / ATLANTIC / INDIAN / MEDITERRANEAN), with optional
# 'ALL' for executives / fleet-wide roles.
DDS_DDL = [
    ("agent_authorizations", (
        "CREATE TABLE agent_authorizations ("
        "  end_user     VARCHAR2(128) NOT NULL,"
        "  auth_region  VARCHAR2(20)  NOT NULL,"
        "  PRIMARY KEY (end_user, auth_region)"
        ")"
    )),
    ("agent_clearances", (
        "CREATE TABLE agent_clearances ("
        "  end_user   VARCHAR2(128) PRIMARY KEY,"
        "  clearance  VARCHAR2(32)  NOT NULL,"
        "  notes      VARCHAR2(400)"
        ")"
    )),
]

with agent_conn.cursor() as cur:
    for tname, ddl in DDS_DDL:
        try:
            cur.execute(ddl)
            print(f"created {tname}")
        except oracledb.DatabaseError as e:
            if e.args[0].code == 955:
                pass
            else:
                raise

    cur.execute("DELETE FROM agent_authorizations")
    cur.executemany(
        "INSERT INTO agent_authorizations (end_user, auth_region) VALUES (:1, :2)",
        [
            # Regional fleet managers
            ("apac.fleet@acme.com",      "PACIFIC"),
            ("apac.fleet@acme.com",      "INDIAN"),
            ("emea.fleet@acme.com",      "ATLANTIC"),
            ("emea.fleet@acme.com",      "MEDITERRANEAN"),
            ("americas.fleet@acme.com",  "PACIFIC"),
            ("americas.fleet@acme.com",  "ATLANTIC"),
            # C-suite — all oceans
            ("ceo@acme.com",             "ALL"),
            ("cfo@acme.com",             "ALL"),
        ],
    )
    cur.execute("DELETE FROM agent_clearances")
    cur.executemany(
        "INSERT INTO agent_clearances (end_user, clearance, notes) VALUES (:1, :2, :3)",
        [
            ("apac.fleet@acme.com",     "STANDARD",  "APAC + Indian Ocean fleet manager"),
            ("emea.fleet@acme.com",     "STANDARD",  "EMEA + Mediterranean fleet manager"),
            ("americas.fleet@acme.com", "STANDARD",  "Americas fleet manager"),
            ("ceo@acme.com",            "EXECUTIVE", "Chief Executive — sees declared values"),
            ("cfo@acme.com",            "EXECUTIVE", "Chief Financial Officer — sees declared values"),
        ],
    )
agent_conn.commit()

with sys_conn.cursor() as cur:
    cur.execute(f"GRANT SELECT ON {AGENT_USER}.agent_authorizations TO {DEMO_USER}")
    cur.execute(f"GRANT SELECT ON {AGENT_USER}.agent_clearances     TO {DEMO_USER}")
sys_conn.commit()
print("seeded authorizations + clearances; granted SELECT to SUPPLYCHAIN")


#### 14.4.3 Application context (`EDA_CTX`) and the setter procedure

DDS reads identity via `SYS_CONTEXT('EDA_CTX', 'END_USER')`. We create a namespace **bound to a specific procedure** — only that procedure can write into `EDA_CTX`, so a hostile agent can't `SET_CONTEXT` itself an executive clearance.


In [ ]:
# 1. Procedure that owns writes to EDA_CTX.
SETTER_SQL = """
CREATE OR REPLACE PROCEDURE set_eda_ctx(
    p_end_user  IN VARCHAR2,
    p_clearance IN VARCHAR2 DEFAULT NULL
) AS
    v_clearance VARCHAR2(32);
BEGIN
    -- Resolve clearance from the table if not explicitly passed.
    IF p_clearance IS NULL AND p_end_user IS NOT NULL THEN
        BEGIN
            SELECT clearance INTO v_clearance
              FROM agent_clearances WHERE end_user = p_end_user;
        EXCEPTION WHEN NO_DATA_FOUND THEN
            v_clearance := 'STANDARD';
        END;
    ELSE
        v_clearance := NVL(p_clearance, 'STANDARD');
    END IF;

    DBMS_SESSION.SET_CONTEXT('EDA_CTX', 'END_USER',  p_end_user);
    DBMS_SESSION.SET_CONTEXT('EDA_CTX', 'CLEARANCE', v_clearance);
END;
"""

with agent_conn.cursor() as cur:
    cur.execute(SETTER_SQL)
print("created procedure AGENT.set_eda_ctx")

# 2. Namespace bound to the setter. AGENT must have CREATE ANY CONTEXT
#    (granted via DB_DEVELOPER_ROLE in §3.2 - if not, do it as SYSDBA).
try:
    with agent_conn.cursor() as cur:
        cur.execute(f"CREATE OR REPLACE CONTEXT eda_ctx USING {AGENT_USER}.set_eda_ctx")
    print(f"created context EDA_CTX (writes restricted to {AGENT_USER}.set_eda_ctx)")
except oracledb.DatabaseError:
    with sys_conn.cursor() as cur:
        cur.execute(f"GRANT CREATE ANY CONTEXT TO {AGENT_USER}")
        cur.execute(f"CREATE OR REPLACE CONTEXT eda_ctx USING {AGENT_USER}.set_eda_ctx")
    print("granted CREATE ANY CONTEXT to AGENT and created EDA_CTX")

# 3. Also need DBMS_SESSION execute (already covered by RESOURCE/CONNECT).
# Smoke test: set + read back.
def set_identity(end_user: str | None, clearance: str | None = None) -> None:
    """Set the EDA_CTX identity on agent_conn for subsequent SQL."""
    with agent_conn.cursor() as cur:
        cur.callproc(f"{AGENT_USER}.set_eda_ctx", [end_user, clearance])

set_identity("cfo@acme.com")
with agent_conn.cursor() as cur:
    cur.execute("SELECT SYS_CONTEXT('EDA_CTX','END_USER'), SYS_CONTEXT('EDA_CTX','CLEARANCE') FROM DUAL")
    print("context after set:", cur.fetchone())

# Clear for the rest of setup; legacy paths see NULL = bypass.
set_identity(None)


#### 14.4.4 Declare the policies — declarative DDS, with a `DBMS_RLS` fallback

We try the declarative `CREATE DATA SECURITY POLICY` syntax first. If the running image doesn't ship that surface yet (you'll see `ORA-00900` / `ORA-00901` / `ORA-02000`), the cell falls back to **`DBMS_RLS`** — the kernel-level row/column security mechanism that DDS layers on top of. The semantics are identical for our scenario:

| Goal | Declarative DDS | `DBMS_RLS` fallback |
|---|---|---|
| Row policy on `voyages.ocean_region` | `ON voyages USING (...)` | predicate function returning a `WHERE` fragment, attached via `DBMS_RLS.ADD_POLICY` |
| Column hide on `cargo_items.unit_value_cents` | `HIDE COLUMNS (unit_value_cents) WHEN ...` | `sec_relevant_cols => 'UNIT_VALUE_CENTS'` + `sec_relevant_cols_opt => DBMS_RLS.ALL_ROWS`; predicate returns `'1=0'` to mask, `'1=1'` to reveal |
| Identity bypass | `WHERE END_USER IS NULL` literal | same — predicate returns `'1=1'` |

Either way the trust boundary is the database kernel: the agent constructs SQL, the kernel filters before the result lands in `tool_run_sql`.


In [ ]:
DDS_AVAILABLE = False
DDS_MODE = None

# ---------- Path A: declarative DDS (preferred, 26ai) ----------
ROW_POLICY_DDS = """
CREATE DATA SECURITY POLICY voyages_region_policy
ON voyages
USING (
    SYS_CONTEXT('EDA_CTX','END_USER') IS NULL
 OR EXISTS (
        SELECT 1 FROM AGENT.agent_authorizations a
         WHERE a.end_user = SYS_CONTEXT('EDA_CTX','END_USER')
           AND (a.auth_region = 'ALL' OR a.auth_region = voyages.ocean_region)
    )
)
"""

COL_POLICY_DDS = """
CREATE DATA SECURITY POLICY cargo_value_policy
ON cargo_items
HIDE COLUMNS (unit_value_cents)
WHEN SYS_CONTEXT('EDA_CTX','END_USER') IS NOT NULL
 AND COALESCE(SYS_CONTEXT('EDA_CTX','CLEARANCE'),'STANDARD') <> 'EXECUTIVE'
"""

for stmt in [
    "DROP DATA SECURITY POLICY voyages_region_policy",
    "DROP DATA SECURITY POLICY cargo_value_policy",
]:
    try:
        with demo_conn.cursor() as cur:
            cur.execute(stmt)
    except oracledb.DatabaseError:
        pass

_FALLBACK_CODES = {900, 901, 922, 2000, 942, 1031}
dds_unsupported = False
created_count = 0
for label, ddl in [("row policy on voyages",       ROW_POLICY_DDS),
                   ("col policy on cargo_items",   COL_POLICY_DDS)]:
    try:
        with demo_conn.cursor() as cur:
            cur.execute(ddl)
        print(f"  declarative DDS: created {label}")
        created_count += 1
    except oracledb.DatabaseError as e:
        code_ = e.args[0].code
        if code_ in _FALLBACK_CODES:
            print(f"  declarative DDS not available (ORA-{code_:05d}); will fall back to DBMS_RLS.")
            dds_unsupported = True
            break
        raise

if created_count == 2 and not dds_unsupported:
    DDS_AVAILABLE = True
    DDS_MODE = "declarative"

# ---------- Path B: DBMS_RLS fallback ----------
if dds_unsupported:
    with sys_conn.cursor() as cur:
        try:
            cur.execute(f"GRANT EXECUTE ON SYS.DBMS_RLS TO {DEMO_USER}")
            print(f"  granted EXECUTE ON DBMS_RLS to {DEMO_USER}")
        except oracledb.DatabaseError as e:
            if e.args[0].code != 1031:
                raise
    sys_conn.commit()

    DROP_BLOCK = """
DECLARE
  e_no_policy EXCEPTION;
  PRAGMA EXCEPTION_INIT(e_no_policy, -28102);
BEGIN
  BEGIN DBMS_RLS.DROP_POLICY(:s, 'VOYAGES',     'VOYAGES_REGION_POLICY');
  EXCEPTION WHEN e_no_policy THEN NULL; END;
  BEGIN DBMS_RLS.DROP_POLICY(:s, 'CARGO_ITEMS', 'CARGO_VALUE_POLICY');
  EXCEPTION WHEN e_no_policy THEN NULL; END;
END;
"""
    with demo_conn.cursor() as cur:
        try:
            cur.execute(DROP_BLOCK, s=DEMO_USER)
        except oracledb.DatabaseError:
            pass

    PRED_FN_VOYAGES = """
CREATE OR REPLACE FUNCTION voyages_region_predicate(
    p_schema IN VARCHAR2, p_object IN VARCHAR2
) RETURN VARCHAR2 AS
BEGIN
    -- Legacy / no identity = full visibility (so §13 demos still work).
    IF SYS_CONTEXT('EDA_CTX','END_USER') IS NULL THEN
        RETURN '1=1';
    END IF;
    -- Filter by the end-user's authorized ocean regions.
    RETURN q'[EXISTS (
        SELECT 1 FROM AGENT.agent_authorizations a
         WHERE a.end_user = SYS_CONTEXT('EDA_CTX','END_USER')
           AND (a.auth_region = 'ALL' OR a.auth_region = ocean_region)
    )]';
END;
"""

    PRED_FN_CARGO = """
CREATE OR REPLACE FUNCTION cargo_value_predicate(
    p_schema IN VARCHAR2, p_object IN VARCHAR2
) RETURN VARCHAR2 AS
BEGIN
    -- With sec_relevant_cols_opt => DBMS_RLS.ALL_ROWS, predicate-FALSE means
    -- "return the row but mask the listed columns (unit_value_cents) to NULL".
    --   * legacy / no identity     -> '1=1'  (no masking)
    --   * EXECUTIVE                 -> '1=1'  (no masking)
    --   * anyone else               -> '1=0'  (commercial value masked)
    IF SYS_CONTEXT('EDA_CTX','END_USER') IS NULL THEN
        RETURN '1=1';
    END IF;
    IF NVL(SYS_CONTEXT('EDA_CTX','CLEARANCE'),'STANDARD') = 'EXECUTIVE' THEN
        RETURN '1=1';
    END IF;
    RETURN '1=0';
END;
"""

    with demo_conn.cursor() as cur:
        cur.execute(PRED_FN_VOYAGES)
        cur.execute(PRED_FN_CARGO)
    print("  created predicate functions: voyages_region_predicate, cargo_value_predicate")

    ADD_ROW = """
BEGIN
  DBMS_RLS.ADD_POLICY(
    object_schema   => 'SUPPLYCHAIN',
    object_name     => 'VOYAGES',
    policy_name     => 'VOYAGES_REGION_POLICY',
    function_schema => 'SUPPLYCHAIN',
    policy_function => 'VOYAGES_REGION_PREDICATE',
    statement_types => 'SELECT'
  );
END;
"""
    ADD_COL = """
BEGIN
  DBMS_RLS.ADD_POLICY(
    object_schema         => 'SUPPLYCHAIN',
    object_name           => 'CARGO_ITEMS',
    policy_name           => 'CARGO_VALUE_POLICY',
    function_schema       => 'SUPPLYCHAIN',
    policy_function       => 'CARGO_VALUE_PREDICATE',
    statement_types       => 'SELECT',
    sec_relevant_cols     => 'UNIT_VALUE_CENTS',
    sec_relevant_cols_opt => DBMS_RLS.ALL_ROWS
  );
END;
"""
    with demo_conn.cursor() as cur:
        cur.execute(ADD_ROW)
        print("  added VPD policy VOYAGES_REGION_POLICY  (row-level on voyages.ocean_region)")
        cur.execute(ADD_COL)
        print("  added VPD policy CARGO_VALUE_POLICY     (column-mask on cargo_items.unit_value_cents)")

    DDS_AVAILABLE = True
    DDS_MODE = "dbms_rls"

print(f"\nDDS_AVAILABLE = {DDS_AVAILABLE}  (mode: {DDS_MODE})")


#### 14.4.5 Thread identity through `agent_turn`

`agent_turn` now accepts `end_user` and `clearance` and calls the setter once at the start of the turn. Every `tool_run_sql` call inside that turn inherits the identity from the connection's session context — no per-tool plumbing needed.

We also wrap §10's `tool_run_sql` to surface DDS-induced empty results with a friendlier message, so the model can explain *why* a query came back empty for one user but not another.


In [ ]:
import functools

# Wrap §10's tool_run_sql so an empty result set under a non-NULL identity
# is flagged. The model then knows to mention the identity restriction in its
# answer rather than claiming "there are no orders".
_inner_run_sql = TOOLS["run_sql"][0]

@register
def tool_run_sql(sql: str, max_rows: int = 50) -> str:
    """Execute a READ-ONLY SQL statement against the target Oracle AI Database.
    Results are filtered by Deep Data Security policies for the current end-user.
    If a SELECT returns 0 rows or NULL columns when an end-user identity is set,
    the result is annotated so the agent can explain the restriction.
    """
    raw = _inner_run_sql(sql, max_rows=max_rows)
    try:
        payload = json.loads(raw)
    except Exception:
        return raw
    if "error" in payload:
        return raw

    with agent_conn.cursor() as cur:
        cur.execute("SELECT SYS_CONTEXT('EDA_CTX','END_USER'), SYS_CONTEXT('EDA_CTX','CLEARANCE') FROM DUAL")
        end_user, clearance = cur.fetchone()

    if end_user:
        payload["dds_identity"] = {"end_user": end_user, "clearance": clearance}
        if payload.get("row_count", 0) == 0:
            payload["dds_note"] = (
                f"0 rows returned. Deep Data Security may be filtering on identity "
                f"end_user={end_user!r} clearance={clearance!r}."
            )
        # Heuristic: flag columns where every value is NULL — likely DDS column-hide.
        cols, rows = payload.get("columns", []), payload.get("rows", [])
        if cols and rows:
            null_cols = [
                cols[i] for i in range(len(cols))
                if all((row[i] is None) for row in rows)
            ]
            if null_cols:
                payload["dds_masked_columns"] = null_cols
    return json.dumps(payload, default=str)


# Redefine agent_turn to set EDA_CTX before the loop runs. Prior signature
# (no end_user) keeps working — when end_user is None, the setter clears the
# context and the legacy bypass in the row policy returns all rows.
_prior_agent_turn = agent_turn  # keep a reference in case anything else captures it

def agent_turn(user_query: str, thread_id: str = "default",
               max_iterations: int = 8, budget_seconds: float = 360.0,
               verbose: bool = True,
               end_user: str | None = None,
               clearance: str | None = None) -> str:
    """As before, plus per-turn identity propagation into Oracle DDS via EDA_CTX."""
    set_identity(end_user, clearance)
    if verbose and end_user:
        print(f"  [identity: end_user={end_user!r} clearance={clearance!r}]")
    try:
        return _prior_agent_turn(user_query, thread_id=thread_id,
                                 max_iterations=max_iterations,
                                 budget_seconds=budget_seconds,
                                 verbose=verbose)
    finally:
        set_identity(None)  # don't leak identity to the next caller

print("patched: tool_run_sql (DDS-aware result annotation), agent_turn (end_user/clearance params)")


#### 14.4.6 Demo: same SQL, two identities, different results

We run the same `SELECT` with each identity, no agent in the way, so you can see DDS doing the filtering.


In [ ]:
def probe(label: str, end_user: str | None, clearance: str | None, sql: str):
    print(f"--- {label} (end_user={end_user!r}, clearance={clearance!r}) ---")
    set_identity(end_user, clearance)
    try:
        with agent_conn.cursor() as cur:
            cur.execute(sql)
            cols = [d[0] for d in cur.description]
            rows = list(cur)
        print("  columns:", cols)
        for r in rows[:8]:
            print("   ", r)
        if len(rows) > 8:
            print(f"    ...({len(rows) - 8} more)")
        print(f"  row_count = {len(rows)}\n")
    finally:
        set_identity(None)


# Row-level filtering on voyages.ocean_region
probe("CEO sees all oceans", "ceo@acme.com", "EXECUTIVE",
      "SELECT ocean_region, COUNT(*) AS n FROM SUPPLYCHAIN.voyages GROUP BY ocean_region ORDER BY ocean_region")

probe("APAC fleet sees PACIFIC + INDIAN only", "apac.fleet@acme.com", "STANDARD",
      "SELECT ocean_region, COUNT(*) AS n FROM SUPPLYCHAIN.voyages GROUP BY ocean_region ORDER BY ocean_region")

probe("EMEA fleet sees ATLANTIC + MEDITERRANEAN only", "emea.fleet@acme.com", "STANDARD",
      "SELECT ocean_region, COUNT(*) AS n FROM SUPPLYCHAIN.voyages GROUP BY ocean_region ORDER BY ocean_region")

# Column-level masking on cargo_items.unit_value_cents
probe("CFO sees declared cargo values", "cfo@acme.com", "EXECUTIVE",
      "SELECT cargo_item_id, description, quantity, unit_value_cents FROM SUPPLYCHAIN.cargo_items FETCH FIRST 5 ROWS ONLY")

probe("APAC fleet: cargo values masked to NULL", "apac.fleet@acme.com", "STANDARD",
      "SELECT cargo_item_id, description, quantity, unit_value_cents FROM SUPPLYCHAIN.cargo_items FETCH FIRST 5 ROWS ONLY")

probe("Legacy / no identity (bypass)", None, None,
      "SELECT cargo_item_id, description, unit_value_cents FROM SUPPLYCHAIN.cargo_items FETCH FIRST 3 ROWS ONLY")


#### 14.4.7 Through the agent

Same natural-language question, asked twice with different `end_user` arguments. The agent constructs whatever SQL it likes; the database filters it.


In [ ]:
dds_thread_cfo     = "demo-dds-cfo"
dds_thread_apac    = "demo-dds-apac"

q = ("How many voyages do we have in each ocean region, and what's the total "
     "declared cargo value (in USD) currently in transit?")

print("=" * 70)
print("ASKED AS CFO (clearance=EXECUTIVE, regions=ALL)")
print("=" * 70)
print(agent_turn(q, thread_id=dds_thread_cfo,
                 end_user="cfo@acme.com", clearance="EXECUTIVE"))

print("\n" + "=" * 70)
print("ASKED AS apac.fleet (clearance=STANDARD, regions=PACIFIC+INDIAN)")
print("=" * 70)
print(agent_turn(q, thread_id=dds_thread_apac,
                 end_user="apac.fleet@acme.com", clearance="STANDARD"))


> ### 💡 Key takeaways — §14.4 (DDS)
>
> - **Move the trust boundary into the kernel.** A jailbroken prompt that synthesizes `SELECT * FROM cargo_items` should still come back filtered for the *human* in the loop — not the AGENT DB user. The agent doesn't decide what it can see; the database does.
> - **Identity propagates via application context.** `DBMS_SESSION.SET_CONTEXT('EDA_CTX', 'END_USER', ...)` is the single channel. DDS / DBMS_RLS row policies read from it on every query, transparently to the agent code.
> - **TRUSTED PROCEDURE clause prevents self-elevation.** Only `AGENT.set_eda_ctx` can write to `EDA_CTX`. The agent can't bypass DDS by calling `DBMS_SESSION.SET_CONTEXT` from inside a tool — the namespace rejects writes from any other procedure.
> - **Same SQL, two identities, different rows.** Persona changes outside the agent; the agent code is unchanged. Authorization stops being an application-layer concern.


## Part 15 — Closing thoughts

*Parts 1–14 built and hardened the harness. Part 15 looks at what's left if you push this further — better SQL parsing, evaluations, OAMP tuning, multi-tenant ACLs.*

Everything that makes this an *agent* — memory, tool use, iterative reasoning, grounding in enterprise data, compaction under pressure, audited access — is in the cells above. No framework was required. The core loop in §11 plus the three extensions in §14 is roughly 300 lines of Python. You can read it end to end and know exactly where every decision is made and every side effect lands.

What's left if you want to take this further:

- **Replace the regex owner parser with a real SQL parser.** `antlr` with Oracle's grammar, or a lightweight stage that runs `EXPLAIN PLAN FOR` and mines `plan_table` for accessed objects. The regex is fine for a demo; not fine for a service.
- **Layer evaluations on top.** Pair each golden question with a target SQL, execute both, and grade with the LLM. OpenAI's `evals` API pattern applies directly — the model is the judge, the SQL execution is the ground truth. Regressions show up as failed grades in CI.
- **Tune OAMP's extraction cadence.** `memory_extraction_frequency` and `context_summary_update_frequency` trade fidelity against LLM cost. Profile against your traffic and pick numbers that keep the running summary fresh without over-spending on extraction calls.

The harness is small on purpose. Make it boring, make it legible, and let the model do the interesting work.
